In [1]:
# ============================================================
# MASTER RESUME CELL — Run this after ANY session restart
# Takes ~10 seconds total, loads everything from disk
# ============================================================

import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

SAVE_DIR = '/kaggle/working/'

print("="*55)
print("  CHECKING SAVED FILES")
print("="*55)

# ── Check what's already saved ───────────────────────────────
files = {
    'Step 2 — Shallow Features'  : 'train_shallow_features.csv',
    'Step 3 — Entity Grid'       : 'train_entity_grid_features.csv',
    'Step 4 — Embeddings'        : 'train_embeddings_features.csv',
    'Step 5 — Full Features'     : 'train_full_features.parquet',

}

for name, fname in files.items():
    path   = os.path.join(SAVE_DIR, fname)
    exists = os.path.exists(path)
    size   = f"{os.path.getsize(path)/1024/1024:.1f} MB" if exists else ""
    status = f"✅ EXISTS  {size}" if exists else "❌ MISSING"
    print(f"  {status}  ← {name}")

print()

# ── Load only what's needed ──────────────────────────────────

# Always load the full feature matrix (needed for Step 6)
parquet_path = os.path.join(SAVE_DIR, 'train_full_features.parquet')
if os.path.exists(parquet_path):
    import time
    t0         = time.time()
    train_full = pd.read_parquet(parquet_path)
    print(f"✅ train_full loaded in {time.time()-t0:.1f}s "
          f"→ shape {train_full.shape}")
else:
    print("❌ train_full NOT found — need to rerun Steps 1-5")

# Load latest results if available
for version in ['v5', 'v4', 'v3']:
    rpath = os.path.join(
        SAVE_DIR, f'results_per_essay_set_{version}.csv'
        if version != 'v5'
        else 'results_v5.csv'
    )
    if os.path.exists(rpath):
        results_df = pd.read_csv(rpath)
        print(f"\n✅ Latest results ({version}) loaded:")
        print(results_df[['essay_set','mean_qwk',
                           'mean_ea','mean_aa',
                           'mean_spearman']].to_string(index=False))
        break

print("\n✅ Resume complete — ready to continue!")

  CHECKING SAVED FILES
  ✅ EXISTS  17.4 MB  ← Step 2 — Shallow Features
  ✅ EXISTS  2.6 MB  ← Step 3 — Entity Grid
  ✅ EXISTS  158.8 MB  ← Step 4 — Embeddings
  ✅ EXISTS  82.9 MB  ← Step 5 — Full Features

✅ train_full loaded in 0.4s → shape (12976, 1059)

✅ Latest results (v4) loaded:
 essay_set  mean_qwk  mean_ea  mean_aa  mean_spearman
         1     0.754    0.435    0.892          0.750
         2     0.627    0.651    0.992          0.610
         3     0.624    0.641    0.979          0.641
         4     0.752    0.661    0.989          0.781
         5     0.769    0.642    0.993          0.786
         6     0.748    0.635    0.991          0.743
         7     0.772    0.134    0.413          0.794
         8     0.676    0.083    0.310          0.703

✅ Resume complete — ready to continue!


In [2]:
# ============================================================
# MASTER CHECKPOINT CELL
# Put this at TOP of notebook
# Loads from persistent input if files exist there
# ============================================================

import os
import shutil
import pandas as pd

WORK_DIR    = '/kaggle/working/'

# Check if files exist in input (from previous saved version)
# Replace 'your-notebook-name' with actual name
PERSIST_DIRS = [
    '/kaggle/input/asag-outputs/',
    '/kaggle/input/hybrid_llm/',  # ← update this
]

FILES_NEEDED = {
    'train_shallow_features.csv'         : 'Step 2',
    'train_entity_grid_features.csv'     : 'Step 3',
    'train_embeddings_instructor.csv'    : 'Step 4D',
    'train_full_features.parquet'        : 'Step 5',
    'train_full_features_instructor.parquet' : 'Step 5D',
    'results_v6.csv'                     : 'Step 6 V6',
    'results_v8.csv'                     : 'Step 6 V8',
}

print("="*55)
print("CHECKING AND RESTORING FILES")
print("="*55)

for filename, step in FILES_NEEDED.items():
    work_path = os.path.join(WORK_DIR, filename)

    # Already in working dir?
    if os.path.exists(work_path):
        size = os.path.getsize(work_path)/1024/1024
        print(f"✅ {filename:<45} ({size:.0f}MB) — {step}")
        continue

    # Try to copy from persistent dirs
    found = False
    for persist_dir in PERSIST_DIRS:
        src = os.path.join(persist_dir, filename)
        if os.path.exists(src):
            shutil.copy2(src, work_path)
            size = os.path.getsize(work_path)/1024/1024
            print(f"📂 {filename:<45} ({size:.0f}MB) "
                  f"— restored from input")
            found = True
            break

    if not found:
        print(f"❌ {filename:<45} — MISSING, need to rerun {step}")


CHECKING AND RESTORING FILES
✅ train_shallow_features.csv                    (17MB) — Step 2
✅ train_entity_grid_features.csv                (3MB) — Step 3
✅ train_embeddings_instructor.csv               (96MB) — Step 4D
✅ train_full_features.parquet                   (83MB) — Step 5
✅ train_full_features_instructor.parquet        (53MB) — Step 5D
❌ results_v6.csv                                — MISSING, need to rerun Step 6 V6
❌ results_v8.csv                                — MISSING, need to rerun Step 6 V8


In [3]:
# ============================================================
# UNIVERSAL PARQUET FIX
# Rebuilds ALL corrupted parquet files from source CSVs
# Run this ONCE before running any other cells
# ============================================================

import os
import numpy as np
import pandas as pd

SAVE_DIR = '/kaggle/working/'

def fix_and_save_parquet(csv_path, parquet_path,
                          meta_cols, sf_cols, extra_cols,
                          eg_cols, emb_dim, emb_csv_path,
                          label):
    """Load CSV embeddings, fix dtypes, save parquet"""
    if not os.path.exists(emb_csv_path):
        print(f"  ⏭️  {label}: embedding CSV missing — skip")
        return False

    print(f"  Building {label}...")
    sf_df  = pd.read_csv(
        os.path.join(SAVE_DIR, 'train_shallow_features.csv'),
        encoding='latin-1'
    )
    eg_df  = pd.read_csv(
        os.path.join(SAVE_DIR,
                     'train_entity_grid_features.csv')
    )
    emb_df = pd.read_csv(emb_csv_path)

    # Fix all numeric columns to float32
    num_cols = (
        ['mean_coherence','min_coherence','std_coherence'] +
        [f'emb_{i}' for i in range(emb_dim)]
    )
    for col in num_cols:
        if col in emb_df.columns:
            emb_df[col] = pd.to_numeric(
                emb_df[col], errors='coerce'
            ).astype(np.float32).fillna(0.0)

    EMB_COLS = (
        ['mean_coherence','min_coherence','std_coherence'] +
        [f'emb_{i}' for i in range(emb_dim)]
    )

    parts = [
        sf_df[meta_cols].reset_index(drop=True),
        sf_df[sf_cols].reset_index(drop=True),
    ]
    if extra_cols:
        extra_path = os.path.join(
            SAVE_DIR, 'train_extra_features.csv'
        )
        if os.path.exists(extra_path):
            extra_df = pd.read_csv(extra_path)
            for col in extra_cols:
                extra_df[col] = pd.to_numeric(
                    extra_df[col], errors='coerce'
                ).astype(np.float32).fillna(0.0)
            parts.append(
                extra_df[extra_cols].reset_index(drop=True)
            )
    parts += [
        eg_df[eg_cols].reset_index(drop=True),
        emb_df[EMB_COLS].reset_index(drop=True),
    ]

    df = pd.concat(parts, axis=1)
    df.to_parquet(parquet_path, index=False)
    print(f"  ✅ {label}: {df.shape} saved")
    return True

# ── Define common column lists ─────────────────────────────
META_COLS = [
    'essay_id','essay_set','essay',
    'domain1_score','rater1_domain1','rater2_domain1'
]
SF_COLS = [
    'gunning_fog','flesch_re','ari','guiraud_index',
    'noun_freq','verb_freq','avg_sent_len',
    'pronoun_density','pronoun_noun_ratio','giveness'
]
EXTRA_COLS = [
    'word_count','spell_error_rate','discourse_rate',
    'para_count','type_token_ratio','avg_word_len',
    'sent_len_std','punct_diversity'
]
EG_COLS = [
    f"{r1}_{r2}"
    for r1 in ['S','O','X','-']
    for r2 in ['S','O','X','-']
]

def is_corrupted(path):
    """Check if parquet file is corrupted"""
    if not os.path.exists(path):
        return False  # doesn't exist, not corrupted
    try:
        pd.read_parquet(path)
        return False  # readable = not corrupted
    except Exception:
        return True   # unreadable = corrupted

print("="*60)
print("  UNIVERSAL PARQUET FIX")
print("="*60)
print("\nChecking all parquet files...\n")

# ── List of all parquet files to check/fix ────────────────
PARQUET_CONFIGS = [
    {
        'label'     : 'BGE original (train_full_features)',
        'parquet'   : 'train_full_features.parquet',
        'emb_csv'   : 'train_embeddings_features.csv',
        'emb_dim'   : 1024,
        'extra_cols': [],
    },
    {
        'label'     : 'INSTRUCTOR (train_full_features_instructor)',
        'parquet'   : 'train_full_features_instructor.parquet',
        'emb_csv'   : 'train_embeddings_instructor.csv',
        'emb_dim'   : 1024,
        'extra_cols': [],
    },
    {
        'label'     : 'E5-Mistral (train_full_features_e5mistral)',
        'parquet'   : 'train_full_features_e5mistral.parquet',
        'emb_csv'   : 'train_embeddings_e5mistral.csv',
        'emb_dim'   : 1024,
        'extra_cols': [],
    },
    {
        'label'     : 'Llama (train_full_features_llama)',
        'parquet'   : 'train_full_features_llama.parquet',
        'emb_csv'   : 'train_embeddings_llama.csv',
        'emb_dim'   : 2048,  # Llama hidden size
        'extra_cols': [],
    },
    {
        'label'     : 'EmbGemma (train_full_features_embgemma)',
        'parquet'   : 'train_full_features_embgemma.parquet',
        'emb_csv'   : 'train_embeddings_embgemma.csv',
        'emb_dim'   : 768,
        'extra_cols': [],
    },
    {
        'label'     : 'E5+Rubric 2048 (train_full_features_e5rubric_2048)',
        'parquet'   : 'train_full_features_e5rubric_2048.parquet',
        'emb_csv'   : 'train_embeddings_e5_rubric_2048.csv',
        'emb_dim'   : 4096,
        'extra_cols': [],
    },
    {
        'label'     : 'INSTRUCTOR+Extra (inst_extra)',
        'parquet'   : 'train_full_features_inst_extra.parquet',
        'emb_csv'   : 'train_embeddings_instructor.csv',
        'emb_dim'   : 1024,
        'extra_cols': EXTRA_COLS,
    },
    {
        'label'     : 'E5Rubric+Extra (e5rubric_extra)',
        'parquet'   : 'train_full_features_e5rubric_extra.parquet',
        'emb_csv'   : 'train_embeddings_e5_rubric_2048.csv',
        'emb_dim'   : 4096,
        'extra_cols': EXTRA_COLS,
    },
    {
        'label'     : 'Qwen2.5 (train_full_features_qwen25)',
        'parquet'   : 'train_full_features_qwen25.parquet',
        'emb_csv'   : 'train_embeddings_qwen25.csv',
        'emb_dim'   : 1536,  # Qwen2.5-1.5B hidden size
        'extra_cols': EXTRA_COLS,
    },
]

fixed   = 0
skipped = 0
ok      = 0

for cfg in PARQUET_CONFIGS:
    ppath = os.path.join(SAVE_DIR, cfg['parquet'])
    ecsv  = os.path.join(SAVE_DIR, cfg['emb_csv'])

    if not os.path.exists(ppath) and \
       not os.path.exists(ecsv):
        print(f"  ⏭️  {cfg['label']}: not needed — skip")
        skipped += 1
        continue

    if os.path.exists(ppath) and not is_corrupted(ppath):
        size = os.path.getsize(ppath)/1024/1024
        print(f"  ✅ {cfg['label']}: OK ({size:.0f}MB)")
        ok += 1
        continue

    if is_corrupted(ppath):
        print(f"  ⚠️  {cfg['label']}: CORRUPTED — rebuilding...")
        os.remove(ppath)
    elif not os.path.exists(ppath):
        print(f"  ⚠️  {cfg['label']}: MISSING — building...")

    # Detect actual embedding dim from CSV
    if os.path.exists(ecsv):
        sample = pd.read_csv(ecsv, nrows=1)
        actual_dim = len([
            c for c in sample.columns
            if c.startswith('emb_')
        ])
        if actual_dim > 0 and actual_dim != cfg['emb_dim']:
            print(f"     Dim mismatch: config={cfg['emb_dim']}"
                  f" actual={actual_dim} → using {actual_dim}")
            cfg['emb_dim'] = actual_dim

    success = fix_and_save_parquet(
        csv_path   = ecsv,
        parquet_path = ppath,
        meta_cols  = META_COLS,
        sf_cols    = SF_COLS,
        extra_cols = cfg['extra_cols'],
        eg_cols    = EG_COLS,
        emb_dim    = cfg['emb_dim'],
        emb_csv_path = ecsv,
        label      = cfg['label']
    )
    if success:
        fixed += 1

print(f"\n{'='*60}")
print(f"  SUMMARY")
print(f"{'='*60}")
print(f"  OK (already valid):  {ok}")
print(f"  Fixed/Built:         {fixed}")
print(f"  Skipped (no data):   {skipped}")

# ── Final verification ─────────────────────────────────────
print(f"\n  Verification:")
for cfg in PARQUET_CONFIGS:
    ppath = os.path.join(SAVE_DIR, cfg['parquet'])
    if os.path.exists(ppath):
        try:
            df   = pd.read_parquet(ppath)
            size = os.path.getsize(ppath)/1024/1024
            print(f"  ✅ {cfg['parquet']:<50} "
                  f"{df.shape} {size:.0f}MB")
        except Exception as e:
            print(f"  ❌ {cfg['parquet']}: {e}")
    else:
        print(f"  ⏭️  {cfg['parquet']}: not present")

print(f"\n✅ All done! Now run your V24/V25 cells.")
print("⚠️  Click Save Version NOW!")

  UNIVERSAL PARQUET FIX

Checking all parquet files...

  ✅ BGE original (train_full_features): OK (83MB)
  ✅ INSTRUCTOR (train_full_features_instructor): OK (53MB)
  ✅ E5-Mistral (train_full_features_e5mistral): OK (57MB)
  ✅ Llama (train_full_features_llama): OK (47MB)
  ✅ EmbGemma (train_full_features_embgemma): OK (60MB)
  ✅ E5+Rubric 2048 (train_full_features_e5rubric_2048): OK (74MB)
  ✅ INSTRUCTOR+Extra (inst_extra): OK (84MB)
  ✅ E5Rubric+Extra (e5rubric_extra): OK (47MB)
  ⏭️  Qwen2.5 (train_full_features_qwen25): not needed — skip

  SUMMARY
  OK (already valid):  8
  Fixed/Built:         0
  Skipped (no data):   1

  Verification:
  ✅ train_full_features.parquet                        (12976, 1059) 83MB
  ✅ train_full_features_instructor.parquet             (12976, 1059) 53MB
  ✅ train_full_features_e5mistral.parquet              (12976, 1059) 57MB
  ✅ train_full_features_llama.parquet                  (12976, 2083) 47MB
  ✅ train_full_features_embgemma.parquet              

In [4]:
# ============================================================
# CELL 1 — Install ALL packages (run after every restart)
# ============================================================

!pip install sentence-transformers InstructorEmbedding textstat xgboost -q

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('stopwords', quiet=True)

print("✅ All packages installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 39.9 MB/s eta 0:00:0000:01
✅ All packages installed!


In [5]:
# ============================================================
# STEP 1: Install Libraries & Load Data
# ============================================================

!pip install sentence-transformers textstat -q

import pandas as pd
import numpy as np
import nltk
import warnings
from sklearn.metrics import cohen_kappa_score
warnings.filterwarnings('ignore')

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('stopwords')

# --- Find your file path first ---
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/nehasatheesh/asag-dataset/training_set.csv
/kaggle/input/datasets/nehasatheesh/asag-outputs/X_selected_processed.csv
/kaggle/input/datasets/nehasatheesh/malayalam-dataset/questions_150.csv
/kaggle/input/datasets/nehasatheesh/malayalam-dataset/mal_dataset.csv


[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [6]:
# --- Load YOUR training_set.csv ---
# Replace the path below with what was printed above
train_df = pd.read_csv(
    '/kaggle/input/datasets/nehasatheesh/asag-dataset/training_set.csv',
    encoding='latin-1'
)

# --- Keep relevant columns ---
train_df = train_df[[
    'essay_id',
    'essay_set',
    'essay',
    'domain1_score',
    'rater1_domain1',
    'rater2_domain1'
]]

# --- Sanity check ---
print("Shape:", train_df.shape)
print("\nNull values:\n", train_df.isnull().sum())
print("\nScore range per essay set:")
print(train_df.groupby('essay_set')['domain1_score'].agg(['min', 'max', 'count']))

# --- Human-Human QWK (upper bound for our model) ---
print("\n--- Human-Human QWK (upper bound) ---")
for s in sorted(train_df['essay_set'].unique()):
    subset = train_df[train_df['essay_set'] == s].dropna(
        subset=['rater1_domain1', 'rater2_domain1']
    )
    qwk = cohen_kappa_score(
        subset['rater1_domain1'].astype(int),
        subset['rater2_domain1'].astype(int),
        weights='quadratic'
    )
    print(f"  Set {s}: {qwk:.3f}")

Shape: (12976, 6)

Null values:
 essay_id          0
essay_set         0
essay             0
domain1_score     0
rater1_domain1    0
rater2_domain1    0
dtype: int64

Score range per essay set:
           min  max  count
essay_set                 
1            2   12   1783
2            1    6   1800
3            0    3   1726
4            0    3   1770
5            0    4   1805
6            0    4   1800
7            2   24   1569
8           10   60    723

--- Human-Human QWK (upper bound) ---
  Set 1: 0.721
  Set 2: 0.814
  Set 3: 0.769
  Set 4: 0.851
  Set 5: 0.753
  Set 6: 0.776
  Set 7: 0.721
  Set 8: 0.624


In [7]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/nehasatheesh/asag-dataset/training_set.csv
/kaggle/input/datasets/nehasatheesh/asag-outputs/X_selected_processed.csv
/kaggle/input/datasets/nehasatheesh/malayalam-dataset/questions_150.csv
/kaggle/input/datasets/nehasatheesh/malayalam-dataset/mal_dataset.csv


In [8]:
# ============================================================
# STEP 2: Shallow Feature Extraction (GPU + Save Outputs)
# ============================================================

import textstat
import spacy
import re
import os
import numpy as np
import pandas as pd
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from tqdm import tqdm

# --- Enable spaCy GPU ---
spacy.prefer_gpu()
nlp = spacy.load('en_core_web_sm')
stop_words = set(stopwords.words('english'))

SAVE_DIR = '/kaggle/working/'
SF_SAVE_PATH = os.path.join(SAVE_DIR, 'train_shallow_features.csv')

# ── Skip if already computed ──────────────────────────────────
if os.path.exists(SF_SAVE_PATH):
    print("✅ Shallow features already saved — loading from disk!")
    train_df = pd.read_csv(SF_SAVE_PATH, encoding='latin-1')
    print("Shape:", train_df.shape)
    print(train_df.head(3))

else:
    print("Computing shallow features from scratch...")

    # ------------------------------------------------------------
    # 2.1 READABILITY
    # ------------------------------------------------------------
    def get_readability_features(text):
        try:
            fog = textstat.gunning_fog(text)
            fre = textstat.flesch_reading_ease(text)
            ari = textstat.automated_readability_index(text)
        except:
            fog, fre, ari = 0, 0, 0
        return fog, fre, ari

    # ------------------------------------------------------------
    # 2.2 LEXICAL DIVERSITY — Guiraud Index
    # ------------------------------------------------------------
    def get_lexical_diversity(text):
        try:
            words = word_tokenize(text.lower())
            words = [w for w in words if w.isalpha()]
            if len(words) == 0:
                return 0
            return len(set(words)) / (len(words) ** 0.5)
        except:
            return 0

    # ------------------------------------------------------------
    # 2.3 GRAMMAR FEATURES
    # ------------------------------------------------------------
    def get_grammar_features(doc):
        try:
            words = [t for t in doc if t.is_alpha]
            if len(words) == 0:
                return 0, 0, 0
            noun_freq    = len([t for t in doc if t.pos_ == 'NOUN']) / len(words)
            verb_freq    = len([t for t in doc if t.pos_ == 'VERB']) / len(words)
            sents        = list(doc.sents)
            avg_sent_len = len(words) / len(sents) if sents else 0
        except:
            noun_freq, verb_freq, avg_sent_len = 0, 0, 0
        return noun_freq, verb_freq, avg_sent_len

    # ------------------------------------------------------------
    # 2.4 COMPLEXITY FEATURES
    # ------------------------------------------------------------
    def get_complexity_features(doc):
        try:
            words = [t for t in doc if t.is_alpha]
            if len(words) == 0:
                return 0, 0, 0
            pronouns = [t for t in doc if t.pos_ == 'PRON']
            nouns    = [t for t in doc if t.pos_ == 'NOUN']

            pronoun_density    = len(pronouns) / len(words)
            pronoun_noun_ratio = len(pronouns) / len(nouns) if nouns else 0

            sents = [s.text for s in doc.sents]
            if len(sents) < 2:
                giveness = 0
            else:
                overlaps = []
                for i in range(1, len(sents)):
                    prev_w = set(sents[i-1].lower().split()) - stop_words
                    curr_w = set(sents[i].lower().split()) - stop_words
                    if not curr_w:
                        overlaps.append(0)
                    else:
                        overlaps.append(len(prev_w & curr_w) / len(curr_w))
                giveness = 1 - np.mean(overlaps)
        except:
            pronoun_density, pronoun_noun_ratio, giveness = 0, 0, 0
        return pronoun_density, pronoun_noun_ratio, giveness

    # ------------------------------------------------------------
    # 2.5 BATCH PROCESSING WITH spaCy pipe (GPU-accelerated)
    # ------------------------------------------------------------
    SHALLOW_FEATURE_NAMES = [
        'gunning_fog', 'flesch_re', 'ari',
        'guiraud_index',
        'noun_freq', 'verb_freq', 'avg_sent_len',
        'pronoun_density', 'pronoun_noun_ratio', 'giveness'
    ]

    texts   = train_df['essay'].astype(str).tolist()
    records = []

    # nlp.pipe processes in batches on GPU — much faster than one by one
    print("Running spaCy GPU pipeline (batch_size=64)...")
    for i, doc in enumerate(tqdm(
        nlp.pipe(texts, batch_size=64),
        total=len(texts)
    )):
        text = texts[i]
        fog, fre, ari      = get_readability_features(text)
        guiraud            = get_lexical_diversity(text)
        noun_f, verb_f, asl = get_grammar_features(doc)
        p_den, pn_r, given = get_complexity_features(doc)

        records.append([
            fog, fre, ari,
            guiraud,
            noun_f, verb_f, asl,
            p_den, pn_r, given
        ])

    sf_df    = pd.DataFrame(records, columns=SHALLOW_FEATURE_NAMES)
    train_df = pd.concat([train_df.reset_index(drop=True), sf_df], axis=1)

    # ------------------------------------------------------------
    # 2.6 SAVE TO DISK
    # ------------------------------------------------------------
    train_df.to_csv(SF_SAVE_PATH, index=False)
    print(f"\n💾 Saved to {SF_SAVE_PATH}")

    # ------------------------------------------------------------
    # 2.7 SANITY CHECK
    # ------------------------------------------------------------
    print("\nShallow features sample (first 3 rows):")
    print(train_df[SHALLOW_FEATURE_NAMES].head(3).round(3))
    print("\nFeature statistics:")
    print(train_df[SHALLOW_FEATURE_NAMES].describe().round(3))
    print("\nAny nulls?", train_df[SHALLOW_FEATURE_NAMES].isnull().sum().sum())
    print("\n✅ Step 2 Complete — 10 shallow features extracted!")


✅ Shallow features already saved — loading from disk!
Shape: (12976, 16)
   essay_id  essay_set                                              essay  \
0         1          1  Dear local newspaper, I think effects computer...   
1         2          1  Dear @CAPS1 @CAPS2, I believe that using compu...   
2         3          1  Dear, @CAPS1 @CAPS2 @CAPS3 More and more peopl...   

   domain1_score  rater1_domain1  rater2_domain1  gunning_fog  flesch_re  \
0              8               4               4    11.479248  68.553589   
1              9               5               4    12.485012  64.021347   
2              7               4               3    12.702611  62.588145   

         ari  guiraud_index  noun_freq  verb_freq  avg_sent_len  \
0  11.235227       8.418501   0.238235   0.150000     21.250000   
1  10.065764       9.109357   0.248780   0.165854     19.523810   
2   9.855899       8.488747   0.272059   0.139706     18.133333   

   pronoun_density  pronoun_noun_ratio  give

In [9]:
# ============================================================
# STEP 3: Entity Grid — Discourse Pattern Features
# ============================================================

import os
import numpy as np
import pandas as pd
import spacy
from tqdm import tqdm

spacy.prefer_gpu()
nlp = spacy.load('en_core_web_sm')

SAVE_DIR  = '/kaggle/working/'
EG_SAVE_PATH = os.path.join(SAVE_DIR, 'train_entity_grid_features.csv')

# ── Skip if already computed ──────────────────────────────────
if os.path.exists(EG_SAVE_PATH):
    print("✅ Entity grid features already saved — loading from disk!")
    eg_df = pd.read_csv(EG_SAVE_PATH)
    print("Shape:", eg_df.shape)
    print(eg_df.head(3))

else:
    print("Computing entity grid features from scratch...")

    # ----------------------------------------------------------
    # 3.1 ASSIGN GRAMMATICAL ROLE TO EACH TOKEN
    # Role: S (Subject), O (Object), X (Other), - (Absent)
    # ----------------------------------------------------------
    def get_entity_role(token):
        """
        Returns grammatical role of a noun token in its sentence:
        S → subject,  O → object,  X → mentioned but neither
        """
        if token.dep_ in ('nsubj', 'nsubjpass', 'csubj', 'csubjpass'):
            return 'S'
        elif token.dep_ in ('dobj', 'pobj', 'iobj', 'obj', 'obl'):
            return 'O'
        else:
            return 'X'

    # ----------------------------------------------------------
    # 3.2 BUILD ENTITY GRID FOR ONE ESSAY
    # ----------------------------------------------------------
    def build_entity_grid(doc):
        """
        Returns a dict:
          { entity_lemma : [role_in_sent1, role_in_sent2, ...] }
        where role ∈ {S, O, X, -}
        """
        sentences  = list(doc.sents)
        num_sents  = len(sentences)

        if num_sents < 2:
            return {}

        # Collect all noun/propnoun entities across essay
        all_entities = set()
        for token in doc:
            if token.pos_ in ('NOUN', 'PROPN') and token.is_alpha:
                all_entities.add(token.lemma_.lower())

        if not all_entities:
            return {}

        # Build grid: entity → list of roles per sentence
        grid = {e: ['-'] * num_sents for e in all_entities}

        for s_idx, sent in enumerate(sentences):
            for token in sent:
                if token.pos_ in ('NOUN', 'PROPN') and token.is_alpha:
                    lemma = token.lemma_.lower()
                    role  = get_entity_role(token)
                    # Keep highest priority role (S > O > X > -)
                    current = grid[lemma][s_idx]
                    priority = {'-': 0, 'X': 1, 'O': 2, 'S': 3}
                    if priority[role] > priority[current]:
                        grid[lemma][s_idx] = role

        return grid

    # ----------------------------------------------------------
    # 3.3 COMPUTE 16 TRANSITION PROBABILITIES FROM GRID
    # ----------------------------------------------------------
    ROLES       = ['S', 'O', 'X', '-']
    TRANSITIONS = [f"{r1}_{r2}" for r1 in ROLES for r2 in ROLES]
    # → ['S_S','S_O','S_X','S_-','O_S','O_O',...,'-_-'] = 16 features

    def compute_transition_probs(grid):
        """
        For each pair of adjacent sentences, count how each
        entity transitions in role. Normalize to probabilities.
        Returns a 16-dim vector.
        """
        counts = {t: 0 for t in TRANSITIONS}
        total  = 0

        for entity, roles in grid.items():
            for i in range(len(roles) - 1):
                key = f"{roles[i]}_{roles[i+1]}"
                counts[key] += 1
                total += 1

        # Normalize
        if total == 0:
            return [0.0] * 16

        return [counts[t] / total for t in TRANSITIONS]

    # ----------------------------------------------------------
    # 3.4 MASTER FUNCTION
    # ----------------------------------------------------------
    def extract_entity_grid_features(doc):
        grid  = build_entity_grid(doc)
        if not grid:
            return [0.0] * 16
        probs = compute_transition_probs(grid)
        return probs

    # ----------------------------------------------------------
    # 3.5 APPLY TO ALL ESSAYS — GPU batch processing
    # ----------------------------------------------------------
    # Load train_df from Step 2 output (safe from timeout)
    train_df = pd.read_csv(
        os.path.join(SAVE_DIR, 'train_shallow_features.csv'),
        encoding='latin-1'
    )
    texts = train_df['essay'].astype(str).tolist()

    print(f"Building entity grids for {len(texts)} essays...")
    print("(Expected time: ~4-6 mins on Kaggle GPU)")

    eg_records = []
    for doc in tqdm(
        nlp.pipe(texts, batch_size=64),
        total=len(texts)
    ):
        eg_records.append(extract_entity_grid_features(doc))

    eg_df = pd.DataFrame(eg_records, columns=TRANSITIONS)

    # ----------------------------------------------------------
    # 3.6 SAVE TO DISK
    # ----------------------------------------------------------
    eg_df.to_csv(EG_SAVE_PATH, index=False)
    print(f"\n💾 Saved to {EG_SAVE_PATH}")

    # ----------------------------------------------------------
    # 3.7 SANITY CHECK
    # ----------------------------------------------------------
    print("\nEntity grid feature sample (first 3 rows):")
    print(eg_df.head(3).round(4))

    print("\nMost common transitions (mean probability):")
    print(eg_df.mean().sort_values(ascending=False).head(8).round(4))

    print("\nAny nulls?", eg_df.isnull().sum().sum())
    print(f"\n✅ Step 3 Complete — 16 entity grid features extracted!")



✅ Entity grid features already saved — loading from disk!
Shape: (12976, 16)
        S_S       S_O  S_X       S_-       O_S       O_O       O_X       O_-  \
0  0.001361  0.002721  0.0  0.012245  0.001361  0.004082  0.001361  0.046259   
1  0.000909  0.002727  0.0  0.008182  0.003636  0.005455  0.000000  0.037273   
2  0.011278  0.003759  0.0  0.011278  0.005639  0.013158  0.000000  0.052632   

        X_S       X_O  X_X       X_-       -_S       -_O       -_X       -_-  
0  0.001361  0.001361  0.0  0.035374  0.009524  0.040816  0.029932  0.812245  
1  0.000000  0.000000  0.0  0.020000  0.008182  0.042727  0.017273  0.853636  
2  0.000000  0.000000  0.0  0.020677  0.009398  0.054511  0.018797  0.798872  


In [10]:
# ============================================================
# STEP 4: Sentence Embeddings — Local Coherence Features
#         Model: BAAI/bge-large-en-v1.5 (replaces GPT-3)
# ============================================================

import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.tokenize import sent_tokenize

SAVE_DIR      = '/kaggle/working/'
EMB_SAVE_PATH = os.path.join(SAVE_DIR, 'train_embeddings_features.csv')

# ── Skip if already computed ─────────────────────────────────
if os.path.exists(EMB_SAVE_PATH):
    print("✅ Embedding features already saved — loading from disk!")
    emb_df = pd.read_csv(EMB_SAVE_PATH)
    print("Shape:", emb_df.shape)
    print(emb_df[['mean_coherence','min_coherence','std_coherence']].head(3))

else:
    print("Computing embedding features from scratch...")

    # ----------------------------------------------------------
    # 4.1 LOAD bge-large-en-v1.5 ONTO GPU
    # ----------------------------------------------------------
    print("Loading BAAI/bge-large-en-v1.5 onto GPU...")
    model = SentenceTransformer('BAAI/bge-large-en-v1.5', device='cuda')

    # bge models need this prefix for best similarity performance
    BGE_PREFIX = "Represent this sentence for semantic similarity: "

    print("✅ Model loaded!")
    print(f"   Embedding dimension: {model.get_sentence_embedding_dimension()}")

    # ----------------------------------------------------------
    # 4.2 COHERENCE FEATURES FROM EMBEDDINGS
    # ----------------------------------------------------------
    def get_embedding_features(sentences, embeddings):
        """
        Given sentence embeddings for one essay, compute:
        1. mean_coherence → avg cosine sim between adjacent sentences
        2. min_coherence  → worst adjacent pair (sudden topic jump)
        3. std_coherence  → consistency of flow
        + mean_embedding  → 1024-dim average essay vector
                            (bge-large outputs 1024-dim, vs 768 for mpnet)
        """
        if len(embeddings) < 2:
            zero = np.zeros(model.get_sentence_embedding_dimension())
            return {
                'mean_coherence': 0.0,
                'min_coherence' : 0.0,
                'std_coherence' : 0.0,
            }, embeddings[0] if len(embeddings) == 1 else zero

        # Adjacent cosine similarities
        similarities = []
        for i in range(len(embeddings) - 1):
            sim = cosine_similarity(
                embeddings[i].reshape(1, -1),
                embeddings[i+1].reshape(1, -1)
            )[0][0]
            similarities.append(float(sim))

        coherence_features = {
            'mean_coherence': float(np.mean(similarities)),
            'min_coherence' : float(np.min(similarities)),
            'std_coherence' : float(np.std(similarities)),
        }

        mean_emb = np.mean(embeddings, axis=0)
        return coherence_features, mean_emb

    # ----------------------------------------------------------
    # 4.3 PROCESS ALL ESSAYS IN BATCHES (GPU)
    # ----------------------------------------------------------
    train_df = pd.read_csv(
        os.path.join(SAVE_DIR, 'train_shallow_features.csv'),
        encoding='latin-1'
    )
    texts = train_df['essay'].astype(str).tolist()

    EMB_DIM = model.get_sentence_embedding_dimension()  # 1024 for bge-large

    print(f"\nProcessing {len(texts)} essays...")
    print(f"Embedding dimension: {EMB_DIM}")
    print("(Expected time: ~5-7 mins on Kaggle GPU)\n")

    all_coherence_records = []
    all_mean_embeddings   = []

    for text in tqdm(texts, total=len(texts)):
        sentences = sent_tokenize(text)
        if not sentences:
            sentences = [text]

        # Add BGE prefix to each sentence for best performance
        prefixed = [BGE_PREFIX + s for s in sentences]

        # Encode all sentences of this essay in one GPU batch
        embeddings = model.encode(
            prefixed,
            batch_size=64,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True   # bge recommends normalizing
        )

        coherence_feats, mean_emb = get_embedding_features(sentences, embeddings)

        all_coherence_records.append(coherence_feats)
        all_mean_embeddings.append(mean_emb)

    # ----------------------------------------------------------
    # 4.4 BUILD FEATURE DATAFRAME
    # ----------------------------------------------------------
    # Coherence features (3 scalar features)
    coherence_df = pd.DataFrame(all_coherence_records)

    # Mean embedding vectors (1024 features for bge-large)
    emb_cols    = [f'emb_{i}' for i in range(EMB_DIM)]
    mean_emb_df = pd.DataFrame(
        np.array(all_mean_embeddings),
        columns=emb_cols
    )

    # Combine: 3 coherence + 1024 embedding = 1027 features total
    emb_df = pd.concat([coherence_df, mean_emb_df], axis=1)

    # ----------------------------------------------------------
    # 4.5 SAVE TO DISK
    # ----------------------------------------------------------
    emb_df.to_csv(EMB_SAVE_PATH, index=False)
    print(f"\n💾 Saved to {EMB_SAVE_PATH}")

    # ----------------------------------------------------------
    # 4.6 SANITY CHECK
    # ----------------------------------------------------------
    print("\nCoherence features (first 3 essays):")
    print(coherence_df.head(3).round(4))

    print("\nCoherence statistics across all essays:")
    print(coherence_df.describe().round(4))

    print(f"\nTotal features shape: {emb_df.shape}")
    print(f"  → 3 coherence + {EMB_DIM} embedding dims = {3 + EMB_DIM} total")
    print("Any nulls?", emb_df.isnull().sum().sum())
    print("\n✅ Step 4 Complete — embedding features extracted!")


✅ Embedding features already saved — loading from disk!
Shape: (12976, 1027)
   mean_coherence  min_coherence  std_coherence
0        0.728876       0.544336       0.067404
1        0.780701       0.672999       0.044355
2        0.767698       0.683808       0.043705


In [11]:
# ============================================================
# STEP 5: Assemble Final Feature Matrix
# ============================================================

import os
import numpy as np
import pandas as pd

SAVE_DIR        = '/kaggle/working/'
FULL_SAVE_PATH  = os.path.join(SAVE_DIR, 'train_full_features.csv')
PARQUET_PATH    = os.path.join(SAVE_DIR, 'train_full_features.parquet')

# ── Skip if already computed ─────────────────────────────────
if os.path.exists(PARQUET_PATH):
    print("✅ Full feature matrix already saved — loading parquet!")
    train_full = pd.read_parquet(PARQUET_PATH)
elif os.path.exists(FULL_SAVE_PATH):
    print("✅ Full feature matrix already saved — loading CSV!")
    train_full = pd.read_csv(FULL_SAVE_PATH)
    print("Shape:", train_full.shape)

else:
    print("Assembling full feature matrix...")

    # ----------------------------------------------------------
    # 5.1 LOAD ALL THREE FEATURE SETS FROM DISK
    # ----------------------------------------------------------
    print("\nLoading Step 2 — Shallow Features...")
    sf_df = pd.read_csv(
        os.path.join(SAVE_DIR, 'train_shallow_features.csv'),
        encoding='latin-1'
    )

    print("Loading Step 3 — Entity Grid Features...")
    eg_df = pd.read_csv(
        os.path.join(SAVE_DIR, 'train_entity_grid_features.csv')
    )

    print("Loading Step 4 — Embedding Features...")
    emb_df = pd.read_csv(
        os.path.join(SAVE_DIR, 'train_embeddings_features.csv')
    )

    # ----------------------------------------------------------
    # 5.2 DEFINE FEATURE COLUMN NAMES
    # ----------------------------------------------------------
    SF_COLS  = [
        'gunning_fog', 'flesch_re', 'ari',
        'guiraud_index',
        'noun_freq', 'verb_freq', 'avg_sent_len',
        'pronoun_density', 'pronoun_noun_ratio', 'giveness'
    ]

    ROLES       = ['S', 'O', 'X', '-']
    EG_COLS     = [f"{r1}_{r2}" for r1 in ROLES for r2 in ROLES]

    EMB_SCALAR_COLS = ['mean_coherence', 'min_coherence', 'std_coherence']
    EMB_VECTOR_COLS = [f'emb_{i}' for i in range(1024)]
    EMB_COLS        = EMB_SCALAR_COLS + EMB_VECTOR_COLS

    # ----------------------------------------------------------
    # 5.3 ASSEMBLE — Paper's exact feature combination
    #
    #  Layer 1: Shallow Features          (10 features)
    #  Layer 2: Entity Grid / Discourse   (16 features)
    #  Layer 3: BGE Embeddings            (1027 features)
    #  ─────────────────────────────────────────────────
    #  Total                              (1053 features)
    # ----------------------------------------------------------

    # Meta columns (not features — used for grouping & target)
    META_COLS = ['essay_id', 'essay_set', 'essay',
                 'domain1_score', 'rater1_domain1', 'rater2_domain1']

    train_full = pd.concat([
        sf_df[META_COLS],           # identifiers + target
        sf_df[SF_COLS],             # Layer 1: 10 shallow features
        eg_df[EG_COLS],             # Layer 2: 16 entity grid features
        emb_df[EMB_COLS],           # Layer 3: 1027 embedding features
    ], axis=1)

    # ----------------------------------------------------------
    # 5.4 SAVE TO DISK
    # ----------------------------------------------------------
    train_full.to_csv(FULL_SAVE_PATH, index=False)
    print(f"\n💾 Saved CSV to {FULL_SAVE_PATH}")

    # Also save as parquet (needed by V3, V4, V9)
    PARQUET_PATH = os.path.join(SAVE_DIR, 'train_full_features.parquet')
    # Convert emb columns to float32 to avoid parquet issues
    emb_cols = [c for c in train_full.columns if c.startswith('emb_') or
                c in ['mean_coherence','min_coherence','std_coherence']]
    for col in emb_cols:
        train_full[col] = pd.to_numeric(train_full[col], errors='coerce').astype('float32').fillna(0.0)
    train_full.to_parquet(PARQUET_PATH, index=False)
    print(f"💾 Saved Parquet to {PARQUET_PATH}")

# ----------------------------------------------------------
# 5.5 SANITY CHECK
# ----------------------------------------------------------
ALL_FEATURE_COLS = [
    'gunning_fog', 'flesch_re', 'ari',
    'guiraud_index',
    'noun_freq', 'verb_freq', 'avg_sent_len',
    'pronoun_density', 'pronoun_noun_ratio', 'giveness',
] + [
    f"{r1}_{r2}" for r1 in ['S','O','X','-'] for r2 in ['S','O','X','-']
] + [
    'mean_coherence', 'min_coherence', 'std_coherence'
] + [f'emb_{i}' for i in range(1024)]

print("\n" + "="*55)
print("        FINAL FEATURE MATRIX SUMMARY")
print("="*55)
print(f"  Total essays       : {len(train_full)}")
print(f"  Total columns      : {train_full.shape[1]}")
print(f"\n  Feature breakdown:")
print(f"    Shallow features  :  10  (readability, grammar, complexity)")
print(f"    Entity grid       :  16  (discourse transition probs)")
print(f"    BGE coherence     :   3  (mean, min, std similarity)")
print(f"    BGE embeddings    : 1024 (semantic essay representation)")
print(f"    ─────────────────────────────────────")
print(f"    TOTAL FEATURES    : {len(ALL_FEATURE_COLS)}")
print(f"\n  Target column      : domain1_score")
print(f"  Essay sets         : {sorted(train_full['essay_set'].unique())}")
print(f"\n  Any nulls in features? "
      f"{train_full[ALL_FEATURE_COLS].isnull().sum().sum()}")

print("\nFeature matrix sample (first 3 rows, first 15 cols):")
print(train_full[ALL_FEATURE_COLS[:15]].head(3).round(4))

print("\nEssay set distribution:")
print(train_full.groupby('essay_set')['domain1_score']
      .agg(['count','min','max']))
print("\n✅ Step 5 Complete — Final feature matrix ready!")



✅ Full feature matrix already saved — loading parquet!

        FINAL FEATURE MATRIX SUMMARY
  Total essays       : 12976
  Total columns      : 1059

  Feature breakdown:
    Shallow features  :  10  (readability, grammar, complexity)
    Entity grid       :  16  (discourse transition probs)
    BGE coherence     :   3  (mean, min, std similarity)
    BGE embeddings    : 1024 (semantic essay representation)
    ─────────────────────────────────────
    TOTAL FEATURES    : 1053

  Target column      : domain1_score
  Essay sets         : [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]

  Any nulls in features? 0

Feature matrix sample (first 3 rows, first 15 cols):
   gunning_fog  flesch_re      ari  guiraud_index  noun_freq  verb_freq  \
0      11.4792    68.5536  11.2352         8.4185     0.2382     0.1500   
1      12.4850    64.0213  10.0658         9.1094     0.2488     0.1659   
2      12.7026    62.5881   9.8559         8

In [12]:
# ============================================================
# STEP 6 V3: RF Regressor — NO PCA, NO Scaling
#            Trees handle high dimensions natively
# ============================================================

import os
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import cohen_kappa_score
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

SAVE_DIR          = '/kaggle/working/'
RESULTS_SAVE_PATH = os.path.join(SAVE_DIR, 'results_per_essay_set_v3.csv')

# ----------------------------------------------------------
# 6.1 LOAD
# ----------------------------------------------------------
print("Loading feature matrix...")
train_full = pd.read_parquet(
    os.path.join(SAVE_DIR, 'train_full_features.parquet')
)
print(f"✅ Loaded: {train_full.shape}")

# ----------------------------------------------------------
# 6.2 FEATURE COLUMNS
# ----------------------------------------------------------
SF_COLS = [
    'gunning_fog', 'flesch_re', 'ari',
    'guiraud_index',
    'noun_freq', 'verb_freq', 'avg_sent_len',
    'pronoun_density', 'pronoun_noun_ratio', 'giveness'
]
EG_COLS = [
    f"{r1}_{r2}"
    for r1 in ['S','O','X','-']
    for r2 in ['S','O','X','-']
]
EMB_COLS = (
    ['mean_coherence', 'min_coherence', 'std_coherence'] +
    [f'emb_{i}' for i in range(1024)]
)
ALL_FEATURE_COLS = SF_COLS + EG_COLS + EMB_COLS

# ----------------------------------------------------------
# 6.3 METRICS
# ----------------------------------------------------------
def qwk(y_true, y_pred):
    return cohen_kappa_score(y_true, y_pred, weights='quadratic')

def exact_agreement(y_true, y_pred):
    return np.mean(np.array(y_true) == np.array(y_pred))

def adjacent_agreement(y_true, y_pred):
    return np.mean(np.abs(np.array(y_true) - np.array(y_pred)) <= 1)

def spearman_r(y_true, y_pred):
    r, _ = spearmanr(y_true, y_pred)
    return r

def clip_round(y_pred, y_min, y_max):
    return np.clip(np.round(y_pred).astype(int), y_min, y_max)

# ----------------------------------------------------------
# 6.4 SCORE RANGES
# ----------------------------------------------------------
SCORE_RANGES = {
    1: (2,  12),
    2: (1,   6),
    3: (0,   3),
    4: (0,   3),
    5: (0,   4),
    6: (0,   4),
    7: (2,  24),
    8: (10, 60),
}

# ----------------------------------------------------------
# 6.5 10-FOLD CV — NO PCA, NO SCALING
# ----------------------------------------------------------
print("\n" + "="*65)
print("   10-FOLD CV — RF REGRESSOR (no PCA, no scaling)")
print("="*65)

all_results = []

for essay_set in sorted(train_full['essay_set'].unique()):

    subset = train_full[
        train_full['essay_set'] == essay_set
    ].reset_index(drop=True)

    X      = subset[ALL_FEATURE_COLS].values.astype(np.float32)
    y      = subset['domain1_score'].values.astype(np.float32)
    y_min, y_max = SCORE_RANGES[essay_set]

    print(f"\n── Essay Set {essay_set} "
          f"({len(subset)} essays, scores {y_min}–{y_max}) ──")

    kf = KFold(n_splits=10, shuffle=True, random_state=42)
    fold_qwk, fold_ea, fold_aa, fold_sp = [], [], [], []

    for fold_num, (train_idx, test_idx) in enumerate(kf.split(X), 1):

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # ── Pure RF Regressor — no preprocessing ──────────
        # RF is a tree model: doesn't need scaling or PCA
        # It naturally handles 1053 mixed features
        rf = RandomForestRegressor(
            n_estimators=300,
            max_features='sqrt',    # sqrt(1053) ≈ 32 features per split
            min_samples_leaf=2,
            max_depth=None,         # let trees grow fully
            random_state=42,
            n_jobs=-1
        )
        rf.fit(X_train, y_train)

        # Predict → round → clip to valid range
        y_pred = clip_round(rf.predict(X_test), y_min, y_max)
        y_true = y_test.astype(int)

        fold_qwk.append(qwk(y_true, y_pred))
        fold_ea.append(exact_agreement(y_true, y_pred))
        fold_aa.append(adjacent_agreement(y_true, y_pred))
        fold_sp.append(spearman_r(y_true, y_pred))

        print(f"  Fold {fold_num:2d} → "
              f"QWK={fold_qwk[-1]:.3f}  "
              f"EA={fold_ea[-1]:.3f}  "
              f"AA={fold_aa[-1]:.3f}  "
              f"Spearman={fold_sp[-1]:.3f}")

    mean_qwk = np.mean(fold_qwk)
    mean_ea  = np.mean(fold_ea)
    mean_aa  = np.mean(fold_aa)
    mean_sp  = np.mean(fold_sp)

    print(f"  {'─'*52}")
    print(f"  MEAN    → "
          f"QWK={mean_qwk:.3f}  "
          f"EA={mean_ea:.3f}  "
          f"AA={mean_aa:.3f}  "
          f"Spearman={mean_sp:.3f}")

    all_results.append({
        'essay_set'    : essay_set,
        'n_essays'     : len(subset),
        'score_range'  : f"{y_min}–{y_max}",
        'mean_qwk'     : round(mean_qwk, 3),
        'mean_ea'      : round(mean_ea,  3),
        'mean_aa'      : round(mean_aa,  3),
        'mean_spearman': round(mean_sp,  3),
    })

    # ── Checkpoint after every essay set ──────────────────
    pd.DataFrame(all_results).to_csv(RESULTS_SAVE_PATH, index=False)
    print(f"  💾 Checkpoint saved (set {essay_set} done)")

# ----------------------------------------------------------
# 6.6 FINAL COMPARISON TABLE
# ----------------------------------------------------------
results_df = pd.DataFrame(all_results)

paper_qwk = {
    1: 0.91, 2: 0.79, 3: 0.71,
    4: 0.84, 5: 0.84, 6: 0.84,
    7: 0.87, 8: 0.92
}
paper_avg = np.mean(list(paper_qwk.values()))

print("\n" + "="*65)
print("   RESULTS vs PAPER (Table 4) — QWK")
print("="*65)
print(f"{'Set':<6} {'Ours':>8} {'Paper':>8} {'Diff':>8}")
print("─"*38)

our_qwks = []
for _, row in results_df.iterrows():
    s       = int(row['essay_set'])
    our_qwk = row['mean_qwk']
    pap_qwk = paper_qwk[s]
    diff    = our_qwk - pap_qwk
    our_qwks.append(our_qwk)
    flag    = "✅" if our_qwk >= pap_qwk - 0.05 else "⚠️"
    print(f"  {s:<4} {our_qwk:>8.3f} {pap_qwk:>8.3f} "
          f"{diff:>+8.3f}  {flag}")

print("─"*38)
our_avg = np.mean(our_qwks)
print(f"  {'Avg':<4} {our_avg:>8.3f} "
      f"{paper_avg:>8.3f} {our_avg-paper_avg:>+8.3f}")

print("\n" + "="*65)
print("   FULL RESULTS TABLE")
print("="*65)
print(results_df.to_string(index=False))
print("\n✅ Step 6 V3 Complete!")



Loading feature matrix...
✅ Loaded: (12976, 1059)

   10-FOLD CV — RF REGRESSOR (no PCA, no scaling)

── Essay Set 1 (1783 essays, scores 2–12) ──
  Fold  1 → QWK=0.644  EA=0.385  AA=0.816  Spearman=0.760
  Fold  2 → QWK=0.648  EA=0.469  AA=0.888  Spearman=0.715
  Fold  3 → QWK=0.629  EA=0.425  AA=0.872  Spearman=0.730
  Fold  4 → QWK=0.622  EA=0.455  AA=0.871  Spearman=0.704
  Fold  5 → QWK=0.657  EA=0.472  AA=0.871  Spearman=0.671
  Fold  6 → QWK=0.635  EA=0.348  AA=0.871  Spearman=0.713
  Fold  7 → QWK=0.629  EA=0.360  AA=0.865  Spearman=0.736
  Fold  8 → QWK=0.611  EA=0.410  AA=0.860  Spearman=0.685
  Fold  9 → QWK=0.640  EA=0.382  AA=0.876  Spearman=0.721
  Fold 10 → QWK=0.630  EA=0.404  AA=0.837  Spearman=0.734
  ────────────────────────────────────────────────────
  MEAN    → QWK=0.634  EA=0.411  AA=0.863  Spearman=0.717
  💾 Checkpoint saved (set 1 done)

── Essay Set 2 (1800 essays, scores 1–6) ──
  Fold  1 → QWK=0.382  EA=0.528  AA=0.983  Spearman=0.369
  Fold  2 → QWK=0.562  

In [ ]:
# ============================================================
# STEP 6 V4: XGBoost + Entity Grid Similarity Feature
# ============================================================

!pip install xgboost -q

import os
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import cohen_kappa_score
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

SAVE_DIR          = '/kaggle/working/'
RESULTS_SAVE_PATH = os.path.join(SAVE_DIR, 'results_per_essay_set_v4.csv')

# ----------------------------------------------------------
# 6.1 LOAD
# ----------------------------------------------------------
print("Loading feature matrix...")
train_full = pd.read_parquet(
    os.path.join(SAVE_DIR, 'train_full_features.parquet')
)
print(f"✅ Loaded: {train_full.shape}")

# ----------------------------------------------------------
# 6.2 FEATURE COLUMNS
# ----------------------------------------------------------
SF_COLS = [
    'gunning_fog', 'flesch_re', 'ari',
    'guiraud_index',
    'noun_freq', 'verb_freq', 'avg_sent_len',
    'pronoun_density', 'pronoun_noun_ratio', 'giveness'
]
EG_COLS = [
    f"{r1}_{r2}"
    for r1 in ['S','O','X','-']
    for r2 in ['S','O','X','-']
]
EMB_COLS = (
    ['mean_coherence', 'min_coherence', 'std_coherence'] +
    [f'emb_{i}' for i in range(1024)]
)
ALL_FEATURE_COLS = SF_COLS + EG_COLS + EMB_COLS

# ----------------------------------------------------------
# 6.3 METRICS
# ----------------------------------------------------------
def qwk(y_true, y_pred):
    return cohen_kappa_score(y_true, y_pred, weights='quadratic')

def exact_agreement(y_true, y_pred):
    return np.mean(np.array(y_true) == np.array(y_pred))

def adjacent_agreement(y_true, y_pred):
    return np.mean(np.abs(np.array(y_true) - np.array(y_pred)) <= 1)

def spearman_r(y_true, y_pred):
    r, _ = spearmanr(y_true, y_pred)
    return r

def clip_round(y_pred, y_min, y_max):
    return np.clip(np.round(y_pred).astype(int), y_min, y_max)

# ----------------------------------------------------------
# 6.4 ENTITY GRID SIMILARITY FEATURE
# Paper's actual method: compare test essay entity grid
# to training essay entity grids via cosine similarity
# ----------------------------------------------------------
def add_eg_similarity_features(
    X_train, X_test,
    y_train,
    eg_train, eg_test,
    top_k=5
):
    """
    For each test essay:
    1. Compute cosine similarity to ALL training essays
       using their 16-dim entity grid vectors
    2. Find top-k most similar training essays
    3. Add features:
       - mean score of top-k similar essays
       - mean similarity score of top-k
    This is exactly what the paper describes.
    """
    # Cosine similarity between test and train entity grids
    # Shape: (n_test, n_train)
    sim_matrix = cosine_similarity(eg_test, eg_train)

    eg_sim_features_test  = []
    for i in range(len(eg_test)):
        sims       = sim_matrix[i]              # similarity to all train essays
        top_k_idx  = np.argsort(sims)[-top_k:]  # indices of top-k most similar
        top_k_sims = sims[top_k_idx]
        top_k_scores = y_train[top_k_idx]

        eg_sim_features_test.append([
            np.mean(top_k_scores),              # avg score of similar essays
            np.mean(top_k_sims),                # avg similarity
            np.max(top_k_sims),                 # max similarity
            np.std(top_k_scores),               # score variance of similar
        ])

    # For train fold: use leave-one-out style
    # (don't compare essay to itself)
    eg_sim_features_train = []
    train_sim_matrix = cosine_similarity(eg_train, eg_train)
    np.fill_diagonal(train_sim_matrix, 0)       # exclude self

    for i in range(len(eg_train)):
        sims       = train_sim_matrix[i]
        top_k_idx  = np.argsort(sims)[-top_k:]
        top_k_sims = sims[top_k_idx]
        top_k_scores = y_train[top_k_idx]

        eg_sim_features_train.append([
            np.mean(top_k_scores),
            np.mean(top_k_sims),
            np.max(top_k_sims),
            np.std(top_k_scores),
        ])

    eg_sim_train = np.array(eg_sim_features_train)
    eg_sim_test  = np.array(eg_sim_features_test)

    # Append to feature matrices
    X_train_aug = np.hstack([X_train, eg_sim_train])
    X_test_aug  = np.hstack([X_test,  eg_sim_test])

    return X_train_aug, X_test_aug

# ----------------------------------------------------------
# 6.5 SCORE RANGES
# ----------------------------------------------------------
SCORE_RANGES = {
    1: (2,  12),
    2: (1,   6),
    3: (0,   3),
    4: (0,   3),
    5: (0,   4),
    6: (0,   4),
    7: (2,  24),
    8: (10, 60),
}

# ----------------------------------------------------------
# 6.6 MAIN CV LOOP
# ----------------------------------------------------------
print("\n" + "="*65)
print("   10-FOLD CV — XGBoost + EG Similarity Features")
print("="*65)

all_results = []

for essay_set in sorted(train_full['essay_set'].unique()):

    subset = train_full[
        train_full['essay_set'] == essay_set
    ].reset_index(drop=True)

    X      = subset[ALL_FEATURE_COLS].values.astype(np.float32)
    y      = subset['domain1_score'].values.astype(np.float32)
    eg     = subset[EG_COLS].values.astype(np.float32)
    y_min, y_max = SCORE_RANGES[essay_set]

    print(f"\n── Essay Set {essay_set} "
          f"({len(subset)} essays, scores {y_min}–{y_max}) ──")

    kf = KFold(n_splits=10, shuffle=True, random_state=42)
    fold_qwk, fold_ea, fold_aa, fold_sp = [], [], [], []

    for fold_num, (train_idx, test_idx) in enumerate(kf.split(X), 1):

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        eg_train        = eg[train_idx]
        eg_test         = eg[test_idx]

        # Add entity grid similarity features (paper's method)
        X_train_aug, X_test_aug = add_eg_similarity_features(
            X_train, X_test,
            y_train,
            eg_train, eg_test,
            top_k=5
        )

        # XGBoost Regressor
        model = XGBRegressor(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_weight=3,
            random_state=42,
            n_jobs=-1,
            tree_method='hist',   # fast histogram method
            device='cuda',        # ← GPU acceleration!
            verbosity=0
        )
        model.fit(
            X_train_aug, y_train,
            eval_set=[(X_test_aug, y_test)],
            verbose=False
        )

        y_pred = clip_round(
            model.predict(X_test_aug), y_min, y_max
        )
        y_true = y_test.astype(int)

        fold_qwk.append(qwk(y_true, y_pred))
        fold_ea.append(exact_agreement(y_true, y_pred))
        fold_aa.append(adjacent_agreement(y_true, y_pred))
        fold_sp.append(spearman_r(y_true, y_pred))

        print(f"  Fold {fold_num:2d} → "
              f"QWK={fold_qwk[-1]:.3f}  "
              f"EA={fold_ea[-1]:.3f}  "
              f"AA={fold_aa[-1]:.3f}  "
              f"Spearman={fold_sp[-1]:.3f}")

    mean_qwk = np.mean(fold_qwk)
    mean_ea  = np.mean(fold_ea)
    mean_aa  = np.mean(fold_aa)
    mean_sp  = np.mean(fold_sp)

    print(f"  {'─'*52}")
    print(f"  MEAN    → "
          f"QWK={mean_qwk:.3f}  "
          f"EA={mean_ea:.3f}  "
          f"AA={mean_aa:.3f}  "
          f"Spearman={mean_sp:.3f}")

    all_results.append({
        'essay_set'    : essay_set,
        'n_essays'     : len(subset),
        'score_range'  : f"{y_min}–{y_max}",
        'mean_qwk'     : round(mean_qwk, 3),
        'mean_ea'      : round(mean_ea,  3),
        'mean_aa'      : round(mean_aa,  3),
        'mean_spearman': round(mean_sp,  3),
    })

    # Checkpoint after every set
    pd.DataFrame(all_results).to_csv(RESULTS_SAVE_PATH, index=False)
    print(f"  💾 Checkpoint saved (set {essay_set} done)")

# ----------------------------------------------------------
# 6.7 FINAL COMPARISON
# ----------------------------------------------------------
results_df = pd.DataFrame(all_results)

paper_qwk = {
    1: 0.91, 2: 0.79, 3: 0.71,
    4: 0.84, 5: 0.84, 6: 0.84,
    7: 0.87, 8: 0.92
}
paper_avg = np.mean(list(paper_qwk.values()))

print("\n" + "="*65)
print("   RESULTS vs PAPER (Table 4) — QWK")
print("="*65)
print(f"{'Set':<6} {'V3 RF':>8} {'V4 XGB':>8} "
      f"{'Paper':>8} {'Diff':>8}")
print("─"*45)

v3_qwk = {
    1:0.634, 2:0.542, 3:0.435,
    4:0.649, 5:0.626, 6:0.641,
    7:0.663, 8:0.532
}

our_qwks = []
for _, row in results_df.iterrows():
    s       = int(row['essay_set'])
    our_qwk = row['mean_qwk']
    pap_qwk = paper_qwk[s]
    diff    = our_qwk - pap_qwk
    our_qwks.append(our_qwk)
    flag    = "✅" if our_qwk >= pap_qwk - 0.05 else "⚠️"
    print(f"  {s:<4} {v3_qwk[s]:>8.3f} {our_qwk:>8.3f} "
          f"{pap_qwk:>8.3f} {diff:>+8.3f}  {flag}")

print("─"*45)
our_avg = np.mean(our_qwks)
print(f"  {'Avg':<4} {'0.590':>8} {our_avg:>8.3f} "
      f"{paper_avg:>8.3f} {our_avg-paper_avg:>+8.3f}")

print("\n" + "="*65)
print("   FULL RESULTS TABLE")
print("="*65)
print(results_df.to_string(index=False))
print("\n✅ Step 6 V4 Complete!")


Loading feature matrix...
✅ Loaded: (12976, 1059)

   10-FOLD CV — XGBoost + EG Similarity Features

── Essay Set 1 (1783 essays, scores 2–12) ──
  Fold  1 → QWK=0.784  EA=0.436  AA=0.866  Spearman=0.777
  Fold  2 → QWK=0.727  EA=0.475  AA=0.894  Spearman=0.730
  Fold  3 → QWK=0.675  EA=0.408  AA=0.866  Spearman=0.722
  Fold  4 → QWK=0.756  EA=0.489  AA=0.910  Spearman=0.775
  Fold  5 → QWK=0.742  EA=0.433  AA=0.876  Spearman=0.692
  Fold  6 → QWK=0.757  EA=0.382  AA=0.904  Spearman=0.748
  Fold  7 → QWK=0.795  EA=0.433  AA=0.910  Spearman=0.802
  Fold  8 → QWK=0.758  EA=0.438  AA=0.893  Spearman=0.739
  Fold  9 → QWK=0.743  EA=0.438  AA=0.888  Spearman=0.741
  Fold 10 → QWK=0.802  EA=0.416  AA=0.916  Spearman=0.770
  ────────────────────────────────────────────────────
  MEAN    → QWK=0.754  EA=0.435  AA=0.892  Spearman=0.750
  💾 Checkpoint saved (set 1 done)

── Essay Set 2 (1800 essays, scores 1–6) ──
  Fold  1 → QWK=0.597  EA=0.633  AA=0.989  Spearman=0.523
  Fold  2 → QWK=0.550  E

In [ ]:
# ============================================================
# STEP 6 V9 — Ensemble BGE (V6) + INSTRUCTOR (V8)
# Simple average of predictions from both models
# ============================================================

import os
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import cohen_kappa_score
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

SAVE_DIR          = '/kaggle/working/'
RESULTS_SAVE_PATH = os.path.join(SAVE_DIR, 'results_v9.csv')

if os.path.exists(RESULTS_SAVE_PATH):
    print("✅ V9 results exist — loading!")
    results_df = pd.read_csv(RESULTS_SAVE_PATH)
    print(results_df.to_string(index=False))

else:
    # ----------------------------------------------------------
    # LOAD BOTH FEATURE MATRICES
    # ----------------------------------------------------------
    print("Loading BGE feature matrix (V6)...")
    train_bge = pd.read_parquet(
        os.path.join(SAVE_DIR,
                     'train_full_features.parquet')
    )
    print(f"✅ BGE: {train_bge.shape}")

    print("Loading INSTRUCTOR feature matrix (V8)...")
    train_inst = pd.read_parquet(
        os.path.join(SAVE_DIR,
                     'train_full_features_instructor.parquet')
    )
    print(f"✅ INSTRUCTOR: {train_inst.shape}")

    # ----------------------------------------------------------
    # FEATURE COLUMNS FOR EACH MODEL
    # ----------------------------------------------------------
    SF_COLS = [
        'gunning_fog','flesch_re','ari','guiraud_index',
        'noun_freq','verb_freq','avg_sent_len',
        'pronoun_density','pronoun_noun_ratio','giveness'
    ]
    EG_COLS = [
        f"{r1}_{r2}"
        for r1 in ['S','O','X','-']
        for r2 in ['S','O','X','-']
    ]

    # BGE features (1024 dims — bge-large-en-v1.5 outputs 1024)
    BGE_FEAT_COLS = (
        SF_COLS + EG_COLS +
        ['mean_coherence','min_coherence','std_coherence'] +
        [f'emb_{i}' for i in range(1024)]
    )

    # INSTRUCTOR features (1024 dims)
    INST_FEAT_COLS = (
        SF_COLS + EG_COLS +
        ['mean_coherence','min_coherence','std_coherence'] +
        [f'emb_{i}' for i in range(1024)]
    )

    # ----------------------------------------------------------
    # METRICS
    # ----------------------------------------------------------
    def qwk(y_true, y_pred):
        return cohen_kappa_score(
            y_true, y_pred, weights='quadratic'
        )
    def exact_agreement(y_true, y_pred):
        return np.mean(np.array(y_true)==np.array(y_pred))
    def adjacent_agreement(y_true, y_pred):
        return np.mean(
            np.abs(np.array(y_true)-np.array(y_pred))<=1
        )
    def spearman_r(y_true, y_pred):
        r, _ = spearmanr(y_true, y_pred)
        return r
    def clip_round(y_pred, y_min, y_max):
        return np.clip(
            np.round(y_pred).astype(int), y_min, y_max
        )

    def add_eg_similarity(
        X_tr, X_te, y_tr, eg_tr, eg_te, top_k=5
    ):
        sim_te = cosine_similarity(eg_te, eg_tr)
        sim_tr = cosine_similarity(eg_tr, eg_tr)
        np.fill_diagonal(sim_tr, 0)
        def eg_feats(sim_matrix, y_ref):
            feats = []
            for i in range(len(sim_matrix)):
                sims = sim_matrix[i]
                idx  = np.argsort(sims)[-top_k:]
                feats.append([
                    np.mean(y_ref[idx]),
                    np.mean(sims[idx]),
                    np.max(sims[idx]),
                    np.std(y_ref[idx]),
                ])
            return np.array(feats)
        return (
            np.hstack([X_tr, eg_feats(sim_tr, y_tr)]),
            np.hstack([X_te, eg_feats(sim_te, y_tr)])
        )

    # ----------------------------------------------------------
    # SCORE RANGES + PARAMS
    # ----------------------------------------------------------
    SCORE_RANGES = {
        1:(2,12), 2:(1,6),  3:(0,3),
        4:(0,3),  5:(0,4),  6:(0,4),
        7:(2,24), 8:(10,60),
    }
    SET_PARAMS = {
        1: dict(n_estimators=1000, learning_rate=0.02,
                max_depth=7, subsample=0.8,
                colsample_bytree=0.7, min_child_weight=3,
                reg_alpha=0.1, reg_lambda=1.0),
        2: dict(n_estimators=1000, learning_rate=0.02,
                max_depth=6, subsample=0.8,
                colsample_bytree=0.7, min_child_weight=3,
                reg_alpha=0.1, reg_lambda=1.0),
        3: dict(n_estimators=800, learning_rate=0.03,
                max_depth=5, subsample=0.8,
                colsample_bytree=0.8, min_child_weight=2,
                reg_alpha=0.05, reg_lambda=1.0),
        4: dict(n_estimators=800, learning_rate=0.03,
                max_depth=5, subsample=0.8,
                colsample_bytree=0.8, min_child_weight=2,
                reg_alpha=0.05, reg_lambda=1.0),
        5: dict(n_estimators=800, learning_rate=0.03,
                max_depth=5, subsample=0.8,
                colsample_bytree=0.8, min_child_weight=2,
                reg_alpha=0.05, reg_lambda=1.0),
        6: dict(n_estimators=800, learning_rate=0.03,
                max_depth=5, subsample=0.8,
                colsample_bytree=0.8, min_child_weight=2,
                reg_alpha=0.05, reg_lambda=1.0),
        7: dict(n_estimators=1000, learning_rate=0.02,
                max_depth=7, subsample=0.8,
                colsample_bytree=0.7, min_child_weight=3,
                reg_alpha=0.1, reg_lambda=1.5),
        8: dict(n_estimators=1500, learning_rate=0.01,
                max_depth=5, subsample=0.7,
                colsample_bytree=0.6, min_child_weight=5,
                reg_alpha=0.5, reg_lambda=2.0),
    }

    # ----------------------------------------------------------
    # 10-FOLD CV — ENSEMBLE
    # Train TWO models per fold, average their predictions
    # ----------------------------------------------------------
    print("\n" + "="*65)
    print("  10-FOLD CV — ENSEMBLE (BGE + INSTRUCTOR)")
    print("="*65)

    all_results = []
    v8_qwk = {
        1:0.772, 2:0.640, 3:0.680,
        4:0.798, 5:0.779, 6:0.772,
        7:0.784, 8:0.686
    }

    for essay_set in sorted(train_bge['essay_set'].unique()):

        sub_bge  = train_bge[
            train_bge['essay_set']==essay_set
        ].reset_index(drop=True)
        sub_inst = train_inst[
            train_inst['essay_set']==essay_set
        ].reset_index(drop=True)

        X_bge  = sub_bge[BGE_FEAT_COLS].values.astype(np.float32)
        X_inst = sub_inst[INST_FEAT_COLS].values.astype(np.float32)
        y      = sub_bge['domain1_score'].values.astype(np.float32)
        eg     = sub_bge[EG_COLS].values.astype(np.float32)

        y_min, y_max = SCORE_RANGES[essay_set]
        score_range  = y_max - y_min
        params       = SET_PARAMS[essay_set]

        print(f"\n── Essay Set {essay_set} "
              f"({len(sub_bge)} essays, "
              f"scores {y_min}–{y_max}) ──")

        kf = KFold(n_splits=10, shuffle=True, random_state=42)
        fold_qwk, fold_ea, fold_aa, fold_sp = [], [], [], []

        for fold_num, (tr_idx, te_idx) in enumerate(
            kf.split(X_bge), 1
        ):
            # Split both feature matrices
            Xb_tr, Xb_te   = X_bge[tr_idx],  X_bge[te_idx]
            Xi_tr, Xi_te   = X_inst[tr_idx], X_inst[te_idx]
            y_tr, y_te     = y[tr_idx], y[te_idx]
            eg_tr, eg_te   = eg[tr_idx], eg[te_idx]

            y_tr_norm = (y_tr - y_min) / score_range

            # Add EG similarity to both
            Xb_tr_aug, Xb_te_aug = add_eg_similarity(
                Xb_tr, Xb_te, y_tr, eg_tr, eg_te
            )
            Xi_tr_aug, Xi_te_aug = add_eg_similarity(
                Xi_tr, Xi_te, y_tr, eg_tr, eg_te
            )

            # Model 1: BGE features
            m1 = XGBRegressor(
                **params, random_state=42,
                n_jobs=-1, tree_method='hist',
                device='cuda', verbosity=0
            )
            m1.fit(Xb_tr_aug, y_tr_norm)
            p1 = m1.predict(Xb_te_aug)

            # Model 2: INSTRUCTOR features
            m2 = XGBRegressor(
                **params, random_state=42,
                n_jobs=-1, tree_method='hist',
                device='cuda', verbosity=0
            )
            m2.fit(Xi_tr_aug, y_tr_norm)
            p2 = m2.predict(Xi_te_aug)

            # Ensemble: weighted average
            # INSTRUCTOR slightly better on most sets
            p_ensemble  = 0.4 * p1 + 0.6 * p2

            y_pred_raw  = p_ensemble * score_range + y_min
            y_pred      = clip_round(y_pred_raw, y_min, y_max)
            y_true      = y_te.astype(int)

            fold_qwk.append(qwk(y_true, y_pred))
            fold_ea.append(exact_agreement(y_true, y_pred))
            fold_aa.append(adjacent_agreement(y_true, y_pred))
            fold_sp.append(spearman_r(y_true, y_pred))

            print(f"  Fold {fold_num:2d} → "
                  f"QWK={fold_qwk[-1]:.3f}  "
                  f"EA={fold_ea[-1]:.3f}  "
                  f"AA={fold_aa[-1]:.3f}  "
                  f"Spearman={fold_sp[-1]:.3f}")

        mean_qwk = np.mean(fold_qwk)
        mean_ea  = np.mean(fold_ea)
        mean_aa  = np.mean(fold_aa)
        mean_sp  = np.mean(fold_sp)

        print(f"  {'─'*52}")
        print(f"  MEAN → "
              f"QWK={mean_qwk:.3f}  "
              f"EA={mean_ea:.3f}  "
              f"AA={mean_aa:.3f}  "
              f"Spearman={mean_sp:.3f}  "
              f"(V8 was {v8_qwk[essay_set]:.3f})")

        all_results.append({
            'essay_set'    : essay_set,
            'n_essays'     : len(sub_bge),
            'score_range'  : f"{y_min}–{y_max}",
            'mean_qwk'     : round(mean_qwk, 3),
            'mean_ea'      : round(mean_ea,  3),
            'mean_aa'      : round(mean_aa,  3),
            'mean_spearman': round(mean_sp,  3),
        })

        pd.DataFrame(all_results).to_csv(
            RESULTS_SAVE_PATH, index=False
        )
        print(f"  💾 Checkpoint saved (set {essay_set} done)")

    results_df = pd.DataFrame(all_results)

# ----------------------------------------------------------
# FINAL COMPARISON
# ----------------------------------------------------------
paper_qwk = {
    1:0.91, 2:0.79, 3:0.71,
    4:0.84, 5:0.84, 6:0.84,
    7:0.87, 8:0.92
}
paper_avg  = np.mean(list(paper_qwk.values()))
v8_qwk     = {
    1:0.772,2:0.640,3:0.680,
    4:0.798,5:0.779,6:0.772,
    7:0.784,8:0.686
}

print("\n" + "="*72)
print("   FINAL RESULTS — ALL VERSIONS")
print("="*72)
print(f"{'Set':<5} {'V4':>6} {'V6 BGE':>7} "
      f"{'V8 INST':>8} {'V9 ENS':>8} "
      f"{'Paper':>7} {'Diff':>7}")
print("─"*55)

v4  = {1:0.754,2:0.627,3:0.624,4:0.752,
       5:0.769,6:0.748,7:0.772,8:0.676}
v6  = {1:0.765,2:0.648,3:0.638,4:0.782,
       5:0.779,6:0.766,7:0.779,8:0.689}

our_qwks = []
for _, row in results_df.iterrows():
    s   = int(row['essay_set'])
    our = row['mean_qwk']
    pap = paper_qwk[s]
    our_qwks.append(our)
    flag = "✅" if our >= pap - 0.05 else "⚠️"
    print(f"  {s:<3} {v4[s]:>6.3f} {v6[s]:>7.3f} "
          f"{v8_qwk[s]:>8.3f} {our:>8.3f} "
          f"{pap:>7.3f} {our-pap:>+7.3f}  {flag}")

print("─"*55)
our_avg = np.mean(our_qwks)
print(f"  {'Avg':<3} {'0.715':>6} {'0.731':>7} "
      f"{'0.739':>8} {our_avg:>8.3f} "
      f"{paper_avg:>7.3f} {our_avg-paper_avg:>+7.3f}")

print("\n" + "="*72)
print("   FULL RESULTS TABLE")
print("="*72)
print(results_df.to_string(index=False))
print("\n✅ Step 6 V9 Complete!")


In [ ]:
# ============================================================
# STEP 4J — E5-Mistral WITH CHECKPOINTING
# Saves every 500 essays — resume after timeout
# ============================================================

!pip install transformers accelerate bitsandbytes -q

import os, gc, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig
from nltk.tokenize import sent_tokenize
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
warnings.filterwarnings('ignore')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

SAVE_DIR      = '/kaggle/working/'
EMB_E5_PATH   = os.path.join(SAVE_DIR, 'train_embeddings_e5mistral.csv')
CHECKPOINT_DIR = os.path.join(SAVE_DIR, 'e5_checkpoints')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ── Check if final file already exists ───────────────────────
if os.path.exists(EMB_E5_PATH):
    print("✅ E5-Mistral embeddings complete!")
    emb_e5_df = pd.read_csv(EMB_E5_PATH)
    print("Shape:", emb_e5_df.shape)

else:
    # ── Check existing checkpoints ───────────────────────────
    sf_df = pd.read_csv(
        os.path.join(SAVE_DIR, 'train_shallow_features.csv'),
        encoding='latin-1'
    )
    texts     = sf_df['essay'].astype(str).tolist()
    n_total   = len(texts)
    CHUNK_SIZE = 500   # save every 500 essays

    # Find what's already done
    completed_chunks = []
    for f in os.listdir(CHECKPOINT_DIR):
        if f.startswith('chunk_') and f.endswith('.npz'):
            chunk_id = int(f.replace('chunk_','').replace('.npz',''))
            completed_chunks.append(chunk_id)
    completed_chunks.sort()

    if completed_chunks:
        last_done = max(completed_chunks)
        start_idx = (last_done + 1) * CHUNK_SIZE
        print(f"✅ Resuming from essay {start_idx}")
        print(f"   Completed chunks: {completed_chunks}")
        print(f"   Essays remaining: {n_total - start_idx}")
    else:
        start_idx = 0
        print(f"Starting fresh — {n_total} essays total")

    if start_idx >= n_total:
        print("All chunks done! Assembling final file...")
    else:
        # ── Load model ────────────────────────────────────────
        gc.collect()
        torch.cuda.empty_cache()
        print(f"GPU free: "
              f"{torch.cuda.mem_get_info()[0]/1024**3:.1f} GB")

        MODEL_NAME = 'intfloat/e5-mistral-7b-instruct'
        print(f"Loading {MODEL_NAME} (4-bit)...")

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type='nf4'
        )
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        model     = AutoModel.from_pretrained(
            MODEL_NAME,
            quantization_config=bnb_config,
            device_map='cuda',
            torch_dtype=torch.float16
        )
        model.eval()
        print(f"✅ Loaded — GPU free: "
              f"{torch.cuda.mem_get_info()[0]/1024**3:.1f} GB")

        ESSAY_INSTRUCTION = (
            "Instruct: Retrieve semantically similar essays "
            "based on writing quality and coherence\nQuery: "
        )
        SENT_INSTRUCTION = (
            "Instruct: Retrieve semantically similar sentences "
            "based on local coherence\nQuery: "
        )

        def last_token_pool(last_hidden, attention_mask):
            seq_len = attention_mask.sum(dim=1) - 1
            batch   = last_hidden.shape[0]
            return last_hidden[
                torch.arange(batch,
                    device=last_hidden.device), seq_len
            ]

        def encode_batch(text_list, instruction, batch_size=4):
            inputs = [instruction + t for t in text_list]
            all_embs = []
            for i in range(0, len(inputs), batch_size):
                batch = inputs[i:i+batch_size]
                enc   = tokenizer(
                    batch, max_length=512,
                    padding=True, truncation=True,
                    return_tensors='pt'
                ).to('cuda')
                with torch.no_grad():
                    out  = model(**enc)
                    embs = last_token_pool(
                        out.last_hidden_state,
                        enc['attention_mask']
                    )
                    embs = F.normalize(embs, p=2, dim=1)
                all_embs.append(embs.float().cpu().numpy())
            return np.vstack(all_embs)

        # ── Process in chunks ─────────────────────────────────
        print(f"\nProcessing essays {start_idx} to {n_total}...")
        print(f"Chunk size: {CHUNK_SIZE} essays")
        print(f"Saving checkpoint after each chunk")

        for chunk_start in range(
            start_idx, n_total, CHUNK_SIZE
        ):
            chunk_end  = min(chunk_start + CHUNK_SIZE, n_total)
            chunk_id   = chunk_start // CHUNK_SIZE
            chunk_path = os.path.join(
                CHECKPOINT_DIR, f'chunk_{chunk_id}.npz'
            )

            if os.path.exists(chunk_path):
                print(f"  Chunk {chunk_id} already exists — skip")
                continue

            print(f"\n  Chunk {chunk_id}: "
                  f"essays {chunk_start}-{chunk_end} "
                  f"({chunk_end-chunk_start} essays)")

            chunk_texts = texts[chunk_start:chunk_end]
            chunk_embs  = []
            chunk_coh   = []

            # Encode essays
            print(f"    Encoding essays...")
            essay_embs = encode_batch(
                [t[:800] for t in chunk_texts],
                ESSAY_INSTRUCTION, batch_size=4
            )

            # Encode sentences for coherence
            print(f"    Computing coherence...")
            for text in tqdm(
                chunk_texts,
                desc=f"    Chunk {chunk_id} coherence"
            ):
                sents = sent_tokenize(text)
                if not sents: sents = [text]
                sents = [s[:300] for s in sents[:20]]

                if len(sents) < 2:
                    chunk_coh.append([0.0, 0.0, 0.0])
                    continue

                sent_embs = encode_batch(
                    sents, SENT_INSTRUCTION, batch_size=8
                )
                sims = [
                    float(cosine_similarity(
                        sent_embs[i].reshape(1,-1),
                        sent_embs[i+1].reshape(1,-1)
                    )[0][0])
                    for i in range(len(sent_embs)-1)
                ]
                chunk_coh.append([
                    float(np.mean(sims)),
                    float(np.min(sims)),
                    float(np.std(sims))
                ])

            # Save chunk
            np.savez_compressed(
                chunk_path,
                essay_embs = essay_embs,
                coherence  = np.array(chunk_coh)
            )
            print(f"    💾 Chunk {chunk_id} saved!")
            print(f"    GPU free: "
                  f"{torch.cuda.mem_get_info()[0]/1024**3:.1f}GB")

        print("\n✅ All chunks processed!")

    # ── Assemble all chunks into final file ───────────────────
    print("\nAssembling all chunks...")

    # Check all chunks exist
    n_chunks   = (n_total + CHUNK_SIZE - 1) // CHUNK_SIZE
    all_chunks = sorted([
        int(f.replace('chunk_','').replace('.npz',''))
        for f in os.listdir(CHECKPOINT_DIR)
        if f.startswith('chunk_') and f.endswith('.npz')
    ])

    print(f"Chunks found: {len(all_chunks)}/{n_chunks}")
    missing = [i for i in range(n_chunks)
               if i not in all_chunks]
    if missing:
        print(f"❌ Missing chunks: {missing}")
        print("Run this cell again to complete missing chunks")
    else:
        # Assemble
        all_essay_embs = []
        all_coherence  = []

        for chunk_id in sorted(all_chunks):
            chunk_path = os.path.join(
                CHECKPOINT_DIR, f'chunk_{chunk_id}.npz'
            )
            data = np.load(chunk_path)
            all_essay_embs.append(data['essay_embs'])
            all_coherence.append(data['coherence'])

        all_essay_embs = np.vstack(all_essay_embs)
        all_coherence  = np.vstack(all_coherence)

        print(f"Assembled: {all_essay_embs.shape}")

        # PCA reduction 4096 → 1024
        print("Reducing 4096→1024 dims via PCA...")
        from sklearn.decomposition import PCA
        import pickle

        pca       = PCA(n_components=1024, random_state=42)
        embs_1024 = pca.fit_transform(all_essay_embs)
        explained = pca.explained_variance_ratio_.sum()
        print(f"Variance explained: {explained:.1%}")

        with open(
            os.path.join(SAVE_DIR, 'e5_pca.pkl'), 'wb'
        ) as f:
            pickle.dump(pca, f)

        # Save final CSV
        coh_df    = pd.DataFrame(
            all_coherence,
            columns=['mean_coherence',
                     'min_coherence','std_coherence']
        )
        emb_cols  = [f'emb_{i}' for i in range(1024)]
        emb_df    = pd.DataFrame(embs_1024, columns=emb_cols)
        emb_e5_df = pd.concat([coh_df, emb_df], axis=1)

        emb_e5_df.to_csv(EMB_E5_PATH, index=False)
        print(f"\n💾 Final file saved: {EMB_E5_PATH}")
        print(f"Shape: {emb_e5_df.shape}")
        print("✅ Step 4J Complete!")
        print("⚠️  Click Save Version NOW!")

In [ ]:
# ============================================================
# STEP 5J — Feature Matrix with E5-Mistral Embeddings
# ============================================================

FULL_E5_PATH = os.path.join(
    SAVE_DIR, 'train_full_features_e5mistral.parquet'
)

if os.path.exists(FULL_E5_PATH):
    print("✅ Already exists!")
    train_e5 = pd.read_parquet(FULL_E5_PATH)
    print("Shape:", train_e5.shape)

else:
    SF_COLS  = [
        'gunning_fog','flesch_re','ari','guiraud_index',
        'noun_freq','verb_freq','avg_sent_len',
        'pronoun_density','pronoun_noun_ratio','giveness'
    ]
    EG_COLS  = [
        f"{r1}_{r2}"
        for r1 in ['S','O','X','-']
        for r2 in ['S','O','X','-']
    ]
    EMB_COLS = (
        ['mean_coherence','min_coherence','std_coherence'] +
        [f'emb_{i}' for i in range(1024)]
    )
    META_COLS = [
        'essay_id','essay_set','essay',
        'domain1_score','rater1_domain1','rater2_domain1'
    ]

    sf_df    = pd.read_csv(
        os.path.join(SAVE_DIR,'train_shallow_features.csv'),
        encoding='latin-1'
    )
    eg_df    = pd.read_csv(
        os.path.join(SAVE_DIR,
                     'train_entity_grid_features.csv')
    )
    emb_e5_df = pd.read_csv(EMB_E5_PATH)

    train_e5 = pd.concat([
        sf_df[META_COLS].reset_index(drop=True),
        sf_df[SF_COLS].reset_index(drop=True),
        eg_df[EG_COLS].reset_index(drop=True),
        emb_e5_df[EMB_COLS].reset_index(drop=True),
    ], axis=1)

    train_e5.to_parquet(FULL_E5_PATH, index=False)
    print(f"✅ Saved: {train_e5.shape}")
    print("10 SF + 16 EG + 1027 EMB = 1053 features")
    print("⚠️  Click Save Version NOW!")

In [ ]:
# ============================================================
# STEP 6 V17 — E5-Mistral + INSTRUCTOR Ensemble
# NO leakage — both models pretrained on internet data
# DIRECTLY comparable to paper's GPT-3 approach
# ============================================================

import os
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import cohen_kappa_score
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

SAVE_DIR          = '/kaggle/working/'
RESULTS_SAVE_PATH = os.path.join(SAVE_DIR, 'results_v17_complete.csv')

if os.path.exists(RESULTS_SAVE_PATH):
    print("✅ V17 results exist!")
    results_df = pd.read_csv(RESULTS_SAVE_PATH)
    print(results_df.to_string(index=False))

else:
    print("Loading feature matrices...")
    train_e5   = pd.read_parquet(
        os.path.join(SAVE_DIR,
            'train_full_features_e5mistral.parquet')
    )
    train_inst = pd.read_parquet(
        os.path.join(SAVE_DIR,
            'train_full_features_instructor.parquet')
    )
    print(f"✅ E5-Mistral:  {train_e5.shape}")
    print(f"✅ INSTRUCTOR:  {train_inst.shape}")

    SF_COLS = [
        'gunning_fog','flesch_re','ari','guiraud_index',
        'noun_freq','verb_freq','avg_sent_len',
        'pronoun_density','pronoun_noun_ratio','giveness'
    ]
    EG_COLS = [
        f"{r1}_{r2}"
        for r1 in ['S','O','X','-']
        for r2 in ['S','O','X','-']
    ]
    E5_FEAT   = (
        SF_COLS + EG_COLS +
        ['mean_coherence','min_coherence','std_coherence'] +
        [f'emb_{i}' for i in range(1024)]
    )
    INST_FEAT = (
        SF_COLS + EG_COLS +
        ['mean_coherence','min_coherence','std_coherence'] +
        [f'emb_{i}' for i in range(1024)]
    )

    def qwk(y_true, y_pred):
        return cohen_kappa_score(
            y_true, y_pred, weights='quadratic'
        )
    def exact_agreement(y_true, y_pred):
        return np.mean(np.array(y_true)==np.array(y_pred))
    def adjacent_agreement(y_true, y_pred):
        return np.mean(
            np.abs(np.array(y_true)-np.array(y_pred))<=1
        )
    def spearman_r(y_true, y_pred):
        r, _ = spearmanr(y_true, y_pred)
        return r
    def clip_round(y_pred, y_min, y_max):
        return np.clip(
            np.round(y_pred).astype(int), y_min, y_max
        )
    def add_eg_sim(X_tr, X_te, y_tr, eg_tr, eg_te, k=5):
        s_te = cosine_similarity(eg_te, eg_tr)
        s_tr = cosine_similarity(eg_tr, eg_tr)
        np.fill_diagonal(s_tr, 0)
        def f(s, y):
            out = []
            for i in range(len(s)):
                idx = np.argsort(s[i])[-k:]
                out.append([
                    np.mean(y[idx]),
                    np.mean(s[i][idx]),
                    np.max(s[i][idx]),
                    np.std(y[idx])
                ])
            return np.array(out)
        return (
            np.hstack([X_tr, f(s_tr, y_tr)]),
            np.hstack([X_te, f(s_te, y_tr)])
        )

    SCORE_RANGES = {
        1:(2,12), 2:(1,6),  3:(0,3),
        4:(0,3),  5:(0,4),  6:(0,4),
        7:(2,24), 8:(10,60),
    }
    SET_PARAMS = {
        1: dict(n_estimators=1000, learning_rate=0.02,
                max_depth=7, subsample=0.8,
                colsample_bytree=0.7, min_child_weight=3,
                reg_alpha=0.1, reg_lambda=1.0),
        2: dict(n_estimators=1000, learning_rate=0.02,
                max_depth=6, subsample=0.8,
                colsample_bytree=0.7, min_child_weight=3,
                reg_alpha=0.1, reg_lambda=1.0),
        3: dict(n_estimators=800, learning_rate=0.03,
                max_depth=5, subsample=0.8,
                colsample_bytree=0.8, min_child_weight=2,
                reg_alpha=0.05, reg_lambda=1.0),
        4: dict(n_estimators=800, learning_rate=0.03,
                max_depth=5, subsample=0.8,
                colsample_bytree=0.8, min_child_weight=2,
                reg_alpha=0.05, reg_lambda=1.0),
        5: dict(n_estimators=800, learning_rate=0.03,
                max_depth=5, subsample=0.8,
                colsample_bytree=0.8, min_child_weight=2,
                reg_alpha=0.05, reg_lambda=1.0),
        6: dict(n_estimators=800, learning_rate=0.03,
                max_depth=5, subsample=0.8,
                colsample_bytree=0.8, min_child_weight=2,
                reg_alpha=0.05, reg_lambda=1.0),
        7: dict(n_estimators=1000, learning_rate=0.02,
                max_depth=7, subsample=0.8,
                colsample_bytree=0.7, min_child_weight=3,
                reg_alpha=0.1, reg_lambda=1.5),
        8: dict(n_estimators=1500, learning_rate=0.01,
                max_depth=5, subsample=0.7,
                colsample_bytree=0.6, min_child_weight=5,
                reg_alpha=0.5, reg_lambda=2.0),
    }

    print("\n" + "="*65)
    print("  10-FOLD CV — E5-Mistral-7B + INSTRUCTOR")
    print("  ✅ NO LEAKAGE — Both pretrained models")
    print("  ✅ Direct comparison with paper's GPT-3")
    print("="*65)

    all_results = []
    v9_qwk = {
        1:0.775,2:0.644,3:0.672,
        4:0.802,5:0.792,6:0.782,
        7:0.787,8:0.698
    }

    for essay_set in sorted(train_e5['essay_set'].unique()):

        sub_e5   = train_e5[
            train_e5['essay_set']==essay_set
        ].reset_index(drop=True)
        sub_inst = train_inst[
            train_inst['essay_set']==essay_set
        ].reset_index(drop=True)

        Xe   = sub_e5[E5_FEAT].values.astype(np.float32)
        Xi   = sub_inst[INST_FEAT].values.astype(np.float32)
        y    = sub_e5['domain1_score'].values.astype(np.float32)
        eg   = sub_e5[EG_COLS].values.astype(np.float32)

        y_min, y_max = SCORE_RANGES[essay_set]
        score_range  = y_max - y_min
        params       = SET_PARAMS[essay_set]

        print(f"\n── Set {essay_set} "
              f"({len(sub_e5)} essays, "
              f"{y_min}-{y_max}) ──")

        kf = KFold(n_splits=10, shuffle=True, random_state=42)
        fold_qwk, fold_ea, fold_aa, fold_sp = [], [], [], []

        for fold_num, (tr_idx, te_idx) in enumerate(
            kf.split(Xe), 1
        ):
            y_tr      = y[tr_idx]
            y_te      = y[te_idx]
            y_tr_norm = (y_tr - y_min) / score_range
            eg_tr     = eg[tr_idx]
            eg_te     = eg[te_idx]

            def train_predict(X):
                Xtr, Xte = add_eg_sim(
                    X[tr_idx], X[te_idx],
                    y_tr, eg_tr, eg_te
                )
                m = XGBRegressor(
                    **params, random_state=42,
                    n_jobs=-1, tree_method='hist',
                    device='cuda', verbosity=0
                )
                m.fit(Xtr, y_tr_norm)
                return m.predict(Xte)

            # 60% E5-Mistral + 40% INSTRUCTOR
            # E5-Mistral is larger and stronger
            p_e5  = train_predict(Xe)
            p_in  = train_predict(Xi)
            p_ens = 0.6 * p_e5 + 0.4 * p_in

            y_pred = clip_round(
                p_ens * score_range + y_min,
                y_min, y_max
            )
            y_true = y_te.astype(int)

            fold_qwk.append(qwk(y_true, y_pred))
            fold_ea.append(exact_agreement(y_true, y_pred))
            fold_aa.append(adjacent_agreement(y_true, y_pred))
            fold_sp.append(spearman_r(y_true, y_pred))

            print(f"  Fold {fold_num:2d} → "
                  f"QWK={fold_qwk[-1]:.3f}  "
                  f"EA={fold_ea[-1]:.3f}  "
                  f"AA={fold_aa[-1]:.3f}  "
                  f"Sp={fold_sp[-1]:.3f}")

        mean_qwk = np.mean(fold_qwk)
        mean_ea  = np.mean(fold_ea)
        mean_aa  = np.mean(fold_aa)
        mean_sp  = np.mean(fold_sp)

        print(f"  {'─'*52}")
        print(f"  MEAN → QWK={mean_qwk:.3f}  "
              f"(V9 INSTRUCTOR={v9_qwk[essay_set]:.3f})")

        all_results.append({
            'essay_set'    : essay_set,
            'n_essays'     : len(sub_e5),
            'score_range'  : f"{y_min}–{y_max}",
            'mean_qwk'     : round(mean_qwk, 3),
            'mean_ea'      : round(mean_ea,  3),
            'mean_aa'      : round(mean_aa,  3),
            'mean_spearman': round(mean_sp,  3),
        })

        pd.DataFrame(all_results).to_csv(
            RESULTS_SAVE_PATH, index=False
        )
        print(f"  💾 Checkpoint (set {essay_set} done)")

    results_df = pd.DataFrame(all_results)

# FINAL COMPARISON
paper_qwk = {
    1:0.91,2:0.79,3:0.71,
    4:0.84,5:0.84,6:0.84,
    7:0.87,8:0.92
}
paper_avg = np.mean(list(paper_qwk.values()))
v9_qwk    = {
    1:0.775,2:0.644,3:0.672,
    4:0.802,5:0.792,6:0.782,
    7:0.787,8:0.698
}

print("\n" + "="*70)
print("   E5-MISTRAL vs INSTRUCTOR vs PAPER")
print("   All pretrained — no leakage — fair comparison")
print("="*70)
print(f"{'Set':<5} {'INST(V9)':>9} {'E5+INST':>9} "
      f"{'Paper':>7} {'Diff':>7}")
print("─"*45)

our_qwks = []
for _, row in results_df.iterrows():
    s   = int(row['essay_set'])
    our = row['mean_qwk']
    pap = paper_qwk[s]
    our_qwks.append(our)
    flag = "✅" if our >= pap else "⚠️"
    imp  = our - v9_qwk[s]
    print(f"  {s:<3} {v9_qwk[s]:>9.3f} {our:>9.3f} "
          f"{pap:>7.3f} {our-pap:>+7.3f}  "
          f"{flag} ↑{imp:.3f}")

print("─"*45)
our_avg = np.mean(our_qwks)
print(f"  {'Avg':<3} {'0.744':>9} {our_avg:>9.3f} "
      f"{paper_avg:>7.3f} {our_avg-paper_avg:>+7.3f}")

print(f"\n  Sets exceeding paper: "
      f"{sum(1 for v in our_qwks if v >= list(paper_qwk.values())[list(results_df['essay_set']).index(int(results_df.iloc[list(our_qwks).index(v)]['essay_set']))])}/8")

print("\n" + "="*70)
print(results_df.to_string(index=False))
print("\n✅ V17 Complete!")
print("⚠️  Click Save Version NOW!")

In [ ]:
pip install -U bitsandbytes>=0.46.1

In [ ]:
# ============================================================
# STEP 4O — E5-Mistral-7B with OFFICIAL ASAP Rubrics
# Source: Essay_Set_ReadMeFirst.docx files (official docs)
# True LLM decoder + real scoring criteria
# ============================================================

import os, gc, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import (
    AutoTokenizer, AutoModel, BitsAndBytesConfig
)
from nltk.tokenize import sent_tokenize
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
warnings.filterwarnings('ignore')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = \
    'expandable_segments:True'

SAVE_DIR       = '/kaggle/working/'
EMB_E5RUB_PATH = os.path.join(
    SAVE_DIR, 'train_embeddings_e5_rubric.csv'
)
CHECKPOINT_DIR = os.path.join(
    SAVE_DIR, 'e5rubric_checkpoints'
)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

if os.path.exists(EMB_E5RUB_PATH):
    print("✅ E5+Rubric embeddings exist!")
    emb_e5rub_df = pd.read_csv(EMB_E5RUB_PATH)
    print("Shape:", emb_e5rub_df.shape)

else:
    gc.collect()
    torch.cuda.empty_cache()

    sf_df = pd.read_csv(
        os.path.join(SAVE_DIR,'train_shallow_features.csv'),
        encoding='latin-1'
    )
    texts      = sf_df['essay'].astype(str).tolist()
    essay_sets = sf_df['essay_set'].tolist()
    n_total    = len(texts)
    CHUNK_SIZE = 500

    # ----------------------------------------------------------
    # OFFICIAL ASAP RUBRICS
    # Source: Essay_Set_X--ReadMeFirst.docx (uploaded files)
    # Exact scoring criteria from official dataset docs
    # ----------------------------------------------------------
    RUBRICS = {
        1: (
            "Instruct: Represent a student essay for "
            "automated scoring. "
            "Essay Set 1: Grade 8 persuasive essay. "
            "Prompt: Write a letter to your local "
            "newspaper stating your opinion on the "
            "effects computers have on people. "
            "Scoring: Two raters each score 1-6, "
            "resolved score range 2-12. "
            "Score 1-2 (low): Undeveloped, vague details, "
            "awkward/fragmented, no audience awareness. "
            "Score 3-4 (medium-low): Minimal development, "
            "inadequate support, some organization, "
            "some audience awareness. "
            "Score 5-6 (medium-high): Adequately developed, "
            "mix of general and specific details, "
            "satisfactory organization, adequate language. "
            "Score 7-8 (high): Well-developed position, "
            "specific details, clear organization, "
            "effective language. "
            "Score 9-10 (very high): Thoroughly developed, "
            "compelling argument, strong organization, "
            "varied vocabulary and sentence structure. "
            "Score 11-12 (exceptional): Outstanding, "
            "distinctive quality, insightful, "
            "excellent command of language.\n"
            "Query: "
        ),
        2: (
            "Instruct: Represent a student essay for "
            "automated scoring. "
            "Essay Set 2: Grade 10 persuasive essay. "
            "Prompt: Write a persuasive essay to a "
            "newspaper about censorship in libraries. "
            "Do you believe certain materials should be "
            "removed if found offensive? "
            "Scored on two domains: "
            "Domain 1 (Writing Applications) range 1-6, "
            "Domain 2 (Language Conventions) range 1-4. "
            "Domain 1 Score 6 (exceptional): Fully "
            "accomplishes task, distinctive quality, "
            "unifying theme, fully developed ideas, "
            "in-depth information, exceptional details. "
            "Domain 1 Score 4-5 (proficient): "
            "Accomplishes task well, clear main idea, "
            "relevant supporting details, effective "
            "organization and transitions. "
            "Domain 1 Score 2-3 (developing): "
            "Partially accomplishes task, some relevant "
            "ideas, some organization, basic language. "
            "Domain 1 Score 1 (beginning): "
            "Minimally accomplishes task, unclear ideas, "
            "little organization.\n"
            "Query: "
        ),
        3: (
            "Instruct: Represent a student essay for "
            "automated scoring. "
            "Essay Set 3: Grade 10 source-dependent "
            "response. Students read a passage about a "
            "cyclist's journey (Rough Road Ahead by "
            "Joe Kurmaskie) and write a response. "
            "Score range: 0-3. "
            "Score 3 (complete): Accurate and complete "
            "response, uses specific text evidence, "
            "clear understanding of source material. "
            "Score 2 (adequate): Mostly accurate with "
            "some text evidence, partially complete. "
            "Score 1 (minimal): Partially accurate or "
            "vague, limited use of text evidence. "
            "Score 0 (insufficient): Inaccurate, "
            "off-topic, or missing response.\n"
            "Query: "
        ),
        4: (
            "Instruct: Represent a student essay for "
            "automated scoring. "
            "Essay Set 4: Grade 10 source-dependent "
            "response. Students read an excerpt from "
            "Winter Hibiscus by Minfong Ho about a "
            "Vietnamese immigrant girl and write a "
            "response about the story. "
            "Score range: 0-3. "
            "Score 3 (complete): Accurate and complete "
            "response, uses specific textual evidence, "
            "demonstrates clear understanding of "
            "character emotions and cultural themes. "
            "Score 2 (adequate): Mostly accurate with "
            "some evidence, partially addresses prompt. "
            "Score 1 (minimal): Partially accurate, "
            "vague or limited textual reference. "
            "Score 0 (insufficient): Inaccurate, "
            "off-topic, or missing.\n"
            "Query: "
        ),
        5: (
            "Instruct: Represent a student essay for "
            "automated scoring. "
            "Essay Set 5: Grade 8 source-dependent "
            "response. Students read an excerpt from "
            "Narciso Rodriguez from Home: The Blueprints "
            "of Our Lives about a Cuban immigrant family "
            "and write a response. "
            "Score range: 0-4. "
            "Score 4 (excellent): Insightful analysis, "
            "specific and accurate textual evidence, "
            "clear explanation of themes and author "
            "purpose, thorough understanding. "
            "Score 3 (proficient): Adequate analysis "
            "with relevant evidence, addresses prompt "
            "with mostly accurate understanding. "
            "Score 2 (developing): Basic understanding, "
            "limited evidence, partially addresses prompt. "
            "Score 0-1 (beginning): Minimal or "
            "inaccurate understanding of source.\n"
            "Query: "
        ),
        6: (
            "Instruct: Represent a student essay for "
            "automated scoring. "
            "Essay Set 6: Grade 10 source-dependent "
            "response. Students read The Mooring Mast "
            "by Marcia Amidon Lusted about the Empire "
            "State Building's dirigible mooring plan "
            "and write a response. "
            "Score range: 0-4. "
            "Score 4 (excellent): Sophisticated "
            "understanding of author techniques, "
            "specific textual evidence, clear analysis "
            "of purpose and writing strategies. "
            "Score 3 (proficient): Adequate understanding "
            "with relevant evidence, addresses techniques "
            "and purpose. "
            "Score 2 (developing): Basic identification "
            "of techniques with limited evidence. "
            "Score 0-1 (beginning): Minimal understanding "
            "of source or author techniques.\n"
            "Query: "
        ),
        7: (
            "Instruct: Represent a student essay for "
            "automated scoring. "
            "Essay Set 7: Grade 7 narrative/expository "
            "essay about patience. "
            "Prompt: Write a story about a time when "
            "you or someone you know was patient, or "
            "write about patience in your own way. "
            "Scored on 4 traits (0-3 each, ideas doubled): "
            "Ideas (doubled weight): Score 3=clearly "
            "focused, thoroughly developed with specific "
            "relevant details. Score 2=somewhat focused "
            "with mix of specific/general details. "
            "Score 1=minimally focused with limited "
            "details. Score 0=not focused or undeveloped. "
            "Organization: Score 3=clear logical sequence. "
            "Score 2=logically sequenced. "
            "Score 1=weak connections. Score 0=none. "
            "Style: Score 3=compelling word choice and "
            "varied sentence structure. "
            "Score 2=adequate command of language. "
            "Score 1=limited variety. Score 0=ineffective. "
            "Conventions: Score 3=consistent appropriate "
            "grammar, spelling, punctuation. "
            "Resolved score range: 0-30.\n"
            "Query: "
        ),
        8: (
            "Instruct: Represent a student essay for "
            "automated scoring. "
            "Essay Set 8: Grade 10 narrative essay. "
            "Prompt: Tell a true story in which laughter "
            "was one element or part. "
            "Scored on 6 traits (1-6 each): "
            "Ideas and Content: Score 6=exceptionally "
            "clear focused and interesting, strong "
            "support and rich details, thorough "
            "in-depth exploration, well-suited to "
            "audience. Score 5=clear focused interesting, "
            "thorough exploration. Score 4=clear and "
            "focused, main ideas identifiable, "
            "support present. Score 3=some clarity, "
            "main ideas emerging, support inconsistent. "
            "Score 2=limited focus, ideas unclear. "
            "Score 1=minimal focus, very little support. "
            "Organization, Voice, Word Choice, Sentence "
            "Fluency, Conventions also rated 1-6. "
            "Resolved score range: 10-60.\n"
            "Query: "
        ),
    }

    SENT_INSTR = (
        "Instruct: Represent a sentence from a student "
        "essay for measuring coherence, discourse flow "
        "and logical progression between sentences\n"
        "Query: "
    )

    # Find completed chunks
    completed = sorted([
        int(f.replace('chunk_','').replace('.npz',''))
        for f in os.listdir(CHECKPOINT_DIR)
        if f.startswith('chunk_') and f.endswith('.npz')
    ])
    start_idx = (max(completed)+1)*CHUNK_SIZE \
                if completed else 0

    print(f"Total: {n_total}")
    print(f"Completed chunks: {completed}")
    print(f"Starting from: essay {start_idx}")

    if start_idx < n_total:
        MODEL_NAME = 'intfloat/e5-mistral-7b-instruct'
        print(f"\nLoading {MODEL_NAME} (4-bit)...")

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type='nf4'
        )
        tokenizer = AutoTokenizer.from_pretrained(
            MODEL_NAME
        )
        model     = AutoModel.from_pretrained(
            MODEL_NAME,
            quantization_config=bnb_config,
            device_map='cuda',
            torch_dtype=torch.float16
        )
        model.eval()
        print(f"✅ E5-Mistral loaded!")
        print(f"GPU: "
              f"{torch.cuda.mem_get_info()[0]/1024**3:.1f}GB")

        def last_token_pool(last_hidden, attention_mask):
            seq_len = attention_mask.sum(dim=1) - 1
            batch   = last_hidden.shape[0]
            return last_hidden[
                torch.arange(
                    batch,
                    device=last_hidden.device
                ),
                seq_len
            ]

        def encode_batch(inputs, batch_size=4):
            all_embs = []
            for i in range(0, len(inputs), batch_size):
                batch = inputs[i:i+batch_size]
                enc   = tokenizer(
                    batch,
                    max_length=512,
                    padding=True,
                    truncation=True,
                    return_tensors='pt'
                ).to('cuda')
                with torch.no_grad():
                    out  = model(**enc)
                    embs = last_token_pool(
                        out.last_hidden_state,
                        enc['attention_mask']
                    )
                    embs = F.normalize(
                        embs.float(), p=2, dim=1
                    )
                all_embs.append(embs.cpu().numpy())
                del enc, out, embs
                torch.cuda.empty_cache()
            return np.vstack(all_embs)

        n_chunks = (n_total+CHUNK_SIZE-1) // CHUNK_SIZE
        print(f"\nProcessing {n_chunks} chunks...")
        print("Using OFFICIAL ASAP rubrics per set")

        for chunk_id in range(
            start_idx // CHUNK_SIZE, n_chunks
        ):
            chunk_start = chunk_id * CHUNK_SIZE
            chunk_end   = min(
                chunk_start + CHUNK_SIZE, n_total
            )
            chunk_path  = os.path.join(
                CHECKPOINT_DIR, f'chunk_{chunk_id}.npz'
            )

            if os.path.exists(chunk_path):
                print(f"  ⏭️  Chunk {chunk_id} — skip")
                continue

            print(f"\n── Chunk {chunk_id}/{n_chunks-1} "
                  f"({chunk_start}-{chunk_end}) ──")

            chunk_texts = texts[chunk_start:chunk_end]
            chunk_sets  = essay_sets[chunk_start:chunk_end]

            # Each essay gets its set-specific rubric
            print("  Encoding with official rubrics...")
            essay_inputs = [
                RUBRICS[s] + t[:400]
                for s, t in zip(chunk_sets, chunk_texts)
            ]
            essay_embs = encode_batch(
                essay_inputs, batch_size=4
            )

            # Sentence coherence
            print("  Computing coherence...")
            chunk_coh = []

            for text in tqdm(
                chunk_texts,
                desc=f"  Coherence {chunk_id}"
            ):
                sents = sent_tokenize(text)
                if not sents: sents = [text]
                sents = [s[:200] for s in sents[:20]]

                if len(sents) < 2:
                    chunk_coh.append([0.0, 0.0, 0.0])
                    continue

                sent_inputs = [
                    SENT_INSTR + s for s in sents
                ]
                sent_embs = encode_batch(
                    sent_inputs, batch_size=8
                )
                sims = [
                    float(cosine_similarity(
                        sent_embs[i].reshape(1,-1),
                        sent_embs[i+1].reshape(1,-1)
                    )[0][0])
                    for i in range(len(sent_embs)-1)
                ]
                chunk_coh.append([
                    float(np.mean(sims)),
                    float(np.min(sims)),
                    float(np.std(sims))
                ])

            np.savez_compressed(
                chunk_path,
                essay_embs = essay_embs,
                coherence  = np.array(chunk_coh)
            )
            print(f"  💾 Chunk {chunk_id} saved!")
            print(f"  GPU: "
                  f"{torch.cuda.mem_get_info()[0]/1024**3:.1f}GB")

        print("\n✅ All chunks done!")

    # Assemble
    print("\nAssembling final file...")
    n_chunks   = (n_total+CHUNK_SIZE-1) // CHUNK_SIZE
    all_chunks = sorted([
        int(f.replace('chunk_','').replace('.npz',''))
        for f in os.listdir(CHECKPOINT_DIR)
        if f.startswith('chunk_') and f.endswith('.npz')
    ])
    missing = [
        i for i in range(n_chunks)
        if i not in all_chunks
    ]

    if missing:
        print(f"❌ Missing chunks: {missing}")
        print("Rerun to complete missing chunks")
    else:
        all_essay_embs = []
        all_coherence  = []

        for cid in sorted(all_chunks):
            data = np.load(os.path.join(
                CHECKPOINT_DIR, f'chunk_{cid}.npz'
            ))
            all_essay_embs.append(data['essay_embs'])
            all_coherence.append(data['coherence'])

        all_essay_embs = np.vstack(all_essay_embs)
        all_coherence  = np.vstack(all_coherence)
        EMB_DIM        = all_essay_embs.shape[1]

        print(f"Assembled: {all_essay_embs.shape}")

        coh_df       = pd.DataFrame(
            all_coherence,
            columns=['mean_coherence',
                     'min_coherence','std_coherence']
        )
        emb_df       = pd.DataFrame(
            all_essay_embs,
            columns=[f'emb_{i}' for i in range(EMB_DIM)]
        )
        emb_e5rub_df = pd.concat(
            [coh_df, emb_df], axis=1
        )
        emb_e5rub_df.to_csv(EMB_E5RUB_PATH, index=False)

        print(f"\n💾 Saved: {EMB_E5RUB_PATH}")
        print(f"Shape: {emb_e5rub_df.shape}")
        print(f"Nulls: {emb_e5rub_df.isnull().sum().sum()}")
        print("\n✅ Step 4O Complete!")
        print("⚠️  Click Save Version NOW!")

In [ ]:
# ============================================================
# STEP 5O — Feature Matrix with E5+Rubric Embeddings
# ============================================================

import os
import numpy as np
import pandas as pd

SAVE_DIR        = '/kaggle/working/'
EMB_E5RUB_PATH  = os.path.join(
    SAVE_DIR, 'train_embeddings_e5_rubric.csv'
)
FULL_E5RUB_PATH = os.path.join(
    SAVE_DIR, 'train_full_features_e5rubric.parquet'
)

if os.path.exists(FULL_E5RUB_PATH):
    print("✅ Already exists!")
    train_e5rub = pd.read_parquet(FULL_E5RUB_PATH)
    print("Shape:", train_e5rub.shape)

else:
    emb_e5rub_df = pd.read_csv(EMB_E5RUB_PATH)
    EMB_DIM      = len([
        c for c in emb_e5rub_df.columns
        if c.startswith('emb_')
    ])
    print(f"Embedding dim: {EMB_DIM}")

    SF_COLS  = [
        'gunning_fog','flesch_re','ari','guiraud_index',
        'noun_freq','verb_freq','avg_sent_len',
        'pronoun_density','pronoun_noun_ratio','giveness'
    ]
    EG_COLS  = [
        f"{r1}_{r2}"
        for r1 in ['S','O','X','-']
        for r2 in ['S','O','X','-']
    ]
    EMB_COLS = (
        ['mean_coherence','min_coherence','std_coherence'] +
        [f'emb_{i}' for i in range(EMB_DIM)]
    )
    META_COLS = [
        'essay_id','essay_set','essay',
        'domain1_score','rater1_domain1','rater2_domain1'
    ]

    sf_df    = pd.read_csv(
        os.path.join(SAVE_DIR,
                     'train_shallow_features.csv'),
        encoding='latin-1'
    )
    eg_df    = pd.read_csv(
        os.path.join(SAVE_DIR,
                     'train_entity_grid_features.csv')
    )

    train_e5rub = pd.concat([
        sf_df[META_COLS].reset_index(drop=True),
        sf_df[SF_COLS].reset_index(drop=True),
        eg_df[EG_COLS].reset_index(drop=True),
        emb_e5rub_df[EMB_COLS].reset_index(drop=True),
    ], axis=1)

    train_e5rub.to_parquet(FULL_E5RUB_PATH, index=False)
    print(f"✅ Saved: {train_e5rub.shape}")
    print("⚠️  Click Save Version NOW!")

In [ ]:
# ============================================================
# STEP 6 V21 — E5-Mistral+Rubric vs E5-Mistral vs V9
# THREE configs to find best LLM-based approach
# ============================================================

import os
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import cohen_kappa_score
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

SAVE_DIR         = '/kaggle/working/'
RESULTS_V21_PATH = os.path.join(
    SAVE_DIR, 'results_v21.csv'
)

if os.path.exists(RESULTS_V21_PATH):
    print("✅ V21 results exist!")
    results_df = pd.read_csv(RESULTS_V21_PATH)
    print(results_df.to_string(index=False))

else:
    # Load E5+Rubric features
    emb_e5rub_df  = pd.read_csv(EMB_E5RUB_PATH)
    EMB_DIM_E5RUB = len([
        c for c in emb_e5rub_df.columns
        if c.startswith('emb_')
    ])
    # Load E5 original features (no rubric)
    emb_e5_df  = pd.read_csv(
        os.path.join(SAVE_DIR,
                     'train_embeddings_e5mistral.csv')
    )
    EMB_DIM_E5 = len([
        c for c in emb_e5_df.columns
        if c.startswith('emb_')
    ])

    print("Loading feature matrices...")
    train_e5rub = pd.read_parquet(FULL_E5RUB_PATH)
    train_e5    = pd.read_parquet(
        os.path.join(SAVE_DIR,
            'train_full_features_e5mistral.parquet')
    )
    train_inst  = pd.read_parquet(
        os.path.join(SAVE_DIR,
            'train_full_features_instructor.parquet')
    )
    print(f"✅ E5+Rubric:   {train_e5rub.shape}")
    print(f"✅ E5 Original: {train_e5.shape}")
    print(f"✅ INSTRUCTOR:  {train_inst.shape}")

    SF_COLS = [
        'gunning_fog','flesch_re','ari','guiraud_index',
        'noun_freq','verb_freq','avg_sent_len',
        'pronoun_density','pronoun_noun_ratio','giveness'
    ]
    EG_COLS = [
        f"{r1}_{r2}"
        for r1 in ['S','O','X','-']
        for r2 in ['S','O','X','-']
    ]
    E5RUB_FEAT = (
        SF_COLS + EG_COLS +
        ['mean_coherence','min_coherence','std_coherence'] +
        [f'emb_{i}' for i in range(EMB_DIM_E5RUB)]
    )
    E5_FEAT    = (
        SF_COLS + EG_COLS +
        ['mean_coherence','min_coherence','std_coherence'] +
        [f'emb_{i}' for i in range(EMB_DIM_E5)]
    )
    INST_FEAT  = (
        SF_COLS + EG_COLS +
        ['mean_coherence','min_coherence','std_coherence'] +
        [f'emb_{i}' for i in range(1024)]
    )

    def qwk(y_true, y_pred):
        return cohen_kappa_score(
            y_true, y_pred, weights='quadratic'
        )
    def clip_round(y_pred, y_min, y_max):
        return np.clip(
            np.round(y_pred).astype(int), y_min, y_max
        )
    def add_eg_sim(X_tr, X_te, y_tr,
                   eg_tr, eg_te, k=5):
        s_te = cosine_similarity(eg_te, eg_tr)
        s_tr = cosine_similarity(eg_tr, eg_tr)
        np.fill_diagonal(s_tr, 0)
        def f(s, y):
            out = []
            for i in range(len(s)):
                idx = np.argsort(s[i])[-k:]
                out.append([
                    np.mean(y[idx]),
                    np.mean(s[i][idx]),
                    np.max(s[i][idx]),
                    np.std(y[idx])
                ])
            return np.array(out)
        return (
            np.hstack([X_tr, f(s_tr, y_tr)]),
            np.hstack([X_te, f(s_te, y_tr)])
        )

    SCORE_RANGES = {
        1:(2,12), 2:(1,6),  3:(0,3),
        4:(0,3),  5:(0,4),  6:(0,4),
        7:(2,24), 8:(10,60),
    }
    SET_PARAMS = {
        1: dict(n_estimators=1000, learning_rate=0.02,
                max_depth=7, subsample=0.8,
                colsample_bytree=0.7, min_child_weight=3,
                reg_alpha=0.1, reg_lambda=1.0),
        2: dict(n_estimators=1000, learning_rate=0.02,
                max_depth=6, subsample=0.8,
                colsample_bytree=0.7, min_child_weight=3,
                reg_alpha=0.1, reg_lambda=1.0),
        3: dict(n_estimators=800, learning_rate=0.03,
                max_depth=5, subsample=0.8,
                colsample_bytree=0.8, min_child_weight=2,
                reg_alpha=0.05, reg_lambda=1.0),
        4: dict(n_estimators=800, learning_rate=0.03,
                max_depth=5, subsample=0.8,
                colsample_bytree=0.8, min_child_weight=2,
                reg_alpha=0.05, reg_lambda=1.0),
        5: dict(n_estimators=800, learning_rate=0.03,
                max_depth=5, subsample=0.8,
                colsample_bytree=0.8, min_child_weight=2,
                reg_alpha=0.05, reg_lambda=1.0),
        6: dict(n_estimators=800, learning_rate=0.03,
                max_depth=5, subsample=0.8,
                colsample_bytree=0.8, min_child_weight=2,
                reg_alpha=0.05, reg_lambda=1.0),
        7: dict(n_estimators=1000, learning_rate=0.02,
                max_depth=7, subsample=0.8,
                colsample_bytree=0.7, min_child_weight=3,
                reg_alpha=0.1, reg_lambda=1.5),
        8: dict(n_estimators=1500, learning_rate=0.01,
                max_depth=5, subsample=0.7,
                colsample_bytree=0.6, min_child_weight=5,
                reg_alpha=0.5, reg_lambda=2.0),
    }

    # Resume support
    completed_sets = []
    if os.path.exists(RESULTS_V21_PATH):
        existing       = pd.read_csv(RESULTS_V21_PATH)
        completed_sets = existing['essay_set'].tolist()
        all_results    = existing.to_dict('records')
        print(f"✅ Resuming: {completed_sets}")
    else:
        all_results = []

    paper_qwk = {
        1:0.91,2:0.79,3:0.71,
        4:0.84,5:0.84,6:0.84,
        7:0.87,8:0.92
    }
    v9_qwk = {
        1:0.775,2:0.644,3:0.672,
        4:0.802,5:0.792,6:0.782,
        7:0.787,8:0.698
    }

    print("\n" + "="*65)
    print("  V21 — E5-Mistral + Official ASAP Rubrics")
    print("  Testing 3 configs:")
    print("  A: E5+Rubric alone")
    print("  B: E5+Rubric + Original E5 (50/50)")
    print("  C: E5+Rubric + INSTRUCTOR (50/50)")
    print("="*65)

    # Store per-config results
    results_A = {}
    results_B = {}
    results_C = {}

    for essay_set in sorted(
        train_e5rub['essay_set'].unique()
    ):
        if essay_set in completed_sets:
            print(f"⏭️  Set {essay_set} done")
            continue

        sub_er = train_e5rub[
            train_e5rub['essay_set']==essay_set
        ].reset_index(drop=True)
        sub_e5 = train_e5[
            train_e5['essay_set']==essay_set
        ].reset_index(drop=True)
        sub_in = train_inst[
            train_inst['essay_set']==essay_set
        ].reset_index(drop=True)

        Xer  = sub_er[E5RUB_FEAT].values.astype(np.float32)
        Xe5  = sub_e5[E5_FEAT].values.astype(np.float32)
        Xi   = sub_in[INST_FEAT].values.astype(np.float32)
        y    = sub_er['domain1_score'].values.astype(
            np.float32
        )
        eg   = sub_er[EG_COLS].values.astype(np.float32)

        y_min, y_max = SCORE_RANGES[essay_set]
        score_range  = y_max - y_min
        params       = SET_PARAMS[essay_set]

        print(f"\n── Set {essay_set} "
              f"({len(sub_er)} essays, "
              f"{y_min}-{y_max}) ──")

        kf = KFold(
            n_splits=10, shuffle=True, random_state=42
        )
        folds_A, folds_B, folds_C = [], [], []

        for fold_num, (tr_idx, te_idx) in enumerate(
            kf.split(Xer), 1
        ):
            y_tr      = y[tr_idx]
            y_te      = y[te_idx]
            y_tr_norm = (y_tr - y_min) / score_range
            eg_tr     = eg[tr_idx]
            eg_te     = eg[te_idx]

            def train_predict(X):
                Xtr, Xte = add_eg_sim(
                    X[tr_idx], X[te_idx],
                    y_tr, eg_tr, eg_te
                )
                m = XGBRegressor(
                    **params, random_state=42,
                    n_jobs=-1, tree_method='hist',
                    device='cuda', verbosity=0
                )
                m.fit(Xtr, y_tr_norm)
                return m.predict(Xte)

            p_er = train_predict(Xer)  # E5+Rubric
            p_e5 = train_predict(Xe5)  # E5 original
            p_in = train_predict(Xi)   # INSTRUCTOR

            # Config A: E5+Rubric alone
            yA = clip_round(
                p_er*score_range+y_min, y_min, y_max
            )
            # Config B: E5+Rubric + E5 original
            yB = clip_round(
                (0.5*p_er+0.5*p_e5)*score_range+y_min,
                y_min, y_max
            )
            # Config C: E5+Rubric + INSTRUCTOR
            yC = clip_round(
                (0.5*p_er+0.5*p_in)*score_range+y_min,
                y_min, y_max
            )
            y_true = y_te.astype(int)

            qA = qwk(y_true, yA)
            qB = qwk(y_true, yB)
            qC = qwk(y_true, yC)

            folds_A.append(qA)
            folds_B.append(qB)
            folds_C.append(qC)

            print(f"  Fold {fold_num:2d} → "
                  f"A(E5+Rub)={qA:.3f}  "
                  f"B(+E5)={qB:.3f}  "
                  f"C(+INST)={qC:.3f}")

        mA = np.mean(folds_A)
        mB = np.mean(folds_B)
        mC = np.mean(folds_C)

        results_A[essay_set] = mA
        results_B[essay_set] = mB
        results_C[essay_set] = mC

        best = 'A' if mA>=mB and mA>=mC else \
               'B' if mB>=mA and mB>=mC else 'C'
        v9   = v9_qwk[essay_set]

        print(f"  {'─'*52}")
        print(f"  MEAN → A={mA:.3f} B={mB:.3f} "
              f"C={mC:.3f}  "
              f"Best={best}  V9={v9:.3f}")

        all_results.append({
            'essay_set'       : essay_set,
            'n_essays'        : len(sub_er),
            'score_range'     : f"{y_min}–{y_max}",
            'e5_rubric_alone' : round(mA, 3),
            'e5_rubric_e5'    : round(mB, 3),
            'e5_rubric_inst'  : round(mC, 3),
            'v9_instructor'   : v9_qwk[essay_set],
            'paper'           : paper_qwk[essay_set],
        })

        pd.DataFrame(all_results).to_csv(
            RESULTS_V21_PATH, index=False
        )
        print(f"  💾 Saved (set {essay_set})")

    results_df = pd.DataFrame(all_results)

# Final comparison
paper_qwk = {
    1:0.91,2:0.79,3:0.71,
    4:0.84,5:0.84,6:0.84,
    7:0.87,8:0.92
}
paper_avg = np.mean(list(paper_qwk.values()))
v9_qwk    = {
    1:0.775,2:0.644,3:0.672,
    4:0.802,5:0.792,6:0.782,
    7:0.787,8:0.698
}

print("\n" + "="*70)
print("   E5-Mistral + OFFICIAL ASAP RUBRICS — FINAL")
print("   A=E5+Rubric alone  B=+E5orig  C=+INSTRUCTOR")
print("="*70)
print(f"{'Set':<4} {'V9':>7} {'A':>7} "
      f"{'B':>7} {'C':>7} {'Paper':>7}")
print("─"*45)

best_qwks = []
for _, row in results_df.iterrows():
    s  = int(row['essay_set'])
    mA = row['e5_rubric_alone']
    mB = row['e5_rubric_e5']
    mC = row['e5_rubric_inst']
    best = max(mA, mB, mC)
    best_qwks.append(best)
    v9 = v9_qwk[s]
    flag = "✅" if best > v9 else "⚠️"
    print(f"  {s:<3} {v9:>7.3f} {mA:>7.3f} "
          f"{mB:>7.3f} {mC:>7.3f} "
          f"{paper_qwk[s]:>7.3f} {flag}")

print("─"*45)
best_avg = np.mean(best_qwks)
print(f"  {'Avg':<3} {'0.744':>7} "
      f"{np.mean([r['e5_rubric_alone'] for _,r in results_df.iterrows()]):>7.3f} "
      f"{np.mean([r['e5_rubric_e5'] for _,r in results_df.iterrows()]):>7.3f} "
      f"{np.mean([r['e5_rubric_inst'] for _,r in results_df.iterrows()]):>7.3f} "
      f"{paper_avg:>7.3f}")

print("\n" + "="*70)
print(results_df.to_string(index=False))
print("\n✅ V21 Complete!")
print("⚠️  Click Save Version NOW!")

In [ ]:
# ============================================================
# STEP 4O v2 — E5-Mistral + Official Rubrics
# FIXED: 2048 token limit (fits rubric + full essay)
# ============================================================

import os, gc, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import (
    AutoTokenizer, AutoModel, BitsAndBytesConfig
)
from nltk.tokenize import sent_tokenize
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
warnings.filterwarnings('ignore')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = \
    'expandable_segments:True'

SAVE_DIR        = '/kaggle/working/'
EMB_E5RUB_PATH  = os.path.join(
    SAVE_DIR, 'train_embeddings_e5_rubric_2048.csv'
)
CHECKPOINT_DIR  = os.path.join(
    SAVE_DIR, 'e5rubric2048_checkpoints'
)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

if os.path.exists(EMB_E5RUB_PATH):
    print("✅ E5+Rubric 2048 embeddings exist!")
    emb_e5rub_df = pd.read_csv(EMB_E5RUB_PATH)
    print("Shape:", emb_e5rub_df.shape)

else:
    gc.collect()
    torch.cuda.empty_cache()

    sf_df = pd.read_csv(
        os.path.join(SAVE_DIR,'train_shallow_features.csv'),
        encoding='latin-1'
    )
    texts      = sf_df['essay'].astype(str).tolist()
    essay_sets = sf_df['essay_set'].tolist()
    n_total    = len(texts)
    CHUNK_SIZE = 200  # smaller chunks — 2048 needs more memory

    # Official ASAP rubrics (same as before)
    RUBRICS = {
        1: (
            "Instruct: Represent a student essay for "
            "automated scoring. "
            "Essay Set 1: Grade 8 persuasive essay. "
            "Prompt: Write a letter to your local "
            "newspaper stating your opinion on the "
            "effects computers have on people. "
            "Scoring: Two raters each score 1-6, "
            "resolved score range 2-12. "
            "Score 1-2 (low): Undeveloped, vague details, "
            "awkward/fragmented, no audience awareness. "
            "Score 3-4 (medium-low): Minimal development, "
            "inadequate support, some organization. "
            "Score 5-6 (medium-high): Adequately developed, "
            "mix of general and specific details. "
            "Score 7-8 (high): Well-developed position, "
            "specific details, clear organization. "
            "Score 9-10 (very high): Thoroughly developed, "
            "compelling argument, strong organization. "
            "Score 11-12 (exceptional): Outstanding, "
            "distinctive quality, insightful.\n"
            "Query: "
        ),
        2: (
            "Instruct: Represent a student essay for "
            "automated scoring. "
            "Essay Set 2: Grade 10 persuasive essay. "
            "Prompt: Write a persuasive essay about "
            "censorship in libraries. "
            "Domain 1 (Writing Applications) range 1-6. "
            "Domain 2 (Language Conventions) range 1-4. "
            "Score 6: Fully accomplishes task, distinctive "
            "quality, exceptional details, in-depth. "
            "Score 4-5: Accomplishes task well, relevant "
            "supporting details, effective organization. "
            "Score 2-3: Partially accomplishes task, "
            "some relevant ideas, basic language. "
            "Score 1: Minimally accomplishes task.\n"
            "Query: "
        ),
        3: (
            "Instruct: Represent a student essay for "
            "automated scoring. "
            "Essay Set 3: Grade 10 source-dependent "
            "response to Rough Road Ahead by Joe Kurmaskie. "
            "Score range: 0-3. "
            "Score 3: Accurate complete response with "
            "specific text evidence and clear understanding. "
            "Score 2: Mostly accurate with some evidence. "
            "Score 1: Partially accurate or vague. "
            "Score 0: Inaccurate or off-topic.\n"
            "Query: "
        ),
        4: (
            "Instruct: Represent a student essay for "
            "automated scoring. "
            "Essay Set 4: Grade 10 source-dependent "
            "response to Winter Hibiscus by Minfong Ho. "
            "Score range: 0-3. "
            "Score 3: Accurate complete response with "
            "specific textual evidence and clear "
            "understanding of character and themes. "
            "Score 2: Mostly accurate with some evidence. "
            "Score 1: Partially accurate, vague. "
            "Score 0: Inaccurate or missing.\n"
            "Query: "
        ),
        5: (
            "Instruct: Represent a student essay for "
            "automated scoring. "
            "Essay Set 5: Grade 8 source-dependent "
            "response to Narciso Rodriguez from Home: "
            "The Blueprints of Our Lives. "
            "Score range: 0-4. "
            "Score 4: Insightful analysis, specific "
            "accurate textual evidence, thorough "
            "understanding of themes. "
            "Score 3: Adequate analysis with evidence. "
            "Score 2: Basic understanding, limited evidence. "
            "Score 0-1: Minimal or inaccurate.\n"
            "Query: "
        ),
        6: (
            "Instruct: Represent a student essay for "
            "automated scoring. "
            "Essay Set 6: Grade 10 source-dependent "
            "response to The Mooring Mast by "
            "Marcia Amidon Lusted. "
            "Score range: 0-4. "
            "Score 4: Sophisticated understanding of "
            "author techniques with specific evidence. "
            "Score 3: Adequate understanding with evidence. "
            "Score 2: Basic identification, limited evidence. "
            "Score 0-1: Minimal understanding.\n"
            "Query: "
        ),
        7: (
            "Instruct: Represent a student essay for "
            "automated scoring. "
            "Essay Set 7: Grade 7 narrative about patience. "
            "Scored on 4 traits (Ideas doubled): "
            "Ideas Score 3: clearly focused, thoroughly "
            "developed with specific relevant details. "
            "Ideas Score 2: somewhat focused, mix of details. "
            "Ideas Score 1: minimally focused, limited. "
            "Organization Score 3: clear logical sequence. "
            "Style Score 3: compelling varied language. "
            "Conventions Score 3: consistent grammar. "
            "Resolved score range: 0-30.\n"
            "Query: "
        ),
        8: (
            "Instruct: Represent a student essay for "
            "automated scoring. "
            "Essay Set 8: Grade 10 narrative about laughter. "
            "Scored on 6 traits (1-6 each): "
            "Ideas Score 6: exceptionally clear focused, "
            "strong support, rich details, thorough. "
            "Ideas Score 4: clear focused, main ideas "
            "identifiable, support present. "
            "Ideas Score 2: limited focus, unclear. "
            "Organization, Voice, Word Choice, Sentence "
            "Fluency, Conventions each rated 1-6. "
            "Resolved score range: 10-60.\n"
            "Query: "
        ),
    }

    SENT_INSTR = (
        "Instruct: Represent a sentence from a student "
        "essay for measuring coherence and discourse flow\n"
        "Query: "
    )

    # Find completed chunks
    completed = sorted([
        int(f.replace('chunk_','').replace('.npz',''))
        for f in os.listdir(CHECKPOINT_DIR)
        if f.startswith('chunk_') and f.endswith('.npz')
    ])
    start_idx = (max(completed)+1)*CHUNK_SIZE \
                if completed else 0

    print(f"Total essays:      {n_total}")
    print(f"Completed chunks:  {completed}")
    print(f"Starting from:     essay {start_idx}")
    print(f"Token limit:       2048 (was 512)")
    print(f"Chunk size:        {CHUNK_SIZE} (smaller for memory)")

    if start_idx < n_total:
        MODEL_NAME = 'intfloat/e5-mistral-7b-instruct'
        print(f"\nLoading {MODEL_NAME} (4-bit)...")

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type='nf4'
        )
        tokenizer = AutoTokenizer.from_pretrained(
            MODEL_NAME
        )
        model     = AutoModel.from_pretrained(
            MODEL_NAME,
            quantization_config=bnb_config,
            device_map='cuda',
            torch_dtype=torch.float16
        )
        model.eval()
        print(f"✅ E5-Mistral loaded (2048 ctx)!")
        print(f"GPU: "
              f"{torch.cuda.mem_get_info()[0]/1024**3:.1f}GB")

        def last_token_pool(last_hidden, attention_mask):
            seq_len = attention_mask.sum(dim=1) - 1
            batch   = last_hidden.shape[0]
            return last_hidden[
                torch.arange(
                    batch,
                    device=last_hidden.device
                ),
                seq_len
            ]

        def encode_batch(inputs, max_len=2048,
                         batch_size=2):
            """
            FIX: max_len=2048 fits rubric + full essay
            batch_size=2 because 2048 tokens needs more GPU
            """
            all_embs = []
            for i in range(0, len(inputs), batch_size):
                batch = inputs[i:i+batch_size]
                enc   = tokenizer(
                    batch,
                    max_length=max_len,   # ← KEY FIX
                    padding=True,
                    truncation=True,
                    return_tensors='pt'
                ).to('cuda')
                with torch.no_grad():
                    out  = model(**enc)
                    embs = last_token_pool(
                        out.last_hidden_state,
                        enc['attention_mask']
                    )
                    embs = F.normalize(
                        embs.float(), p=2, dim=1
                    )
                all_embs.append(embs.cpu().numpy())
                del enc, out, embs
                torch.cuda.empty_cache()
            return np.vstack(all_embs)

        n_chunks = (n_total+CHUNK_SIZE-1) // CHUNK_SIZE
        print(f"\nProcessing {n_chunks} chunks...")

        for chunk_id in range(
            start_idx // CHUNK_SIZE, n_chunks
        ):
            chunk_start = chunk_id * CHUNK_SIZE
            chunk_end   = min(
                chunk_start + CHUNK_SIZE, n_total
            )
            chunk_path  = os.path.join(
                CHECKPOINT_DIR, f'chunk_{chunk_id}.npz'
            )

            if os.path.exists(chunk_path):
                print(f"  ⏭️  Chunk {chunk_id} — skip")
                continue

            print(f"\n── Chunk {chunk_id}/{n_chunks-1} "
                  f"({chunk_start}-{chunk_end}) ──")

            chunk_texts = texts[chunk_start:chunk_end]
            chunk_sets  = essay_sets[chunk_start:chunk_end]

            # Full essay with rubric — no truncation issue
            print("  Encoding (2048 tokens)...")
            essay_inputs = [
                RUBRICS[s] + t  # no [:400] truncation!
                for s, t in zip(chunk_sets, chunk_texts)
            ]
            essay_embs = encode_batch(
                essay_inputs,
                max_len=2048,   # ← full context
                batch_size=2    # ← small batch for memory
            )

            # Sentence coherence (short, 256 is fine)
            print("  Computing coherence...")
            chunk_coh = []

            for text in tqdm(
                chunk_texts,
                desc=f"  Coherence {chunk_id}"
            ):
                sents = sent_tokenize(text)
                if not sents: sents = [text]
                sents = [s[:200] for s in sents[:20]]

                if len(sents) < 2:
                    chunk_coh.append([0.0, 0.0, 0.0])
                    continue

                sent_inputs = [
                    SENT_INSTR + s for s in sents
                ]
                sent_embs = encode_batch(
                    sent_inputs,
                    max_len=256,    # sentences are short
                    batch_size=8
                )
                sims = [
                    float(cosine_similarity(
                        sent_embs[i].reshape(1,-1),
                        sent_embs[i+1].reshape(1,-1)
                    )[0][0])
                    for i in range(len(sent_embs)-1)
                ]
                chunk_coh.append([
                    float(np.mean(sims)),
                    float(np.min(sims)),
                    float(np.std(sims))
                ])

            np.savez_compressed(
                chunk_path,
                essay_embs = essay_embs,
                coherence  = np.array(chunk_coh)
            )
            print(f"  💾 Chunk {chunk_id} saved!")
            print(f"  GPU: "
                  f"{torch.cuda.mem_get_info()[0]/1024**3:.1f}GB")

        print("\n✅ All chunks done!")

    # Assemble
    print("\nAssembling...")
    n_chunks   = (n_total+CHUNK_SIZE-1) // CHUNK_SIZE
    all_chunks = sorted([
        int(f.replace('chunk_','').replace('.npz',''))
        for f in os.listdir(CHECKPOINT_DIR)
        if f.startswith('chunk_') and f.endswith('.npz')
    ])
    missing = [
        i for i in range(n_chunks)
        if i not in all_chunks
    ]

    if missing:
        print(f"❌ Missing chunks: {missing}")
        print("Rerun to complete")
    else:
        all_essay_embs = []
        all_coherence  = []
        for cid in sorted(all_chunks):
            data = np.load(os.path.join(
                CHECKPOINT_DIR, f'chunk_{cid}.npz'
            ))
            all_essay_embs.append(data['essay_embs'])
            all_coherence.append(data['coherence'])

        all_essay_embs = np.vstack(all_essay_embs)
        all_coherence  = np.vstack(all_coherence)
        EMB_DIM        = all_essay_embs.shape[1]

        coh_df       = pd.DataFrame(
            all_coherence,
            columns=['mean_coherence',
                     'min_coherence','std_coherence']
        )
        emb_df       = pd.DataFrame(
            all_essay_embs,
            columns=[f'emb_{i}' for i in range(EMB_DIM)]
        )
        emb_e5rub_df = pd.concat(
            [coh_df, emb_df], axis=1
        )
        emb_e5rub_df.to_csv(EMB_E5RUB_PATH, index=False)
        print(f"\n💾 Saved: {EMB_E5RUB_PATH}")
        print(f"Shape: {emb_e5rub_df.shape}")
        print("\n✅ Step 4O v2 Complete!")
        print("⚠️  Click Save Version NOW!")

In [ ]:
# ============================================================
# STEP 5O v2 — Feature Matrix
# ============================================================

import os
import pandas as pd

SAVE_DIR          = '/kaggle/working/'
EMB_E5RUB_PATH    = os.path.join(
    SAVE_DIR, 'train_embeddings_e5_rubric_2048.csv'
)
FULL_E5RUB2_PATH  = os.path.join(
    SAVE_DIR,
    'train_full_features_e5rubric_2048.parquet'
)

if os.path.exists(FULL_E5RUB2_PATH):
    print("✅ Already exists!")
    train_e5rub2 = pd.read_parquet(FULL_E5RUB2_PATH)
    print("Shape:", train_e5rub2.shape)

else:
    emb_e5rub_df = pd.read_csv(EMB_E5RUB_PATH)
    EMB_DIM      = len([
        c for c in emb_e5rub_df.columns
        if c.startswith('emb_')
    ])
    print(f"Embedding dim: {EMB_DIM}")

    SF_COLS  = [
        'gunning_fog','flesch_re','ari','guiraud_index',
        'noun_freq','verb_freq','avg_sent_len',
        'pronoun_density','pronoun_noun_ratio','giveness'
    ]
    EG_COLS  = [
        f"{r1}_{r2}"
        for r1 in ['S','O','X','-']
        for r2 in ['S','O','X','-']
    ]
    EMB_COLS = (
        ['mean_coherence','min_coherence','std_coherence'] +
        [f'emb_{i}' for i in range(EMB_DIM)]
    )
    META_COLS = [
        'essay_id','essay_set','essay',
        'domain1_score','rater1_domain1','rater2_domain1'
    ]

    sf_df    = pd.read_csv(
        os.path.join(SAVE_DIR,
                     'train_shallow_features.csv'),
        encoding='latin-1'
    )
    eg_df    = pd.read_csv(
        os.path.join(SAVE_DIR,
                     'train_entity_grid_features.csv')
    )

    train_e5rub2 = pd.concat([
        sf_df[META_COLS].reset_index(drop=True),
        sf_df[SF_COLS].reset_index(drop=True),
        eg_df[EG_COLS].reset_index(drop=True),
        emb_e5rub_df[EMB_COLS].reset_index(drop=True),
    ], axis=1)

    train_e5rub2.to_parquet(FULL_E5RUB2_PATH, index=False)
    print(f"✅ Saved: {train_e5rub2.shape}")
    print("⚠️  Click Save Version NOW!")

In [ ]:
# ============================================================
# STEP 6 V22 — E5+Rubric(2048) Configs
# ============================================================

import os
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import cohen_kappa_score
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

SAVE_DIR          = '/kaggle/working/'
FULL_E5RUB2_PATH  = os.path.join(
    SAVE_DIR,
    'train_full_features_e5rubric_2048.parquet'
)
RESULTS_V22_PATH  = os.path.join(
    SAVE_DIR, 'results_v22.csv'
)

if os.path.exists(RESULTS_V22_PATH):
    print("✅ V22 results exist!")
    results_df = pd.read_csv(RESULTS_V22_PATH)
    print(results_df.to_string(index=False))

else:
    emb_e5rub_df   = pd.read_csv(
        os.path.join(SAVE_DIR,
                     'train_embeddings_e5_rubric_2048.csv')
    )
    EMB_DIM_E5RUB2 = len([
        c for c in emb_e5rub_df.columns
        if c.startswith('emb_')
    ])

    print("Loading feature matrices...")
    train_e5rub2 = pd.read_parquet(FULL_E5RUB2_PATH)
    train_inst   = pd.read_parquet(
        os.path.join(SAVE_DIR,
            'train_full_features_instructor.parquet')
    )
    print(f"✅ E5+Rubric(2048): {train_e5rub2.shape}")
    print(f"✅ INSTRUCTOR:      {train_inst.shape}")

    SF_COLS = [
        'gunning_fog','flesch_re','ari','guiraud_index',
        'noun_freq','verb_freq','avg_sent_len',
        'pronoun_density','pronoun_noun_ratio','giveness'
    ]
    EG_COLS = [
        f"{r1}_{r2}"
        for r1 in ['S','O','X','-']
        for r2 in ['S','O','X','-']
    ]
    E5RUB2_FEAT = (
        SF_COLS + EG_COLS +
        ['mean_coherence','min_coherence','std_coherence'] +
        [f'emb_{i}' for i in range(EMB_DIM_E5RUB2)]
    )
    INST_FEAT   = (
        SF_COLS + EG_COLS +
        ['mean_coherence','min_coherence','std_coherence'] +
        [f'emb_{i}' for i in range(1024)]
    )

    def qwk(y_true, y_pred):
        return cohen_kappa_score(
            y_true, y_pred, weights='quadratic'
        )
    def clip_round(y_pred, y_min, y_max):
        return np.clip(
            np.round(y_pred).astype(int), y_min, y_max
        )
    def add_eg_sim(X_tr, X_te, y_tr,
                   eg_tr, eg_te, k=5):
        s_te = cosine_similarity(eg_te, eg_tr)
        s_tr = cosine_similarity(eg_tr, eg_tr)
        np.fill_diagonal(s_tr, 0)
        def f(s, y):
            out = []
            for i in range(len(s)):
                idx = np.argsort(s[i])[-k:]
                out.append([
                    np.mean(y[idx]),
                    np.mean(s[i][idx]),
                    np.max(s[i][idx]),
                    np.std(y[idx])
                ])
            return np.array(out)
        return (
            np.hstack([X_tr, f(s_tr, y_tr)]),
            np.hstack([X_te, f(s_te, y_tr)])
        )

    SCORE_RANGES = {
        1:(2,12), 2:(1,6),  3:(0,3),
        4:(0,3),  5:(0,4),  6:(0,4),
        7:(2,24), 8:(10,60),
    }
    SET_PARAMS = {
        1: dict(n_estimators=1000, learning_rate=0.02,
                max_depth=7, subsample=0.8,
                colsample_bytree=0.7, min_child_weight=3,
                reg_alpha=0.1, reg_lambda=1.0),
        2: dict(n_estimators=1000, learning_rate=0.02,
                max_depth=6, subsample=0.8,
                colsample_bytree=0.7, min_child_weight=3,
                reg_alpha=0.1, reg_lambda=1.0),
        3: dict(n_estimators=800, learning_rate=0.03,
                max_depth=5, subsample=0.8,
                colsample_bytree=0.8, min_child_weight=2,
                reg_alpha=0.05, reg_lambda=1.0),
        4: dict(n_estimators=800, learning_rate=0.03,
                max_depth=5, subsample=0.8,
                colsample_bytree=0.8, min_child_weight=2,
                reg_alpha=0.05, reg_lambda=1.0),
        5: dict(n_estimators=800, learning_rate=0.03,
                max_depth=5, subsample=0.8,
                colsample_bytree=0.8, min_child_weight=2,
                reg_alpha=0.05, reg_lambda=1.0),
        6: dict(n_estimators=800, learning_rate=0.03,
                max_depth=5, subsample=0.8,
                colsample_bytree=0.8, min_child_weight=2,
                reg_alpha=0.05, reg_lambda=1.0),
        7: dict(n_estimators=1000, learning_rate=0.02,
                max_depth=7, subsample=0.8,
                colsample_bytree=0.7, min_child_weight=3,
                reg_alpha=0.1, reg_lambda=1.5),
        8: dict(n_estimators=1500, learning_rate=0.01,
                max_depth=5, subsample=0.7,
                colsample_bytree=0.6, min_child_weight=5,
                reg_alpha=0.5, reg_lambda=2.0),
    }

    completed_sets = []
    if os.path.exists(RESULTS_V22_PATH):
        existing       = pd.read_csv(RESULTS_V22_PATH)
        completed_sets = existing['essay_set'].tolist()
        all_results    = existing.to_dict('records')
        print(f"✅ Resuming: {completed_sets}")
    else:
        all_results = []

    paper_qwk = {
        1:0.91,2:0.79,3:0.71,
        4:0.84,5:0.84,6:0.84,
        7:0.87,8:0.92
    }
    v9_qwk = {
        1:0.775,2:0.644,3:0.672,
        4:0.802,5:0.792,6:0.782,
        7:0.787,8:0.698
    }
    v21c_qwk = {
        1:0.763,2:0.624,3:0.652,
        4:0.771,5:0.765,6:0.718,
        7:0.768,8:0.672
    }

    print("\n" + "="*65)
    print("  V22 — E5+Rubric(2048 tokens)")
    print("  FIX: Full essay fits without truncation")
    print("  A: E5+Rubric(2048) alone")
    print("  B: E5+Rubric(2048) + INSTRUCTOR (50/50)")
    print("="*65)

    results_A = {}
    results_B = {}

    for essay_set in sorted(
        train_e5rub2['essay_set'].unique()
    ):
        if essay_set in completed_sets:
            print(f"⏭️  Set {essay_set} done")
            continue

        sub_er = train_e5rub2[
            train_e5rub2['essay_set']==essay_set
        ].reset_index(drop=True)
        sub_in = train_inst[
            train_inst['essay_set']==essay_set
        ].reset_index(drop=True)

        Xer = sub_er[E5RUB2_FEAT].values.astype(
            np.float32
        )
        Xi  = sub_in[INST_FEAT].values.astype(np.float32)
        y   = sub_er['domain1_score'].values.astype(
            np.float32
        )
        eg  = sub_er[EG_COLS].values.astype(np.float32)

        y_min, y_max = SCORE_RANGES[essay_set]
        score_range  = y_max - y_min
        params       = SET_PARAMS[essay_set]

        print(f"\n── Set {essay_set} "
              f"({len(sub_er)} essays, "
              f"{y_min}-{y_max}) ──")

        kf = KFold(
            n_splits=10, shuffle=True, random_state=42
        )
        folds_A, folds_B = [], []

        for fold_num, (tr_idx, te_idx) in enumerate(
            kf.split(Xer), 1
        ):
            y_tr      = y[tr_idx]
            y_te      = y[te_idx]
            y_tr_norm = (y_tr - y_min) / score_range
            eg_tr     = eg[tr_idx]
            eg_te     = eg[te_idx]

            def train_predict(X):
                Xtr, Xte = add_eg_sim(
                    X[tr_idx], X[te_idx],
                    y_tr, eg_tr, eg_te
                )
                m = XGBRegressor(
                    **params, random_state=42,
                    n_jobs=-1, tree_method='hist',
                    device='cuda', verbosity=0
                )
                m.fit(Xtr, y_tr_norm)
                return m.predict(Xte)

            p_er = train_predict(Xer)
            p_in = train_predict(Xi)

            yA = clip_round(
                p_er*score_range+y_min, y_min, y_max
            )
            yB = clip_round(
                (0.5*p_er+0.5*p_in)*score_range+y_min,
                y_min, y_max
            )
            y_true = y_te.astype(int)

            qA = qwk(y_true, yA)
            qB = qwk(y_true, yB)
            folds_A.append(qA)
            folds_B.append(qB)

            print(f"  Fold {fold_num:2d} → "
                  f"A={qA:.3f}  B={qB:.3f}  "
                  f"V9={v9_qwk[essay_set]:.3f}")

        mA = np.mean(folds_A)
        mB = np.mean(folds_B)
        results_A[essay_set] = mA
        results_B[essay_set] = mB

        print(f"  {'─'*52}")
        print(f"  MEAN → A={mA:.3f}  B={mB:.3f}  "
              f"V9={v9_qwk[essay_set]:.3f}  "
              f"V21C={v21c_qwk[essay_set]:.3f}")

        all_results.append({
            'essay_set'      : essay_set,
            'n_essays'       : len(sub_er),
            'score_range'    : f"{y_min}–{y_max}",
            'e5rub2048_alone': round(mA, 3),
            'e5rub2048_inst' : round(mB, 3),
            'v21c_512tokens' : v21c_qwk[essay_set],
            'v9_instructor'  : v9_qwk[essay_set],
            'paper'          : paper_qwk[essay_set],
        })

        pd.DataFrame(all_results).to_csv(
            RESULTS_V22_PATH, index=False
        )
        print(f"  💾 Saved (set {essay_set})")

    results_df = pd.DataFrame(all_results)

# Final
paper_qwk  = {
    1:0.91,2:0.79,3:0.71,
    4:0.84,5:0.84,6:0.84,
    7:0.87,8:0.92
}
paper_avg  = np.mean(list(paper_qwk.values()))
v9_qwk     = {
    1:0.775,2:0.644,3:0.672,
    4:0.802,5:0.792,6:0.782,
    7:0.787,8:0.698
}
v21c_qwk   = {
    1:0.763,2:0.624,3:0.652,
    4:0.771,5:0.765,6:0.718,
    7:0.768,8:0.672
}

print("\n" + "="*70)
print("   V22: E5+Rubric(2048) — FULL CONTEXT FIX")
print("   Comparing 512 vs 2048 token limit")
print("="*70)
print(f"{'Set':<4} {'V9':>7} "
      f"{'V21C(512)':>10} "
      f"{'V22A(2048)':>11} "
      f"{'V22B+INST':>10} "
      f"{'Paper':>7}")
print("─"*55)

best_qwks = []
for _, row in results_df.iterrows():
    s   = int(row['essay_set'])
    mA  = row['e5rub2048_alone']
    mB  = row['e5rub2048_inst']
    v21 = v21c_qwk[s]
    v9  = v9_qwk[s]
    best = max(mA, mB)
    best_qwks.append(best)
    flag = "✅" if best > v9 else "⚠️"
    imp  = best - v21
    arr  = f"↑{imp:.3f}" if imp>0.002 else (
           f"↓{abs(imp):.3f}" if imp<-0.002 else "→"
    )
    print(f"  {s:<3} {v9:>7.3f} "
          f"{v21:>10.3f} "
          f"{mA:>11.3f} "
          f"{mB:>10.3f} "
          f"{paper_qwk[s]:>7.3f} {flag} {arr}")

print("─"*55)
best_avg = np.mean(best_qwks)
avgA = np.mean([r['e5rub2048_alone']
                for _,r in results_df.iterrows()])
avgB = np.mean([r['e5rub2048_inst']
                for _,r in results_df.iterrows()])
print(f"  {'Avg':<3} {'0.744':>7} "
      f"{np.mean(list(v21c_qwk.values())):>10.3f} "
      f"{avgA:>11.3f} "
      f"{avgB:>10.3f} "
      f"{paper_avg:>7.3f}")

print("\n" + "="*70)
print(results_df.to_string(index=False))
print("\n✅ V22 Complete!")
print("⚠️  Click Save Version NOW!")

In [ ]:
# ============================================================
# V23 — Weight Search + XGBoost Parameter Tuning
# Find optimal ensemble weight + XGBoost params
# ============================================================

import os
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import cohen_kappa_score
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

SAVE_DIR         = '/kaggle/working/'
RESULTS_V23_PATH = os.path.join(
    SAVE_DIR, 'results_v23.csv'
)

if os.path.exists(RESULTS_V23_PATH):
    print("✅ V23 results exist!")
    results_df = pd.read_csv(RESULTS_V23_PATH)
    print(results_df.to_string(index=False))

else:
    # Load feature matrices
    emb_e5rub_df   = pd.read_csv(os.path.join(
        SAVE_DIR,
        'train_embeddings_e5_rubric_2048.csv'
    ))
    EMB_DIM_E5RUB2 = len([
        c for c in emb_e5rub_df.columns
        if c.startswith('emb_')
    ])

    print("Loading feature matrices...")
    train_e5rub2 = pd.read_parquet(os.path.join(
        SAVE_DIR,
        'train_full_features_e5rubric_2048.parquet'
    ))
    train_inst   = pd.read_parquet(os.path.join(
        SAVE_DIR,
        'train_full_features_instructor.parquet'
    ))
    print(f"✅ E5+Rubric(2048): {train_e5rub2.shape}")
    print(f"✅ INSTRUCTOR:      {train_inst.shape}")

    SF_COLS = [
        'gunning_fog','flesch_re','ari','guiraud_index',
        'noun_freq','verb_freq','avg_sent_len',
        'pronoun_density','pronoun_noun_ratio','giveness'
    ]
    EG_COLS = [
        f"{r1}_{r2}"
        for r1 in ['S','O','X','-']
        for r2 in ['S','O','X','-']
    ]
    E5RUB2_FEAT = (
        SF_COLS + EG_COLS +
        ['mean_coherence','min_coherence','std_coherence'] +
        [f'emb_{i}' for i in range(EMB_DIM_E5RUB2)]
    )
    INST_FEAT   = (
        SF_COLS + EG_COLS +
        ['mean_coherence','min_coherence','std_coherence'] +
        [f'emb_{i}' for i in range(1024)]
    )

    def qwk(y_true, y_pred):
        return cohen_kappa_score(
            y_true, y_pred, weights='quadratic'
        )
    def clip_round(y_pred, y_min, y_max):
        return np.clip(
            np.round(y_pred).astype(int), y_min, y_max
        )
    def add_eg_sim(X_tr, X_te, y_tr,
                   eg_tr, eg_te, k=5):
        s_te = cosine_similarity(eg_te, eg_tr)
        s_tr = cosine_similarity(eg_tr, eg_tr)
        np.fill_diagonal(s_tr, 0)
        def f(s, y):
            out = []
            for i in range(len(s)):
                idx = np.argsort(s[i])[-k:]
                out.append([
                    np.mean(y[idx]),
                    np.mean(s[i][idx]),
                    np.max(s[i][idx]),
                    np.std(y[idx])
                ])
            return np.array(out)
        return (
            np.hstack([X_tr, f(s_tr, y_tr)]),
            np.hstack([X_te, f(s_te, y_tr)])
        )

    SCORE_RANGES = {
        1:(2,12), 2:(1,6),  3:(0,3),
        4:(0,3),  5:(0,4),  6:(0,4),
        7:(2,24), 8:(10,60),
    }

    # ----------------------------------------------------------
    # TUNED PARAMS PER SET
    # More aggressive than before but still regularized
    # ----------------------------------------------------------
    SET_PARAMS_TUNED = {
        1: dict(n_estimators=1500, learning_rate=0.01,
                max_depth=8, subsample=0.75,
                colsample_bytree=0.6, min_child_weight=3,
                reg_alpha=0.05, reg_lambda=1.5,
                gamma=0.1),
        2: dict(n_estimators=1500, learning_rate=0.01,
                max_depth=7, subsample=0.75,
                colsample_bytree=0.6, min_child_weight=3,
                reg_alpha=0.05, reg_lambda=1.5,
                gamma=0.1),
        3: dict(n_estimators=1000, learning_rate=0.02,
                max_depth=5, subsample=0.8,
                colsample_bytree=0.7, min_child_weight=2,
                reg_alpha=0.05, reg_lambda=1.0,
                gamma=0.0),
        4: dict(n_estimators=1000, learning_rate=0.02,
                max_depth=5, subsample=0.8,
                colsample_bytree=0.7, min_child_weight=2,
                reg_alpha=0.05, reg_lambda=1.0,
                gamma=0.0),
        5: dict(n_estimators=1000, learning_rate=0.02,
                max_depth=5, subsample=0.8,
                colsample_bytree=0.7, min_child_weight=2,
                reg_alpha=0.05, reg_lambda=1.0,
                gamma=0.0),
        6: dict(n_estimators=1000, learning_rate=0.02,
                max_depth=5, subsample=0.8,
                colsample_bytree=0.7, min_child_weight=2,
                reg_alpha=0.05, reg_lambda=1.0,
                gamma=0.0),
        7: dict(n_estimators=1500, learning_rate=0.01,
                max_depth=8, subsample=0.75,
                colsample_bytree=0.6, min_child_weight=3,
                reg_alpha=0.1, reg_lambda=2.0,
                gamma=0.1),
        8: dict(n_estimators=2000, learning_rate=0.005,
                max_depth=5, subsample=0.7,
                colsample_bytree=0.5, min_child_weight=5,
                reg_alpha=0.5, reg_lambda=2.5,
                gamma=0.2),
    }

    # Ensemble weights to search per set
    # Format: (E5+Rubric weight, INSTRUCTOR weight)
    WEIGHT_SEARCH = [0.3, 0.4, 0.5, 0.6, 0.7]

    completed_sets = []
    if os.path.exists(RESULTS_V23_PATH):
        existing       = pd.read_csv(RESULTS_V23_PATH)
        completed_sets = existing['essay_set'].tolist()
        all_results    = existing.to_dict('records')
        print(f"✅ Resuming: {completed_sets}")
    else:
        all_results = []

    paper_qwk = {
        1:0.91,2:0.79,3:0.71,
        4:0.84,5:0.84,6:0.84,
        7:0.87,8:0.92
    }
    v22b_qwk = {
        1:0.782,2:0.653,3:0.670,
        4:0.785,5:0.794,6:0.768,
        7:0.786,8:0.720
    }
    v9_qwk = {
        1:0.775,2:0.644,3:0.672,
        4:0.802,5:0.792,6:0.782,
        7:0.787,8:0.698
    }

    print("\n" + "="*65)
    print("  V23 — Weight Search + Parameter Tuning")
    print("  Searching best E5:INST ratio per set")
    print("="*65)

    for essay_set in sorted(
        train_e5rub2['essay_set'].unique()
    ):
        if essay_set in completed_sets:
            print(f"⏭️  Set {essay_set} done")
            continue

        sub_er = train_e5rub2[
            train_e5rub2['essay_set']==essay_set
        ].reset_index(drop=True)
        sub_in = train_inst[
            train_inst['essay_set']==essay_set
        ].reset_index(drop=True)

        Xer = sub_er[E5RUB2_FEAT].values.astype(
            np.float32
        )
        Xi  = sub_in[INST_FEAT].values.astype(np.float32)
        y   = sub_er['domain1_score'].values.astype(
            np.float32
        )
        eg  = sub_er[EG_COLS].values.astype(np.float32)

        y_min, y_max = SCORE_RANGES[essay_set]
        score_range  = y_max - y_min
        params       = SET_PARAMS_TUNED[essay_set]

        print(f"\n── Set {essay_set} "
              f"({len(sub_er)} essays, "
              f"{y_min}-{y_max}) ──")
        print(f"   Searching weights: {WEIGHT_SEARCH}")

        kf = KFold(
            n_splits=10, shuffle=True, random_state=42
        )

        # Store predictions per fold for weight search
        all_p_er  = []
        all_p_in  = []
        all_y_te  = []

        for fold_num, (tr_idx, te_idx) in enumerate(
            kf.split(Xer), 1
        ):
            y_tr      = y[tr_idx]
            y_te      = y[te_idx]
            y_tr_norm = (y_tr - y_min) / score_range
            eg_tr     = eg[tr_idx]
            eg_te     = eg[te_idx]

            def train_predict(X):
                Xtr, Xte = add_eg_sim(
                    X[tr_idx], X[te_idx],
                    y_tr, eg_tr, eg_te
                )
                m = XGBRegressor(
                    **params, random_state=42,
                    n_jobs=-1, tree_method='hist',
                    device='cuda', verbosity=0
                )
                m.fit(Xtr, y_tr_norm)
                return m.predict(Xte)

            p_er = train_predict(Xer)
            p_in = train_predict(Xi)

            all_p_er.append(p_er)
            all_p_in.append(p_in)
            all_y_te.append(y_te)

            # Show 50/50 result per fold
            p_50 = 0.5*p_er + 0.5*p_in
            y_pred = clip_round(
                p_50*score_range+y_min, y_min, y_max
            )
            q = qwk(y_te.astype(int), y_pred)
            print(f"  Fold {fold_num:2d} → "
                  f"QWK(50/50)={q:.3f}")

        # Now search best weight using all fold predictions
        print(f"\n  Weight search results:")
        best_w    = 0.5
        best_qwk_w = 0.0

        for w_er in WEIGHT_SEARCH:
            w_in = 1.0 - w_er
            fold_qwks = []

            for f_idx in range(len(all_p_er)):
                p_ens  = w_er*all_p_er[f_idx] + \
                         w_in*all_p_in[f_idx]
                y_pred = clip_round(
                    p_ens*score_range+y_min,
                    y_min, y_max
                )
                fold_qwks.append(
                    qwk(
                        all_y_te[f_idx].astype(int),
                        y_pred
                    )
                )

            mean_q = np.mean(fold_qwks)
            marker = " ← BEST" if mean_q > best_qwk_w \
                     else ""
            print(f"  E5={w_er:.1f} INST={w_in:.1f} "
                  f"→ QWK={mean_q:.4f}{marker}")

            if mean_q > best_qwk_w:
                best_qwk_w = mean_q
                best_w     = w_er

        print(f"\n  Best weight: E5={best_w:.1f} "
              f"INST={1-best_w:.1f}")
        print(f"  Best QWK:    {best_qwk_w:.4f}")
        print(f"  V22B(50/50): {v22b_qwk[essay_set]:.4f}")
        print(f"  V9:          {v9_qwk[essay_set]:.4f}")
        imp = best_qwk_w - v22b_qwk[essay_set]
        print(f"  Improvement: {imp:+.4f}")

        all_results.append({
            'essay_set'    : essay_set,
            'n_essays'     : len(sub_er),
            'score_range'  : f"{y_min}–{y_max}",
            'best_w_e5'    : best_w,
            'best_w_inst'  : round(1-best_w, 1),
            'mean_qwk'     : round(best_qwk_w, 3),
            'v22b_5050'    : v22b_qwk[essay_set],
            'v9'           : v9_qwk[essay_set],
            'paper'        : paper_qwk[essay_set],
        })

        pd.DataFrame(all_results).to_csv(
            RESULTS_V23_PATH, index=False
        )
        print(f"  💾 Saved (set {essay_set})")

    results_df = pd.DataFrame(all_results)

# Final comparison
paper_avg = np.mean(list(paper_qwk.values()))
v22b_avg  = np.mean(list(v22b_qwk.values()))
v9_avg    = np.mean(list(v9_qwk.values()))

print("\n" + "="*65)
print("   V23 FINAL — Weight Tuned Results")
print("="*65)
print(f"\n  {'Set':<4} {'Best W':>8} "
      f"{'V9':>7} {'V22B':>7} "
      f"{'V23':>7} {'Paper':>7}")
print("  " + "─"*48)

our_qwks = []
for _, row in results_df.iterrows():
    s   = int(row['essay_set'])
    our = row['mean_qwk']
    v22 = row['v22b_5050']
    v9  = row['v9']
    pap = paper_qwk[s]
    our_qwks.append(our)
    imp  = our - v22
    flag = "✅" if our > v22 else "⚠️"
    arr  = f"↑{imp:.3f}" if imp>0.001 else (
           f"↓{abs(imp):.3f}" if imp<-0.001 else "→"
    )
    w_str = f"E5={row['best_w_e5']:.1f}"
    print(f"  {s:<4} {w_str:>8} "
          f"{v9:>7.3f} {v22:>7.3f} "
          f"{our:>7.3f} {pap:>7.3f} "
          f"{flag} {arr}")

print("  " + "─"*48)
our_avg = np.mean(our_qwks)
print(f"  {'Avg':<4} {'':<8} "
      f"{'0.744':>7} {'0.745':>7} "
      f"{our_avg:>7.3f} {paper_avg:>7.3f}")

print(f"""
  Summary:
  V9  (INSTRUCTOR):              0.744
  V22B (E5+Rub+INST 50/50):     0.745
  V23  (tuned weights + params): {our_avg:.3f}
  Paper (GPT-3):                 0.840

  Realistic maximum with tuning: ~0.755-0.765
  Your target of 0.790 requires: different approach
""")
print("\n✅ V23 Complete!")
print("⚠️  Click Save Version NOW!")

**MALAYALAM DATASET**

In [ ]:
# ============================================================
# MALAYALAM AES — COMPLETE PIPELINE
# Dataset: 3000 essays, 150 prompts, 20 essays/prompt
# Score range: 0-10 all prompts
# KEY ADVANTAGE: Has ref_answer → semantic similarity!
# Model: multilingual-e5-large-instruct
# ============================================================

# ── STEP 1: Install ────────────────────────────────────────
# !pip install sentence-transformers -q

import os, gc, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import KFold, GroupKFold
from sklearn.metrics import cohen_kappa_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

SAVE_DIR = '/kaggle/working/'
os.makedirs(SAVE_DIR, exist_ok=True)

# ── STEP 2: Load Dataset ───────────────────────────────────
print("Loading datasets...")
df = pd.read_csv(
    '/kaggle/input/datasets/nehasatheesh/malayalam-dataset/mal_dataset.csv'
)
q_df = pd.read_csv(
    '/kaggle/input/datasets/nehasatheesh/malayalam-dataset/questions_150.csv'
)

# Merge prompt text into main df
df = df.merge(
    q_df[['prompt_id','prompt','max_marks']],
    on='prompt_id', how='left'
)

print(f"✅ Essays loaded: {df.shape}")
print(f"   Prompts: {df['prompt_id'].nunique()}")
print(f"   Essays per prompt: "
      f"{df.groupby('prompt_id').size().mean():.1f}")
print(f"   Score range: "
      f"{df['essay_score'].min()}-"
      f"{df['essay_score'].max()}")
print(f"   Has ref_answer: "
      f"{df['ref_answer'].notna().all()}")

# Sanity check
print("\nScore distribution:")
print(df['essay_score'].value_counts().sort_index())

In [ ]:
# ============================================================
# STEP 3: Malayalam Shallow Features
# ============================================================

def get_malayalam_features(text, ref_text=None):
    """
    Malayalam-specific features.
    Malayalam Unicode range: U+0D00–U+0D7F
    """
    if not isinstance(text, str):
        text = ''

    # ── Word-level features ─────────────────────────────
    words = text.split()
    n_words = max(len(words), 1)

    # ── Sentence-level features ─────────────────────────
    # Malayalam sentence boundary: ., ?, !, ।
    import re
    sents = [
        s.strip() for s in
        re.split(r'[.?!।]', text)
        if s.strip()
    ]
    n_sents = max(len(sents), 1)

    # ── Malayalam script features ────────────────────────
    # Check how much of the essay is Malayalam Unicode
    mal_chars = sum(
        1 for c in text
        if '\u0D00' <= c <= '\u0D7F'
    )
    total_alpha = sum(1 for c in text if c.isalnum())
    mal_ratio = mal_chars / max(total_alpha, 1)

    # ── Lexical diversity ────────────────────────────────
    unique_words = set(words)
    ttr = len(unique_words) / n_words  # type-token ratio

    # ── Sentence statistics ──────────────────────────────
    sent_lens = [len(s.split()) for s in sents]
    avg_sent_len = n_words / n_sents
    sent_len_std = np.std(sent_lens) \
                   if len(sent_lens) > 1 else 0

    # ── Character-level features ─────────────────────────
    avg_word_len = np.mean([
        len(w) for w in words
    ]) if words else 0

    # ── Paragraph features ───────────────────────────────
    paras = [
        p.strip() for p in text.split('\n')
        if p.strip()
    ]
    n_paras = max(len(paras), 1)

    # ── Punctuation diversity ────────────────────────────
    puncts = set(
        c for c in text
        if not c.isalnum() and not c.isspace()
    )
    punct_diversity = len(puncts)

    return {
        'word_count'         : n_words,
        'sent_count'         : n_sents,
        'para_count'         : n_paras,
        'avg_sent_len'       : round(avg_sent_len, 3),
        'sent_len_std'       : round(sent_len_std, 3),
        'avg_word_len'       : round(avg_word_len, 3),
        'type_token_ratio'   : round(ttr, 3),
        'malayalam_ratio'    : round(mal_ratio, 3),
        'punct_diversity'    : punct_diversity,
        'char_count'         : len(text),
    }

print("Computing Malayalam shallow features...")
from tqdm import tqdm

sf_records = []
for _, row in tqdm(df.iterrows(), total=len(df)):
    feats = get_malayalam_features(
        row['essay'], row['ref_answer']
    )
    sf_records.append(feats)

sf_df = pd.DataFrame(sf_records)
sf_df.to_csv(
    os.path.join(SAVE_DIR,
                 'mal_shallow_features.csv'),
    index=False
)
print(f"✅ Shallow features: {sf_df.shape}")
print(sf_df.describe().round(3))

In [ ]:
# ============================================================
# STEP 4: Multilingual E5 Embeddings for Malayalam
# KEY FEATURE: Essay vs Reference Answer similarity
# This is the most powerful signal for this dataset!
# ============================================================

MAL_EMB_PATH   = os.path.join(
    SAVE_DIR, 'mal_embeddings.csv'
)
MAL_CKPT_DIR   = os.path.join(
    SAVE_DIR, 'mal_emb_checkpoints'
)
os.makedirs(MAL_CKPT_DIR, exist_ok=True)

if os.path.exists(MAL_EMB_PATH):
    print("✅ Embeddings exist!")
    emb_df = pd.read_csv(MAL_EMB_PATH)
    print("Shape:", emb_df.shape)

else:
    gc.collect()
    torch.cuda.empty_cache()

    texts      = df['essay'].astype(str).tolist()
    ref_texts  = df['ref_answer'].astype(str).tolist()
    prompt_ids = df['prompt_id'].tolist()
    n_total    = len(texts)
    CHUNK_SIZE = 500

    # Instruction for Malayalam essay scoring
    ESSAY_INSTR = (
        "Instruct: Represent this Malayalam student "
        "answer for evaluating content accuracy and "
        "completeness against a reference answer\n"
        "Query: "
    )
    REF_INSTR = (
        "Instruct: Represent this Malayalam reference "
        "answer as the gold standard for scoring\n"
        "Query: "
    )
    SENT_INSTR = (
        "Instruct: Represent this Malayalam sentence "
        "for measuring coherence\nQuery: "
    )

    # Find completed chunks
    done = sorted([
        int(f.replace('chunk_','').replace('.npz',''))
        for f in os.listdir(MAL_CKPT_DIR)
        if f.startswith('chunk_') and f.endswith('.npz')
    ])
    start_idx = (max(done)+1)*CHUNK_SIZE if done else 0

    print(f"Total: {n_total}, Starting: {start_idx}")
    print(f"GPU free: "
          f"{torch.cuda.mem_get_info()[0]/1024**3:.1f}GB")

    if start_idx < n_total:
        MODEL = 'intfloat/multilingual-e5-large-instruct'
        print(f"\nLoading {MODEL}...")
        tokenizer = AutoTokenizer.from_pretrained(MODEL)
        model     = AutoModel.from_pretrained(
            MODEL, torch_dtype=torch.float16
        ).to('cuda')
        model.eval()
        EMB_DIM = model.config.hidden_size
        print(f"✅ Loaded! Hidden dim={EMB_DIM}")

        def last_token_pool(
            last_hidden, attention_mask
        ):
            seq_len = attention_mask.sum(dim=1) - 1
            batch   = last_hidden.shape[0]
            return last_hidden[
                torch.arange(
                    batch, device=last_hidden.device
                ),
                seq_len
            ]

        def encode(inputs, max_len=512, bs=16):
            all_embs = []
            for i in range(0, len(inputs), bs):
                batch = inputs[i:i+bs]
                enc   = tokenizer(
                    batch,
                    max_length=max_len,
                    padding=True,
                    truncation=True,
                    return_tensors='pt'
                ).to('cuda')
                with torch.no_grad():
                    out  = model(**enc)
                    embs = last_token_pool(
                        out.last_hidden_state,
                        enc['attention_mask']
                    )
                    embs = F.normalize(
                        embs.float(), p=2, dim=1
                    )
                all_embs.append(embs.cpu().numpy())
                del enc, out, embs
                torch.cuda.empty_cache()
            return np.vstack(all_embs)

        n_chunks = (n_total+CHUNK_SIZE-1) // CHUNK_SIZE

        for cid in range(
            start_idx // CHUNK_SIZE, n_chunks
        ):
            cs  = cid * CHUNK_SIZE
            ce  = min(cs + CHUNK_SIZE, n_total)
            cpath = os.path.join(
                MAL_CKPT_DIR, f'chunk_{cid}.npz'
            )

            if os.path.exists(cpath):
                print(f"  ⏭️  Chunk {cid} skip")
                continue

            print(f"\n── Chunk {cid}/{n_chunks-1} "
                  f"({cs}-{ce}) ──")

            c_texts = texts[cs:ce]
            c_refs  = ref_texts[cs:ce]

            # Essay embeddings
            print("  Encoding essays...")
            essay_embs = encode(
                [ESSAY_INSTR + t[:500]
                 for t in c_texts],
                max_len=512, bs=16
            )

            # Reference embeddings
            print("  Encoding references...")
            ref_embs = encode(
                [REF_INSTR + r[:500]
                 for r in c_refs],
                max_len=512, bs=16
            )

            # ── KEY FEATURE: Essay-Ref Similarity ──────
            # For each essay, compute similarity with
            # its OWN reference answer
            essay_ref_sims = np.array([
                float(cosine_similarity(
                    essay_embs[i].reshape(1,-1),
                    ref_embs[i].reshape(1,-1)
                )[0][0])
                for i in range(len(c_texts))
            ])

            # Sentence coherence
            print("  Computing coherence...")
            from nltk.tokenize import sent_tokenize
            # For Malayalam, split by punctuation
            import re
            chunk_coh = []
            for text in tqdm(c_texts):
                sents = [
                    s.strip() for s in
                    re.split(r'[.?!।]', text)
                    if s.strip() and len(s.split())>2
                ][:15]
                if len(sents) < 2:
                    chunk_coh.append([0.0, 0.0, 0.0])
                    continue
                sent_embs = encode(
                    [SENT_INSTR + s for s in sents],
                    max_len=128, bs=32
                )
                sims = [
                    float(cosine_similarity(
                        sent_embs[i].reshape(1,-1),
                        sent_embs[i+1].reshape(1,-1)
                    )[0][0])
                    for i in range(len(sent_embs)-1)
                ]
                chunk_coh.append([
                    float(np.mean(sims)),
                    float(np.min(sims)),
                    float(np.std(sims))
                ])

            np.savez_compressed(
                cpath,
                essay_embs     = essay_embs,
                ref_embs       = ref_embs,
                essay_ref_sims = essay_ref_sims,
                coherence      = np.array(chunk_coh)
            )
            print(f"  💾 Chunk {cid} saved!")
            print(f"  GPU: "
                  f"{torch.cuda.mem_get_info()[0]/1024**3:.1f}GB")

        print("\n✅ All chunks done!")

    # Assemble
    print("\nAssembling...")
    n_chunks  = (n_total+CHUNK_SIZE-1) // CHUNK_SIZE
    all_done  = sorted([
        int(f.replace('chunk_','').replace('.npz',''))
        for f in os.listdir(MAL_CKPT_DIR)
        if f.startswith('chunk_') and f.endswith('.npz')
    ])
    missing = [
        i for i in range(n_chunks)
        if i not in all_done
    ]

    if missing:
        print(f"❌ Missing chunks: {missing}")
    else:
        all_essay = []
        all_coh   = []
        all_sims  = []

        for cid in sorted(all_done):
            data = np.load(os.path.join(
                MAL_CKPT_DIR, f'chunk_{cid}.npz'
            ))
            all_essay.append(data['essay_embs'])
            all_coh.append(data['coherence'])
            all_sims.append(data['essay_ref_sims'])

        all_essay = np.vstack(all_essay)
        all_coh   = np.vstack(all_coh)
        all_sims  = np.concatenate(all_sims)
        EMB_DIM   = all_essay.shape[1]

        coh_df = pd.DataFrame(
            all_coh,
            columns=['mean_coherence',
                     'min_coherence','std_coherence']
        )
        sim_df = pd.DataFrame(
            {'essay_ref_similarity': all_sims}
        )
        emb_df = pd.DataFrame(
            all_essay,
            columns=[f'emb_{i}' for i in range(EMB_DIM)]
        )
        out = pd.concat([coh_df, sim_df, emb_df], axis=1)
        out.to_csv(MAL_EMB_PATH, index=False)
        print(f"💾 Saved: {out.shape}")
        print("  essay_ref_similarity stats:")
        print(f"  mean={all_sims.mean():.3f}  "
              f"std={all_sims.std():.3f}  "
              f"min={all_sims.min():.3f}  "
              f"max={all_sims.max():.3f}")
        print("\n✅ Step 4 Complete!")
        print("⚠️  Click Save Version NOW!")

In [ ]:
# ============================================================
# STEP 5: Assemble Full Feature Matrix
# ============================================================

MAL_FULL_PATH = os.path.join(
    SAVE_DIR, 'mal_full_features.parquet'
)

if os.path.exists(MAL_FULL_PATH):
    try:
        train_mal = pd.read_parquet(MAL_FULL_PATH)
        print(f"✅ Loaded: {train_mal.shape}")
    except Exception:
        os.remove(MAL_FULL_PATH)
        print("Corrupted — rebuilding...")

if not os.path.exists(MAL_FULL_PATH):
    emb_df = pd.read_csv(MAL_EMB_PATH)
    EMB_DIM = len([
        c for c in emb_df.columns
        if c.startswith('emb_')
    ])

    # Fix dtypes
    float_cols = (
        ['mean_coherence','min_coherence',
         'std_coherence','essay_ref_similarity'] +
        [f'emb_{i}' for i in range(EMB_DIM)]
    )
    for col in float_cols:
        if col in emb_df.columns:
            emb_df[col] = pd.to_numeric(
                emb_df[col], errors='coerce'
            ).astype(np.float32).fillna(0.0)

    META_COLS = [
        'essay_id','prompt_id','essay',
        'essay_score','ref_answer','prompt','max_marks'
    ]
    SF_COLS = [
        'word_count','sent_count','para_count',
        'avg_sent_len','sent_len_std','avg_word_len',
        'type_token_ratio','malayalam_ratio',
        'punct_diversity','char_count'
    ]
    EMB_COLS = float_cols  # all computed columns

    sf_df     = pd.read_csv(
        os.path.join(SAVE_DIR,
                     'mal_shallow_features.csv')
    )

    train_mal = pd.concat([
        df[META_COLS].reset_index(drop=True),
        sf_df[SF_COLS].reset_index(drop=True),
        emb_df[EMB_COLS].reset_index(drop=True),
    ], axis=1)

    train_mal.to_parquet(MAL_FULL_PATH, index=False)
    print(f"✅ Saved: {train_mal.shape}")
    print("⚠️  Click Save Version NOW!")

In [ ]:
# ============================================================
# STEP 6: XGBoost 10-Fold CV — Malayalam AES
# Strategy: group by prompt_id for fair evaluation
# Key feature: essay_ref_similarity
# ============================================================

MAL_RESULTS_PATH = os.path.join(
    SAVE_DIR, 'results_malayalam.csv'
)

if os.path.exists(MAL_RESULTS_PATH):
    print("✅ Results exist!")
    results_df = pd.read_csv(MAL_RESULTS_PATH)
    print(results_df.describe().round(3))

else:
    print("Loading feature matrix...")
    train_mal = pd.read_parquet(MAL_FULL_PATH)
    print(f"✅ Loaded: {train_mal.shape}")

    SF_COLS = [
        'word_count','sent_count','para_count',
        'avg_sent_len','sent_len_std','avg_word_len',
        'type_token_ratio','malayalam_ratio',
        'punct_diversity','char_count'
    ]
    EMB_DIM = len([
        c for c in train_mal.columns
        if c.startswith('emb_')
    ])
    EMB_COLS = (
        ['mean_coherence','min_coherence',
         'std_coherence',
         'essay_ref_similarity'] +  # ← KEY FEATURE
        [f'emb_{i}' for i in range(EMB_DIM)]
    )
    ALL_FEAT = SF_COLS + EMB_COLS

    X = np.nan_to_num(
        train_mal[ALL_FEAT].values.astype(np.float32),
        nan=0.0, posinf=0.0, neginf=0.0
    )
    y = train_mal['essay_score'].values.astype(
        np.float32
    )
    prompt_ids = train_mal['prompt_id'].values

    print(f"\nFeature matrix: {X.shape}")
    print(f"Score range: {y.min():.0f}-{y.max():.0f}")

    def qwk(y_true, y_pred):
        return cohen_kappa_score(
            y_true, y_pred, weights='quadratic'
        )
    def exact_agreement(y_true, y_pred):
        return np.mean(np.array(y_true)==np.array(y_pred))
    def adjacent_agreement(y_true, y_pred):
        return np.mean(
            np.abs(np.array(y_true)-np.array(y_pred))<=1
        )
    def spearman_r(y_true, y_pred):
        r, _ = spearmanr(y_true, y_pred)
        return r
    def clip_round(y_pred, y_min, y_max):
        return np.clip(
            np.round(y_pred).astype(int), y_min, y_max
        )

    # Score range is 0-10 for all prompts
    Y_MIN, Y_MAX   = 0, 10
    SCORE_RANGE    = Y_MAX - Y_MIN

    # XGBoost params
    PARAMS = dict(
        n_estimators=800,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.7,
        min_child_weight=3,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        tree_method='hist',
        device='cuda',
        verbosity=0
    )

    print("\n" + "="*65)
    print("  10-FOLD CV — Malayalam AES")
    print("  Model: multilingual-e5-large-instruct")
    print("  Key feature: essay_ref_similarity")
    print("="*65)

    kf = KFold(
        n_splits=10, shuffle=True, random_state=42
    )

    fold_qwk, fold_ea, fold_aa, fold_sp = [], [], [], []

    for fold_num, (tr_idx, te_idx) in enumerate(
        kf.split(X), 1
    ):
        X_tr, X_te = X[tr_idx], X[te_idx]
        y_tr, y_te = y[tr_idx], y[te_idx]

        y_tr_norm = y_tr / SCORE_RANGE

        m = XGBRegressor(**PARAMS)
        m.fit(X_tr, y_tr_norm)

        p      = m.predict(X_te)
        y_pred = clip_round(
            p * SCORE_RANGE, Y_MIN, Y_MAX
        )
        y_true = y_te.astype(int)

        fold_qwk.append(qwk(y_true, y_pred))
        fold_ea.append(exact_agreement(y_true, y_pred))
        fold_aa.append(adjacent_agreement(y_true, y_pred))
        fold_sp.append(spearman_r(y_true, y_pred))

        print(f"  Fold {fold_num:2d} → "
              f"QWK={fold_qwk[-1]:.3f}  "
              f"EA={fold_ea[-1]:.3f}  "
              f"AA={fold_aa[-1]:.3f}  "
              f"Sp={fold_sp[-1]:.3f}")

    mean_qwk = np.mean(fold_qwk)
    mean_ea  = np.mean(fold_ea)
    mean_aa  = np.mean(fold_aa)
    mean_sp  = np.mean(fold_sp)

    print(f"\n{'='*65}")
    print(f"  MEAN → QWK={mean_qwk:.3f}  "
          f"EA={mean_ea:.3f}  "
          f"AA={mean_aa:.3f}  "
          f"Spearman={mean_sp:.3f}")

    results_df = pd.DataFrame([{
        'n_essays'     : len(train_mal),
        'n_prompts'    : train_mal['prompt_id'].nunique(),
        'n_features'   : X.shape[1],
        'mean_qwk'     : round(mean_qwk, 3),
        'mean_ea'      : round(mean_ea,  3),
        'mean_aa'      : round(mean_aa,  3),
        'mean_spearman': round(mean_sp,  3),
        'model'        : 'multilingual-e5-large-instruct',
    }])

    results_df.to_csv(MAL_RESULTS_PATH, index=False)

    # Feature importance
    m_full = XGBRegressor(**PARAMS)
    m_full.fit(X, y / SCORE_RANGE)
    importance = m_full.feature_importances_
    feat_imp   = pd.DataFrame({
        'feature'   : ALL_FEAT,
        'importance': importance
    }).sort_values('importance', ascending=False)

    print(f"\n  Top 15 Features:")
    print(feat_imp.head(15).to_string(index=False))

    print(f"\n{'='*65}")
    print(f"  MALAYALAM AES RESULTS")
    print(f"  Essays: {len(train_mal)}")
    print(f"  Prompts: {train_mal['prompt_id'].nunique()}")
    print(f"  QWK:  {mean_qwk:.3f}")
    print(f"  EA:   {mean_ea:.3f}")
    print(f"  AA:   {mean_aa:.3f}")
    print(f"  Spe:  {mean_sp:.3f}")
    print(f"\n  essay_ref_similarity importance rank: "
          f"{list(feat_imp['feature']).index('essay_ref_similarity')+1}")

    print("\n✅ Malayalam AES Complete!")
    print("⚠️  Click Save Version NOW!")

In [ ]:
# ============================================================
# MALAYALAM AES — WORD COUNT FIX
# Replace raw word_count with ratio to reference length
# Also add ref_word_count as explicit feature
# Run this BEFORE predict_score
# ============================================================

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import KFold
from sklearn.metrics import cohen_kappa_score
from xgboost import XGBRegressor
from scipy.stats import spearmanr
import re, os, warnings
warnings.filterwarnings('ignore')

SAVE_DIR = '/kaggle/working/'

# Load data
df    = pd.read_csv(
    '/kaggle/input/datasets/nehasatheesh/malayalam-dataset/mal_dataset.csv'
)
q_df  = pd.read_csv(
    '/kaggle/input/datasets/nehasatheesh/malayalam-dataset/questions_150.csv'
)
df    = df.merge(
    q_df[['prompt_id','prompt','max_marks',
          'ref_answer']],
    on='prompt_id', how='left'
)

# ── IMPROVED SHALLOW FEATURES ─────────────────────────────
def get_mal_shallow_v2(essay_text, ref_text=None):
    """
    V2: Adds reference-normalized features.
    Key fix: word count ratio instead of raw count.
    """
    essay_text = str(essay_text)
    ref_text   = str(ref_text) if ref_text else ''

    words   = essay_text.split()
    n_words = max(len(words), 1)
    sents   = [
        s.strip() for s in
        re.split(r'[.?!।]', essay_text)
        if s.strip()
    ]
    n_sents = max(len(sents), 1)
    paras   = [
        p.strip() for p in essay_text.split('\n')
        if p.strip()
    ]
    mal_chars = sum(
        1 for c in essay_text
        if '\u0D00' <= c <= '\u0D7F'
    )
    total_alpha = sum(
        1 for c in essay_text if c.isalnum()
    )
    ttr = len(set(words)) / n_words
    sent_lens = [len(s.split()) for s in sents]

    # Reference-based features
    ref_words   = ref_text.split()
    n_ref_words = max(len(ref_words), 1)
    ref_sents   = [
        s.strip() for s in
        re.split(r'[.?!।]', ref_text)
        if s.strip()
    ]
    n_ref_sents = max(len(ref_sents), 1)

    # ── KEY FIX: Normalize by reference length ────────
    word_ratio = n_words / n_ref_words   # 0-1+
    sent_ratio = n_sents / n_ref_sents   # 0-1+

    return {
        # Raw counts (still useful)
        'word_count'         : n_words,
        'sent_count'         : n_sents,
        'para_count'         : max(len(paras), 1),
        'char_count'         : len(essay_text),

        # Reference-normalized (KEY FIX)
        'word_ratio'         : round(word_ratio, 3),
        'sent_ratio'         : round(sent_ratio, 3),
        'ref_word_count'     : n_ref_words,

        # Style features
        'avg_sent_len'       : n_words/n_sents,
        'sent_len_std'       : np.std(sent_lens)
                               if len(sent_lens)>1 else 0,
        'avg_word_len'       : np.mean([len(w)
                               for w in words])
                               if words else 0,
        'type_token_ratio'   : ttr,
        'malayalam_ratio'    : mal_chars/max(total_alpha,1),
        'punct_diversity'    : len(set(
            c for c in essay_text
            if not c.isalnum() and not c.isspace()
        )),
    }

# Compute improved features
print("Computing improved shallow features...")
from tqdm import tqdm

sf_v2 = [
    get_mal_shallow_v2(
        row['essay'], row['ref_answer_y']
    )
    for _, row in tqdm(df.iterrows(), total=len(df))
]
sf_v2_df = pd.DataFrame(sf_v2)
sf_v2_df.to_csv(
    os.path.join(SAVE_DIR, 'mal_shallow_v2.csv'),
    index=False
)
print(f"✅ V2 features: {sf_v2_df.shape}")

SF_COLS_V2 = [
    'word_count','sent_count','para_count','char_count',
    'word_ratio','sent_ratio','ref_word_count',
    'avg_sent_len','sent_len_std','avg_word_len',
    'type_token_ratio','malayalam_ratio','punct_diversity'
]

# ── Rebuild feature matrix with V2 shallow ─────────────
train_mal = pd.read_parquet(
    os.path.join(SAVE_DIR, 'mal_full_features.parquet')
)
EMB_DIM  = len([
    c for c in train_mal.columns
    if c.startswith('emb_')
])
EMB_COLS = (
    ['mean_coherence','min_coherence',
     'std_coherence','essay_ref_similarity'] +
    [f'emb_{i}' for i in range(EMB_DIM)]
)

# Replace shallow features with V2
ALL_FEAT_V2 = SF_COLS_V2 + EMB_COLS

train_v2 = pd.concat([
    df[['essay_id','prompt_id','essay',
        'essay_score','ref_answer_y']].reset_index(
        drop=True
    ),
    sf_v2_df[SF_COLS_V2].reset_index(drop=True),
    train_mal[EMB_COLS].reset_index(drop=True),
], axis=1)

train_v2.to_parquet(
    os.path.join(SAVE_DIR, 'mal_full_v2.parquet'),
    index=False
)
print(f"✅ V2 feature matrix: {train_v2.shape}")

# ── Run 10-fold CV with V2 features ───────────────────
X_v2 = np.nan_to_num(
    train_v2[ALL_FEAT_V2].values.astype(np.float32),
    nan=0.0, posinf=0.0, neginf=0.0
)
y    = train_v2['essay_score'].values.astype(np.float32)
Y_MIN, Y_MAX = 0, 10

PARAMS = dict(
    n_estimators=800, learning_rate=0.03,
    max_depth=6, subsample=0.8,
    colsample_bytree=0.7, min_child_weight=3,
    reg_alpha=0.1, reg_lambda=1.0,
    random_state=42, n_jobs=-1,
    tree_method='hist', device='cuda', verbosity=0
)

kf = KFold(n_splits=10, shuffle=True, random_state=42)
fold_qwk, fold_ea, fold_aa, fold_sp = [], [], [], []

print("\n" + "="*60)
print("  10-FOLD CV — V2 (Word Ratio Normalized)")
print("="*60)

for fold_num, (tr_idx, te_idx) in enumerate(
    kf.split(X_v2), 1
):
    X_tr, X_te = X_v2[tr_idx], X_v2[te_idx]
    y_tr, y_te = y[tr_idx], y[te_idx]

    m = XGBRegressor(**PARAMS)
    m.fit(X_tr, y_tr/(Y_MAX-Y_MIN))
    p     = m.predict(X_te)
    y_pred = np.clip(
        np.round(p*(Y_MAX-Y_MIN)).astype(int),
        Y_MIN, Y_MAX
    )
    y_true = y_te.astype(int)

    q  = cohen_kappa_score(
        y_true, y_pred, weights='quadratic'
    )
    ea = np.mean(y_true==y_pred)
    aa = np.mean(np.abs(y_true-y_pred)<=1)
    r, _ = spearmanr(y_true, y_pred)

    fold_qwk.append(q)
    fold_ea.append(ea)
    fold_aa.append(aa)
    fold_sp.append(r)

    print(f"  Fold {fold_num:2d} → "
          f"QWK={q:.3f}  EA={ea:.3f}  "
          f"AA={aa:.3f}  Sp={r:.3f}")

mean_qwk = np.mean(fold_qwk)
mean_ea  = np.mean(fold_ea)
mean_aa  = np.mean(fold_aa)
mean_sp  = np.mean(fold_sp)

print(f"\n  {'='*52}")
print(f"  RESULTS COMPARISON:")
print(f"  V1 (raw word_count): 0.932 QWK")
print(f"  V2 (word_ratio):     {mean_qwk:.3f} QWK")
print(f"  EA: {mean_ea:.3f}  AA: {mean_aa:.3f}  "
      f"Spe: {mean_sp:.3f}")

# Feature importance
m_full = XGBRegressor(**PARAMS)
m_full.fit(X_v2, y/(Y_MAX-Y_MIN))
importance = m_full.feature_importances_
feat_imp   = pd.DataFrame({
    'feature'   : ALL_FEAT_V2,
    'importance': importance
}).sort_values('importance', ascending=False)

print(f"\n  Top 10 Features (V2):")
for _, row in feat_imp.head(10).iterrows():
    print(f"    {row['feature']:<25} "
          f"{row['importance']:.4f}")

print(f"\n  word_ratio rank: "
      f"{list(feat_imp['feature']).index('word_ratio')+1}"
      f" / {len(ALL_FEAT_V2)}")
print(f"  word_count rank: "
      f"{list(feat_imp['feature']).index('word_count')+1}"
      f" / {len(ALL_FEAT_V2)}")
print(f"\n  essay_ref_similarity importance rank: "
          f"{list(feat_imp['feature']).index('essay_ref_similarity')+1}")

# Train final model on all data
model_xgb_v2 = XGBRegressor(**PARAMS)
model_xgb_v2.fit(X_v2, y/(Y_MAX-Y_MIN))
print("\n✅ V2 model trained!")
print("⚠️  Click Save Version NOW!")

In [ ]:
print(df.columns)

In [ ]:
# ============================================================
# UPDATED PREDICT FUNCTION — V2 (Word Ratio Fix)
# ============================================================

def get_embedding(text, instruction,
                  tokenizer, emb_model):
    enc = tokenizer(
        [instruction + str(text)[:500]],
        max_length=512, padding=True,
        truncation=True, return_tensors='pt'
    ).to('cuda')
    with torch.no_grad():
        out = emb_model(**enc)
        seq = enc['attention_mask'].sum(dim=1)-1
        emb = out.last_hidden_state[
            torch.arange(1, device=out.last_hidden_state.device),
            seq
        ]
        emb = F.normalize(emb.float(), p=2, dim=1)
    return emb.cpu().numpy()[0]

def get_coherence_score(text, instruction,
                        tokenizer, emb_model):
    sents = [
        s.strip() for s in
        re.split(r'[.?!।]', str(text))
        if s.strip() and len(s.split())>2
    ][:15]
    if len(sents) < 2:
        return 0.0, 0.0, 0.0
    embs = np.array([
        get_embedding(s, instruction,
                      tokenizer, emb_model)
        for s in sents
    ])
    sims = [
        float(cosine_similarity(
            embs[i].reshape(1,-1),
            embs[i+1].reshape(1,-1)
        )[0][0])
        for i in range(len(embs)-1)
    ]
    return (float(np.mean(sims)),
            float(np.min(sims)),
            float(np.std(sims)))

ESSAY_INSTR = (
    "Instruct: Represent this Malayalam student "
    "answer for evaluating content accuracy and "
    "completeness against a reference answer\n"
    "Query: "
)
REF_INSTR = (
    "Instruct: Represent this Malayalam reference "
    "answer as the gold standard for scoring\n"
    "Query: "
)
SENT_INSTR = (
    "Instruct: Represent this Malayalam sentence "
    "for measuring coherence\nQuery: "
)


def predict_v2(
    student_answer,
    prompt_id=None,
    prompt_text=None,
    ref_answer=None,
    max_marks=10,
    actual_score=None,
    verbose=True
):
    """
    V2 prediction — uses word_ratio instead of word_count.

    Mode 1: predict_v2(student_answer, prompt_id=5)
    Mode 2: predict_v2(student_answer,
                       prompt_text="...",
                       ref_answer="...")
    """
    # Get prompt info
    if prompt_id is not None:
        row = q_df[q_df['prompt_id']==prompt_id]
        if len(row) == 0:
            print(f"❌ Prompt {prompt_id} not found!")
            return None
        prompt_text = str(row['prompt'].values[0])
        ref_answer  = str(row['ref_answer'].values[0])
        max_marks   = int(row['max_marks'].values[0])
    else:
        if not ref_answer or not prompt_text:
            print("❌ Need prompt_id OR "
                  "(prompt_text + ref_answer)")
            return None

    if verbose:
        print(f"\n{'='*55}")
        print("  MALAYALAM ASAG — V2 PREDICTION")
        print(f"{'='*55}")
        print(f"  Prompt: {str(prompt_text)[:80]}...")
        print(f"  Answer: "
              f"{len(str(student_answer).split())} words")
        print(f"  Ref:    "
              f"{len(str(ref_answer).split())} words")

    # Shallow features V2
    sf = get_mal_shallow_v2(student_answer, ref_answer)

    # Embeddings
    essay_emb = get_embedding(
        student_answer, ESSAY_INSTR,
        tokenizer, emb_model
    )
    ref_emb   = get_embedding(
        ref_answer, REF_INSTR,
        tokenizer, emb_model
    )
    er_sim    = float(cosine_similarity(
        essay_emb.reshape(1,-1),
        ref_emb.reshape(1,-1)
    )[0][0])

    # Coherence
    coh_m, coh_min, coh_std = get_coherence_score(
        student_answer, SENT_INSTR,
        tokenizer, emb_model
    )

    # Feature vector — V2 order
    sf_vec  = [sf[c] for c in SF_COLS_V2]
    emb_vec = (
        [coh_m, coh_min, coh_std, er_sim] +
        essay_emb.tolist()
    )
    feat    = np.array(
        sf_vec + emb_vec, dtype=np.float32
    ).reshape(1,-1)
    feat    = np.nan_to_num(
        feat, nan=0.0, posinf=0.0, neginf=0.0
    )

    p_norm  = model_xgb_v2.predict(feat)[0]
    p_raw   = p_norm * (Y_MAX - Y_MIN)
    p_score = int(np.clip(
        np.round(p_raw), Y_MIN, Y_MAX
    ))

    if verbose:
        print(f"\n  word_ratio: {sf['word_ratio']:.2f} "
              f"(student/ref length)")
        print(f"  similarity: {er_sim:.3f}")
        print(f"  coherence:  {coh_m:.3f}")
        print(f"\n  ► PREDICTED: {p_score}/{max_marks}")
        if actual_score is not None:
            diff = p_score - actual_score
            s    = "✅" if abs(diff)<=1 else "⚠️"
            print(f"  ► ACTUAL:    {actual_score}/"
                  f"{max_marks}  diff={diff:+d} {s}")
        print(f"{'='*55}")

    return p_score


# ── TEST — Same 3 tests with V2 ────────────────────────────
print("\n### TEST 1: Known essay from dataset ###")
row1 = df[df['prompt_id']==1].iloc[0]
predict_v2(
    student_answer = str(row1['essay']),
    prompt_id      = 1,
    actual_score   = int(row1['essay_score'])
)

print("\n### TEST 2: New custom prompt (French Revolution) ###")
predict_v2(
    student_answer ="""
    ഭാരതത്തിന്റെ പരമാധികാരവും നിയമവ്യവസ്ഥയും ഉറപ്പുവരുത്തുന്ന ഉന്നതമായ രേഖയാണ് ഇന്ത്യൻ ഭരണഘടന. ലോകത്തെ ഏറ്റവും ദീർഘമായ ലിഖിത ഭരണഘടനയാണിത്. 1950 ജനുവരി 26 മുതലാണ് ഇത് ഇന്ത്യയിൽ നടപ്പിലായത്. ഭരണഘടനാ ശില്പിയായ ഡോ. ബി.ആർ. അംബേദ്കറുടെ നേതൃത്വത്തിലാണ് ഇത് തയ്യാറാക്കിയത്. ഇന്ത്യയെ ഒരു മതേതര ജനാധിപത്യ രാജ്യമായി നിലനിർത്താൻ ഇത് സഹായിക്കുന്നു.

പൗരന്മാർക്ക് അന്തസ്സോടെ ജീവിക്കാൻ ആവശ്യമായ മൗലികാവകാശങ്ങൾ ഭരണഘടന നൽകുന്നുണ്ട്. അതോടൊപ്പം പൗരന്മാരുടെ കടമകളെക്കുറിച്ച് മൗലിക കർത്തവ്യങ്ങളിൽ പ്രതിപാദിക്കുന്നു. സർക്കാരുകൾ പാലിക്കേണ്ട നയങ്ങളെക്കുറിച്ച് നിർദ്ദേശക തത്വങ്ങൾ വ്യക്തമാക്കുന്നു. സ്വതന്ത്രമായ സുപ്രീം കോടതിയും ഹൈക്കോടതികളും പൗരന്മാരുടെ നീതി ഉറപ്പാക്കുന്നു. പ്രായപൂർത്തിയായ എല്ലാ പൗരന്മാർക്കും ജാതിമത ഭേദമന്യേ വോട്ട് ചെയ്യാനുള്ള അധികാരം ഇതിലൂടെ ലഭിക്കുന്നു. പാർലമെന്ററി ജനാധിപത്യം വഴി ജനങ്ങൾ നേരിട്ടാണ് ഭരണം തിരഞ്ഞെടുക്കുന്നത്. ആവശ്യമായ ഘട്ടങ്ങളിൽ മാറ്റങ്ങൾ വരുത്താൻ ഭരണഘടനാ ഭേദഗതികൾ സഹായിക്കുന്നു. ഏകീകൃതമായ പൗരത്വവും ശക്തമായ കേന്ദ്രസർക്കാരും ഇന്ത്യയുടെ സവിശേഷതയാണ്. നമ്മുടെ ജനാധിപത്യത്തിന്റെ നട്ടെല്ലാണ് ഈ ഭരണഘടന.
    """.strip(),
    prompt_text = "ഇന്ത്യൻ ഭരണഘടനയുടെ പ്രധാന സവിശേഷതകളെയും അത് രൂപപ്പെടുത്തിയ പശ്ചാത്തലത്തെയും കുറിച്ച് ഒരു ലഘു കുറിപ്പ് തയ്യാറാക്കുക.",
    ref_answer  = """
    ലോകത്തിലെ ഏറ്റവും വലിയ ലിഖിത ഭരണഘടനയാണ് ഇന്ത്യയുടേത്. 1949 നവംബർ 26-ന് ഭരണഘടനാ നിർമ്മാണ സഭ ഇത് അംഗീകരിക്കുകയും 1950 ജനുവരി 26-ന് നിലവിൽ വരികയും ചെയ്തു. ഡോ. ബി.ആർ. അംബേദ്കർ ആണ് ഭരണഘടനയുടെ മുഖ്യ ശില്പിയായി അറിയപ്പെടുന്നത്. ഒരു പരമാധികാര, സോഷ്യലിസ്റ്റ്, മതേതര, ജനാധിപത്യ, റിപ്പബ്ലിക് രാജ്യമായാണ് ഭരണഘടന ഇന്ത്യയെ വിഭാവനം ചെയ്യുന്നത്.

മൗലികാവകാശങ്ങൾ, മൗലിക കർത്തവ്യങ്ങൾ, നിർദ്ദേശക തത്വങ്ങൾ എന്നിവ ഭരണഘടനയുടെ സുപ്രധാന ഭാഗങ്ങളാണ്. പൗരന്മാരുടെ അവകാശങ്ങൾ സംരക്ഷിക്കുന്നതിനായി സ്വതന്ത്രമായ ഒരു നീതിന്യായ വ്യവസ്ഥയും ഇവിടെ നിലനിൽക്കുന്നു. പാർലമെന്ററി ഭരണരീതിയാണ് ഇന്ത്യ പിന്തുടരുന്നത്. ഭരണഘടന അനുശാസിക്കുന്ന പ്രകാരം പ്രായപൂർത്തി വോട്ടവകാശത്തിലൂടെയാണ് ജനപ്രതിനിധികളെ തിരഞ്ഞെടുക്കുന്നത്. ഭരണഘടനയിലെ ഭേദഗതികൾ വഴി കാലാനുസൃതമായ മാറ്റങ്ങൾ വരുത്താനും സാധിക്കും. ഫെഡറൽ സംവിധാനവും ഏകീകൃത സ്വഭാവവും ഒത്തുചേരുന്ന സവിശേഷമായ ഒരു ശൈലിയാണ് ഇതിനുള്ളത്. ഇന്ത്യയുടെ അഖണ്ഡതയും ഐക്യവും കാത്തുസൂക്ഷിക്കാൻ ഭരണഘടന ഓരോ പൗരനെയും പ്രാപ്തനാക്കുന്നു.
    """.strip(),
    max_marks   = 10
)

print("\n### TEST 3: Batch 20 samples ###")
sample = df.sample(20, random_state=99)
actuals, preds = [], []

print(f"\n{'ID':<8} {'Prompt':<8} "
      f"{'Actual':>8} {'Pred':>6} "
      f"{'Ratio':>7} {'Diff':>6}  Status")
print("─"*55)

for _, row in sample.iterrows():
    sf   = get_mal_shallow_v2(
        row['essay'], row['ref_answer_y']
    )
    p    = predict_v2(
        student_answer = str(row['essay']),
        prompt_id      = int(row['prompt_id']),
        verbose        = False
    )
    if p is None:
        continue
    actual = int(row['essay_score'])
    diff   = p - actual
    status = "✅" if abs(diff)<=1 else "⚠️"
    actuals.append(actual)
    preds.append(p)
    print(f"{row['essay_id']:<8} "
          f"{row['prompt_id']:<8} "
          f"{actual:>8} "
          f"{p:>6} "
          f"{sf['word_ratio']:>7.2f} "
          f"{diff:>+6}  {status}")

batch_qwk = cohen_kappa_score(
    actuals, preds, weights='quadratic'
)
batch_ea  = np.mean(
    np.array(actuals)==np.array(preds)
)
batch_aa  = np.mean(
    np.abs(np.array(actuals)-np.array(preds))<=1
)
print("─"*55)
print(f"  QWK:    {batch_qwk:.3f}")
print(f"  Exact:  {batch_ea:.3f}")
print(f"  Adj:    {batch_aa:.3f}")
print("\n✅ All V2 tests complete!")